<a id='seance_1'></a>
# Python et mysql.connector (le tutoriel)
Ce calepin jupyter est lié au tutoriel de la SAE disponible [ici](https://moodle.univ-ubs.fr/mod/page/view.php?id=344601).

Une [deuxième partie](#seance_2) de ce calepin sera consacrée aux processus ETL écrit en Python.

Une [troisième partie](#seance_mondrian) de ce calepin sera consacrée à la configuration d'un serveur Mondrian pour faire du ROLAP.

Une [quatrième partie](#seance_darties_2025) de ce calepin sera consacrée au portail DARTIES_2025.


## Préparation de la base de données
Les instructions SQL à exécuter au niveau de [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2&db=fnuc_python) après avoir sélectionner la base ```fnuc_python``` :
```
drop table if exists Commandes;
drop table if exists Clients;
create table if not exists Clients
 (
   id int not null primary key auto_increment comment 'identifant du client',
   nom varchar (50) not null comment 'Nom du client',
   motdepasse varchar (10) not null comment 'Mot de passe du client',
   cacumul double precision null comment 'Le chiffre d''affaires cumule par client' CHECK (CACUMUL >= 0),
   date_inscription DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
   actif bool default FALSE
 )
engine=InnoDB
row_format=default
comment='Les clients';
```

## Installation du connecteur MySQL
L'installation n'est pas nécessaire si la commande ```!pip show mysql-connector``` renvoie une version disponible.
Sinon, il faut faire :

```!pip install mysql-connector```

Remarque : Je vous ai fourni une distribution python avec tout d'installé !

In [ ]:
!pip show mysql-connector
#!pip show mysql-connector

## Importation du connecteur MySQL

Avant toute connexion à MariaDB/MySQL via le module ```mysql.connector```, il faut l'importer en python :

In [ ]:
import mysql.connector

## Connexion à la base de données

Le module ```mysql.connector``` fournit la méthode ```connect``` qui permet de retourner un objet qui représente la connexion vers la base de données. Vous devez fournir les paramètres ```host```, ```user``` et ```password``` pour donner l’adresse du SGBDR, le ```login``` et le ```mot de passe``` de connexion. Vous pouvez également fournir le paramètre ```database``` pour indiquer quelle base de données vous souhaitez utiliser.

L’interaction avec la base de données se fera à travers un **curseur**. Cet objet permet à la fois d’envoyer des requêtes et de consulter les résultats quand il s’agit d’une requête de consultation de données. Pour créer un curseur, on appelle la méthode ```cursor()``` sur la connexion. Il est recommandé de fermer un curseur lorsqu’il n’est plus utilisé.

In [ ]:
db = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
)
cursor = db.cursor()


## Prise de connaissance de la liste des tables

In [ ]:
cursor.execute('show tables;')
cursor.fetchall()
               

## Insertion de données
L’insertion de données se fait avec une requête SQL de type ```insert```.

Le connecteur MySQL fournit la propriété non standard ```autocommit``` sur la connexion. Si cette propriété vaut ```True``` alors un commit est automatiquement fait après chaque exécution de requête.

In [ ]:
# programme-03.py
db.autocommit=True
cursor.execute("insert into Clients (nom, motdepasse, cacumul) \
                   values ('dubois', 'michel',0)")

Il est souvent plus utile de fournir les valeurs à insérer sous la forme de paramètres. Il suffit de remplacer les valeurs par ```%s``` dans la requête et de passer en deuxième paramètre à la méthode ```execute``` un n-uplet avec les valeurs.

In [ ]:
# programme-04.py
request = """insert into Clients
             (nom, motdepasse, cacumul)
             values (%s, %s, %s)"""
params = ('dubois', 'michel',0)
cursor.execute(request, params)
db.commit()

Remarquez que le curseur se débrouille pour convertir les types Python en types SQL et nous pouvons directement passer des chaînes de caractères, des nombres, des valeurs booléennes et même des dates.

In [ ]:
# programme-05.py
import datetime

request = """insert into Clients
             (nom, motdepasse, cacumul, date_inscription)
             values (%s, %s, %s, %s)"""
params = ("dubois", "michel",0, datetime.date.today())

cursor.execute(request, params)
db.commit()

La méthode ```executemany``` permet d’insérer plusieurs lignes en passant un tableau pour la valeur des paramètres :

In [ ]:
# programme-06.py
request = """insert into Clients
             (nom, motdepasse, cacumul)
             values (%s, %s, %s)"""
params = [
	("dubois","michel",0),
	("budinger","marc",0),
	("pommier","valerie",0),
	("fevrier","yann",0),
	("perez","jc",0)
]
cursor.executemany(request, params)
db.commit()

Si vous voulez connaître le nombre de lignes insérées par l’exécution de la requête, vous pouvez consulter la propriété ```rowcount``` du curseur :

In [ ]:
# programme-07.py
request = """insert into Clients
             (nom, motdepasse, cacumul)
             values (%s, %s, %s)"""
params = [
	("dubois","michel",0),
	("budinger","marc",0),
	("pommier","valerie",0),
	("fevrier","yann",0),
	("perez","jc",0)
]
cursor.executemany(request, params)
db.commit()
print("Nombre de lignes insérées :", cursor.rowcount)

## Sélection de données
Pour récupérer des données depuis la base de données, il suffit de passer un requête SQL de type ```select``` en paramètre de la méthode ```execute``` du curseur et ensuite d’appeler la méthode ```fetchall()``` pour récupérer une liste de n-uplets contenant les résultats.

In [ ]:
# programme-08.py
request = "select id, nom, cacumul, date_inscription, actif from Clients"
cursor.execute(request)
resultats = cursor.fetchall()
for client in resultats:
    print(client)


On voit que les résultats sont bien des n-uplets et que le connecteur MySQL a réalisé une conversion de type sauf pour la colonne actif qui est un nombre au lieu d’être une valeur booléenne. Cela n’est pas trop grave car nous avons vu que ```True``` et ```False``` peuvent presque être considérés comme les nombres ```1``` et ```0```.

Note

La conversion imparfaite pour les valeurs booléennes ne vient pas de Python mais de MySQL. En effet, le type BOOLEAN n’existe pas vraiment en MySQL, il s’agit plus d’un alias pour une colonne de type entier.

## Récupérer les données en flux
La méthode ```fetchall()``` est simple à utiliser mais peut poser un problème de performance. En effet, cette méthode retourne une liste, cela signifie que tous les résultats sont récupérés de la base de données pour être convertis en n-uplets. S’il y a un nombre important de lignes dans le résultat de la requête, cela signifie que la liste peut être très grande et avoir une empreinte mémoire importante.

Parfois, on désire traiter directement la donnée retournée et il n’est pas nécessaire de la stocker dans une liste. Nous pouvons améliorer notre code en optant pour l’appel à la méthode ```fetchone()```. Cette méthode retourne un seul résultat sous la forme d’un n-uplet. Lorsqu’il n’y a plus de résultat à lire, la méthode retourne ```None```. Nous pouvons revoir notre code :

In [ ]:
# programme-09.py
request = "select id, nom, cacumul, date_inscription, actif from Clients"
cursor.execute(request)
while True:
    client = cursor.fetchone()
    if client is None:
        break
    print(client)

Le résultat du programme sera le même que précédemment sauf que les résultats sont extraits un à un et qu’aucune liste n’est créée.

Prudence

Si vous fermez un curseur avant d’avoir extrait tous les résultats, vous obtiendrez une exception. N’utilisez pas la méthode ```fetchone()``` pour ne retourner que le premier résultat, elle n’a pas été conçue pour cela. Son rôle est de permettre de retourner tous les résultats d’une requête mais un à un.

Si vous voulez limiter le nombre de résultats retournés par une requête, utilisez l’instruction SQL ```limit``` :

```
select id, nom, cacumul, date_inscription, actif from Clients limit 1
```
La méthode ```fetchmany()``` est un compromis entre ```fetchone()``` et ```fetchall()```. En effet, si ```fetchall()``` peut avoir un impact sur la consommation mémoire, ```fetchone()``` peut avoir un impact sur les performances car les aller-retour entre le programme et la base de données ont un coût en terme de temps.

La méthode ```fetchmany()``` prend en paramètre le nombre maximum de résultats qu’il faut aller chercher lors de cette appel. Lorsqu’il n’y a plus de résultat, la méthode retourne une liste vide.

In [ ]:
# programme-10.py
request = "select id, nom, cacumul, date_inscription, actif from Clients"
cursor.execute(request)
while True:
    resultats = cursor.fetchmany(10)
    if not resultats:
        break
    for client in resultats:
        print(client)

Si ce code paraît difficile à écrire, il est assez facile de créer une fonction pour simplifier une fois pour toute notre travail :

In [ ]:
# programme-11.py premiere partie
def fetch_from_database(cursor, request, callback, chunck_size=10):
    """
        Utilise le cursor pour exécuter la requête SQL donnée par request.
        callback est une méthode qui est appelée pour chaque résultat extrait.
        Elle recevra en paramètre le tuple des valeurs des colonnes.
        chunck_size indique le nombre max de résultats récupérés en une fois.
    """
    cursor.execute(request)
    while True:
        resultats = cursor.fetchmany(chunck_size)
        if not resultats:
            break
        for colonnes in resultats:
            callback(colonnes)

**Remarque :**
<div class="abstract">
    <p>Vous pouvez documenter une fonction
               <span class="application">Python</span> en lui donnant une chaîne de
               documentation (<tt class="literal">doc string</tt>).
    </p>
</div>
<div class="example">
<a name="odbchelper.triplequotes"></a>
<h4 class="title">Exemple Définition d'une <tt class="literal">doc string</tt> pour la fonction <tt class="function">buildConnectionString</tt></h4><pre class="programlisting"><span class="pykeyword">
def</span> buildConnectionString(params):
<span class="pystring">"""Build a connection string from a dictionary of parameters.

Returns string."""</span></pre><p>Les tripes guillemets indiquent une chaîne multi-lignes. Tout ce qu'il y a entre l'ouverture et la fermeture des guillemets
   fait partie de la chaîne, y compris les retours chariot et les autres guillemets. On peut les utiliser partout, mais vous
   les verrez le plus souvent utilisées pour définir une <tt class="literal">doc string</tt>.
</p>
</div><a name="compare.quoting.perl"></a><table class="note" border="0" summary="">
<tbody><tr>
   <td rowspan="2" align="center" valign="top" width="1%"><img src="../images/note.png" alt="NOTE" title="" width="24" height="24"></td>
</tr>
<tr>
   <td colspan="2" align="left" valign="top" width="99%">Les triples guillemets sont aussi un moyen simple de définir une
              chaîne contenant à la fois des guillemets simples et doubles, comme
              <tt class="literal">qq/.../</tt> en <span class="application">Perl</span>.
   </td>
</tr>
</tbody></table>
<p>Tout ce qui se trouve entre les triples guillemets fait partie de la
      <tt class="literal">doc string</tt> de la fonction, qui décrit ce que fait la
      fonction. Une <tt class="literal">doc string</tt>, si elle existe, doit être
      la première chose déclarée dans une fonction (la première chose après
      les deux points). Techniquement parlant, vous n'êtes pas obligés de
      donner une <tt class="literal">doc string</tt> à votre fonction, mais vous
      devriez toujours le faire. Je sais que vous avez entendu cela à tous les
      cours de programmation auxquels vous avez assisté mais
      <span class="application">Python</span> vous donne une motivation
      supplémentaire : la <tt class="literal">doc string</tt> est disponible à
      l'exécution en tant qu'attribut de fonction.
</p><a name="tip.docstring"></a><table class="note" border="0" summary="">
<tbody><tr>
   <td rowspan="2" align="center" valign="top" width="1%"><img src="../images/note.png" alt="NOTE" title="" width="24" height="24"></td>
</tr>
<tr>
   <td colspan="2" align="left" valign="top" width="99%">Beaucoup d'<span class="acronym">IDE</span> <span class="application">Python</span> utilisent les <tt class="literal">doc string</tt> pour fournir une documentation contextuelle, ainsi lorsque vous tapez le nom d'une fonction, sa <tt class="literal">doc string</tt> apparaît dans une bulle d'aide. Cela peut être incroyablement utile, mais cette utilité est liée à la qualité de votre <tt class="literal">doc string</tt>.
   </td>
</tr>
</tbody></table>
<div class="furtherreading">
<h4>Pour en savoir plus sur la documentation des fonctions</h4>
<ul>
   <li>La <a href="http://www.python.org/peps/pep-0257.html">PEP 257</a> définit les conventions pour les <tt class="literal">doc string</tt>.
   </li>
   <li>Le <a href="https://peps.python.org/pep-0008/"><i class="citetitle"><span class="application">Python</span> Style Guide</i></a> explique la manière d'écrire de bonnes <tt class="literal">doc string</tt>.
   </li>
   <li>Le <a href="https://docs.python.org/fr/3.5/tutorial/index.html"><i class="citetitle"><span class="application">Python</span> Tutorial</i></a> traite des conventions <a href="https://docs.python.org/fr/3.5/tutorial/controlflow.html#documentation-strings">d'espacement dans les <tt class="literal">doc string</tt></a>.
   </li>
</ul>
</div>

Afficher l'attribut de la fonction :

In [ ]:
print(fetch_from_database.__doc__)

Et le **programme** devient alors :

In [ ]:
# programme-11.py deuxieme partie
request = "select id, nom, cacumul, date_inscription, actif from Clients"
fetch_from_database(cursor, request, print)

<a id='req_prepareee_1'></a>
## Utilisation de paramètres dans les conditions where
Comme pour les requêtes d’insertion, il est utile de spécifier des paramètres dans une requête ```SELECT``` pour les valeurs d’une clause ```where```.

Astuce

Ce mécanisme nous prémunit de l’injection SQL car la structure de la requête est fournie quelle que soit la valeur des paramètres.

Les paramètres sont spécifiés par ```%s``` dans la requête et leurs valeurs sont données par un n-uplet passé comme second paramètre à la méthode ```execute``` :

In [ ]:
# programme-12.py
request = "select id, nom, cacumul, date_inscription, actif \
           from Clients where nom = %s"

params = ("dubois",)
cursor.execute(request, params)
resultats = cursor.fetchall()
for client in resultats:
    print(client)

<a id='req_prepareee_2'></a>
## Modification des données
Toutes les autres requêtes SQL de modification suivent le même principe que les requêtes d’insertion. On utilise la méthode ```execute``` d’un curseur. Il faut appeler la méthode ```commit()``` de la connexion pour valider la transaction. Enfin, il est possible de passer des paramètres à la requête et de connaître le nombre de lignes impactées par la requête grâce à la propriété ```rowcount```.

In [ ]:
# programme-13.py
# programme-14.py
request = "update Clients \
           set actif = %s \
           where nom = %s"

params = (True, "dubois")
cursor.execute(request, params)
db.commit()
print("Nombre de lignes mises à jour :", cursor.rowcount)

<a id='req_prepareee_3'></a>
## Supression des données
Voici un exemple de suppression d'un enregistrement :

In [ ]:
# programme-15.py
sql_Delete_query = """Delete from Clients where id = %s"""
# row to delete
clientId = 5
cursor.execute(sql_Delete_query, (clientId,))
db.commit()
print("Suppression d'enregistrement reussie ")

## Exécution de procédures et fonctions stockées
Les fonctions stockées, les procédures stockées, les déclencheurs, les évènements forment les options procédurales d'un serveur MySQL/MariaDB. Elles sont traitées dans [les transparents 213 à 265](https://moodle.univ-ubs.fr/pluginfile.php/582645/mod_resource/content/2/Concurrence-MySQL-BUT2-Collecte_auto_2022-adobe.pdf#page=213) du cours `MySQL/MariaDB (LDD+LMD), vues, sécurité & concurrence, options procédurales`.

Le schéma de la base de données contient des procédures stockées et des fonctions faisant référence aux tables ```Clients```, ```Livres```, ```Stocks``` et ```Commandes```. Parmi ces dernières, il me faut recréer la table ```Commandes``` dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2&db=fnuc_python) :
```
DROP TABLE IF EXISTS Commandes;
CREATE TABLE IF NOT EXISTS Commandes
 (
   ID INT NOT NULL AUTO_INCREMENT COMMENT 'Identifiant de la commande',
   DATECOM DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP COMMENT 'Date de la commande' ,
   ARTICLE INT NOT NULL COMMENT 'Reference de l''article commandee' ,
   CLIENT INT NOT NULL COMMENT 'Reference du client' ,
   QUANTITE DECIMAL NOT NULL COMMENT 'Quantite commandee'  CHECK (QUANTITE > 0),
   INDEX FKIndex_Article(ARTICLE),
   FOREIGN KEY FK_Article(ARTICLE) REFERENCES Livres (ID),
   INDEX FKIndex_Client(CLIENT),
   FOREIGN KEY FK_Client(CLIENT) REFERENCES Clients (ID)
   , PRIMARY KEY (ID) 
 )
ENGINE=InnoDB
ROW_FORMAT=default
COMMENT='Les commandes'
;

```

Puis je peux soumettre le code de création de fonctions et de procédure stockées à mon serveur MySQL, toujours via [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2&db=fnuc_python) :

```
DROP FUNCTION IF EXISTS DISP_ARTICLE;
-- fonction disponibilite (article, disp_stricte)
DELIMITER //
CREATE FUNCTION DISP_ARTICLE(article_cmd INT,disp_strict INT) 
    RETURNS DECIMAL
    LANGUAGE SQL READS SQL DATA
    SQL SECURITY DEFINER 
    COMMENT 'Disponibilite'
BEGIN
    DECLARE qte_disp DECIMAL DEFAULT 0;
    DECLARE niv_surete DECIMAL;
    DECLARE CONTINUE HANDLER FOR not FOUND
      	BEGIN
          SET qte_disp=0;
          SET niv_surete=0;
        END;
    SELECT niveau,securite INTO qte_disp,niv_surete 
    FROM Stocks 
    WHERE article=article_cmd;
    IF disp_strict = 1 THEN
        SET qte_disp = qte_disp-niv_surete;
    END IF;
    IF qte_disp<0 THEN
           SET qte_disp = 0;
    END IF;
    RETURN(qte_disp);
END
//
DELIMITER ;
DROP PROCEDURE IF EXISTS COMMANDER_OUT;
-- fonction Commander aujourd'hui (client,article, quantite)
DELIMITER //
CREATE PROCEDURE COMMANDER_OUT
       (id_client INT,
 	article_cmd INT,
	quantite_cmd DECIMAL,
       out p_sqlcode             INT,
       out p_status_message      VARCHAR(100))
    LANGUAGE SQL MODIFIES SQL DATA
    SQL SECURITY DEFINER
    COMMENT 'Commande en fonction des stocks'
BEGIN
        DECLARE disp DECIMAL DEFAULT 0;
	DECLARE EXIT HANDLER FOR SQLEXCEPTION
      	BEGIN
          SET p_sqlcode=1329;
          SET p_status_message='SQL Error occurred';
          ROLLBACK;
        END;
        SET p_sqlcode=0;
	START TRANSACTION;
        SET disp=DISP_ARTICLE(article_cmd,0);
	IF disp < quantite_cmd THEN
		SET p_sqlcode=1329;
                SET p_status_message='No enough quantity available';
                ROLLBACK;
        ELSE
		INSERT INTO Commandes(ID,DATECOM,ARTICLE,CLIENT,QUANTITE) 
        	VALUES (NULL,CURRENT_DATE,article_cmd,id_client,quantite_cmd);
		UPDATE Stocks
        	SET niveau=(niveau - quantite_cmd) 
        	WHERE article=article_cmd;
        	COMMIT;
        END IF;
END
//
DELIMITER ;
DROP FUNCTION IF EXISTS AUTH_CLIENT;
-- fonction authentifie client (nom, motdepasse)
DELIMITER //
CREATE FUNCTION AUTH_CLIENT(nom_client VARCHAR (50),motdepasse_client VARCHAR (10)) 
    RETURNS INT
    LANGUAGE SQL READS SQL DATA
    SQL SECURITY DEFINER 
    COMMENT 'Authentification'
BEGIN
    DECLARE id_client INT DEFAULT -1;
    DECLARE CONTINUE HANDLER FOR not FOUND
      	BEGIN
          SET id_client=-1;
        END;
    SELECT id INTO id_client 
    FROM Clients
    WHERE RTRIM(nom)=nom_client
    AND RTRIM(motdepasse)=motdepasse_client;
    RETURN(id_client);
END
//
```
Dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2&db=fnuc_python), on peut tester l'appel de la procedure `COMMANDER_OUT` :
```
CALL COMMANDER_OUT(1,1,1,@sqls,@mess);
select @sqls, @mess;
```
On peut obtenir comme résultat, si la commande n'est pas insérée :
```
1329 	No enough quantity available
```
Dans la procédure `COMMANDER_OUT`, il vaudrait mieux utiliser `CURRENT_TIMESTAMP` à la place de `CURRENT_DATE` vu le type de `DATECOM`.

L'instruction python pour exécuter une procédure stockée est :
```
result_args = cursor.callproc(proc_name, args=())
```
La méthode ```callproc()``` appelle la procédure stockée mentionnée dans l'argument ```proc_name```. La séquence de paramètres ```args``` doit contenir une entrée pour chaque argument attendu par la procédure. Par exemple, la procédure peut avoir un ou plusieurs paramètres ```IN``` et ```OUT```.

Le ```callproc()``` renvoie une copie modifiée de la séquence d'entrée. Cette méthode ne modifie pas les paramètres d'entrée. Cependant, il peut remplacer les paramètres de sortie et d'entrée/sortie par de nouvelles valeurs selon le résultat de l'exécution.

La procédure stockée renvoie la sortie dans les jeux de résultats, et elle est automatiquement récupérée et stockée en tant qu'instances ```MySQLCursorBuffered```.

Dans l'exemple suivant, nous allons exécuter la procédure stockée ```commander_out``` en utilisant Python.

In [ ]:
# programme-16.py
# programme-17.py
cursor.callproc('commander_out', [1,1,1,0,0])
# print results
print("Le resultat de l'execution est ")
for result in cursor.stored_results():
    print(result.fetchall())



<a id='req_prepareee_4'></a>
## Utilisation des requêtes paramétrées
Une requête paramétrée est une requête dans laquelle des espaces réservés (```%s```) sont utilisés pour les paramètres (valeurs de colonne) et les valeurs de paramètre fournies au moment de l'exécution.

Voyons l'exemple d'une requête paramétrée : 
```
sql_parameterized_query = """Update Clients set actif = %s where id = %s"""
```

Comme vous pouvez le voir, nous utilisons un espace réservé (```%s```) pour la colonne ```actif``` et ```id```. Nous devons fournir des valeurs dans des espaces réservés (```%s```) avant d'exécuter une requête. Passez des variables Python à la position de l'espace réservé lorsque nous exécutons une requête.

Nous devons passer les deux arguments suivants à une fonction ```cursor.execute()``` pour exécuter une requête paramétrée :
- la requête SQL ;
- Un tuple de valeurs de paramètre. Dans notre cas, nous devons passer deux variables Python, une pour le statut (actif) et une pour l'identifiant.
```
query = """Update Clients set actif = %s where id = %s"""
tuple1 = (TRUE, 5)
cursor.execute(query, tuple1)
```
Il y a **quatre raisons principales** pour utiliser une requête paramétrée avec une instruction préparée :

- Compiler une fois : Requête paramétrée est compilée une seule fois pour tout. Lorsque vous utilisez une requête paramétrée, elle est précompilée et stockée dans un objet ```PreparedStatement```. Maintenant, utilisez cet objet pour exécuter efficacement la même instruction plusieurs fois. Remarque : pour une requête standard, MySQL compile la requête à chaque fois avant de l'exécuter.
- Améliore la vitesse : si vous exécutez des instructions SQL à plusieurs reprises avec une requête précompilée, cela réduit le temps d'exécution.
- Même opération avec des données différentes : vous pouvez l'utiliser pour exécuter plusieurs fois la même requête avec des données différentes. Par exemple, vous souhaitez insérer 200 lignes dans un tableau. Dans de tels cas, utilisez une requête paramétrée pour exécuter à plusieurs reprises la même opération avec un ensemble de valeurs différent.
- Il empêche les attaques par injection SQL.

Comment utiliser la requête paramétrée en Python

Créez un objet d'instruction préparée à l'aide de ```connection.cursor(prepared=True)```. Cela crée un curseur spécifique sur lequel les instructions sont préparées et en fait, cela retourne une instance de classe ```MySQLCursorPrepared```.

**Attention**, il ne faut pas confondre l'exécution avec paramètres d'une requête SQL qui comporte des trous : elle ne sera pas l'objet de l'injection SQL et l'exécution classique d'une requête SQL sous la forme soit d'une interpolation de chaîne, soit d'une chaîne construite par concatenation de litéraux avec des variables. Ces deux dernièrs sont sujet à l'injection SQL et on n'a pas de gains de performances. De plus dans le modèle à interpoler ou dans la chaîne à concatener, la présence d'une variable de type chaîne ou date nécessite des apostrophes.

In [ ]:
# programme-18.py
# ordre sql prepares
# pour un processus ETL, c'est l'idéal !
# je dois redefinir la variable cursor pour un ordre prepare
cursor = db.cursor(prepared=True)
# Parameterized query
sql_insert_query = """ INSERT INTO Clients
                       (id, nom, motdepasse, date_inscription, actif) VALUES (%s,%s,%s,%s,%s)"""
# tuple to insert at placeholder
tuple1 = (100, "raphalen", "michele", "2019-03-23", 1)
tuple2 = (101, "basabe", "leonar", "2019-05-19", 1)

cursor.execute(sql_insert_query, tuple1)
cursor.execute(sql_insert_query, tuple2)
db.commit()
print("Donnees inserees avec succes dans la table Clients en utilisant une instruction preparee")


## Gestion des transactions complexes
Une transaction de base de données représente un tout. Toute opération qui modifie l'état de la base de données MySQL/InnoDB fait partie d'une transaction. Les transactions complexes sont constituées d'une multitude d'opérations et s'appuient sur la gestion des transactions.

Python MySQL Connector fournit les methodes suivantes pour gérer les transactions de base de données : 
- ```commit()``` : la méthode ```MySQLConnection.commit()``` envoie une instruction ```COMMIT``` au serveur MySQL, validant la transaction en cours. Après l'exécution réussie d'une requête, rendez les modifications persistantes dans une base de données à l'aide de l'appel à la méthode ```commit()``` de la classe représentant la connexion. 
- ```rollback()``` : ```MySQLConnection.rollback``` annule les modifications apportées par la transaction en cours. Lorsque l'exécution de l'une des opérations échoue et que vous souhaitez annuler ou annuler toutes vos modifications, appelez la méthode ```rollback()``` de l'objet de connexion MySQL.
- ```autoCommit()``` : la valeur ```MySQLConnection.autocommit``` peut être True ou False pour activer ou désactiver la fonctionnalité de validation automatique de MySQL. Par défaut, sa valeur est ```False```.

Il se peut que entre phpMyAdmin et ce calepin Jupyter, il y ait un conflit de transaction qui rend l'application phpMyAdmin instable : elle est mise en attente de la libération de verrous exclusifs posé par Jupyter

In [ ]:
#liberation des verrous si erreur précédente
db.rollback()

In [ ]:
# programme-19.py
# transaction complexe
# important pour vos processus ETL
db.autocommit = False
# je dois redefinir la variable cursor pour un ordre normal
cursor = db.cursor()
# diminution dans Stocks
sql_update_query = """Update Stocks set niveau = niveau -2 where id = 1"""
cursor.execute(sql_update_query)

# Insertion dans Commandes (ne pas mettre la valeur NULL a DATECOM 
sql_insert_query = """Insert into Commandes(article, client,quantite) values(1,1,2)"""
cursor.execute(sql_insert_query)
print("Succes de la mise a jour et de l'insertion de donnees dans une transaction complexe")

# Commit your changes
db.commit()


Une dernière requête, elle aussi *complexe* du point de vue des jointures :


In [ ]:
query = """
SELECT livres.TITRE, PRIX, stocks.*, datecom, quantite, cacumul FROM livres 
INNER JOIN stocks ON livres.id = stocks.article 
INNER JOIN commandes ON commandes.article = livres.id
INNER JOIN clients ON clients.id=commandes.client
WHERE nom = 'dubois';
"""
cursor.execute(query)
cursor.fetchall()

Avant de se quitter, il faut :
- fermer le curseur ;
- fermer la connexion !

In [ ]:
cursor.close()
db.close()
print ("C'est fini !")

## Comment dans un notebook gérer les exceptions avec une transaction complexe ?


Il faut tout faire dans une cellule ! Mettre les blocs ```try```, ```except``` et ```finally``` dans cette cellule


In [ ]:
import mysql.connector

connection = None
try:
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )
    connection.autocommit = False
    cursor = connection.cursor()
    # diminution dans Stocks
    sql_update_query = """Update Stocks set niveau = niveau -2 where id = 1"""
    cursor.execute(sql_update_query)

    # Insertion dans Commandes (ne pas mettre la valeur NULL a DATECOM 
    sql_insert_query = """Insert into Commandes(article, client,quantite) values(1,1,2)"""
    cursor.execute(sql_insert_query)
    print("Succes de la mise a jour et de l'insertion de donnees dans une transaction complexe")

    # Commit your changes
    connection.commit()

except mysql.connector.Error as error:
    print("Failed to update record to database rollback: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("La connexion est fermee")

<a id='seance_2'></a>
# Programmation avancée mysql python
Bien entendu, cette séance va s'appuyer sur les notions vues lors d'une [séance précédante](#seance_1). Nos processus ETL s'appuient sur l'interpolation de chaîne de caractères via l'opérateur `%` ou via la fonction `format` et non via les ordres SQL préparés (voir [exemple 1](#req_prepareee_1), [exemple 2](#req_prepareee_2), [exemple 3](#req_prepareee_3) et [exemple 4](#req_prepareee_4)), qui offrent des gains de performances et évite l'injection SQL. Cependant l'interpolation de requête qui en est l'alternative, permet de soumettre directement la requête journalisée dans phpMyAdmin pour la tester. A vous éventuellement de les utiliser.


## Début de la séance sur la programmation avancée mysql python
Il faut aller à la fin de cette section en 2024-2025, car l'archive `tools_MYSQL_8_XAMPP_7.2_tomcat_10_python_3.5_D_patch.zip` a dejà tout déployé dans vos outils.

Il faut arréter le serveur MySQL local.

Dans le répertoire `D:\tools\mysql-8.0.18-winx64\bin`, on va télécharger :
- [rename_user_database.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/rename_user_database.bat?forcedownload=1) pour expliciter plus tard exxxxxxx ;
- [sauve_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/sauve_local_data.bat?forcedownload=1) pour garder le résultat de mes manipulations sur mon serveur local sur le H: ;
- [restauration_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/restauration_local_data.bat?forcedownload=1) pour restaurer le contenu de l'archive.
- [split_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/split_local_data.bat?forcedownload=1) pour découper une archive en morceaux de 250 Mo, nécessite gfsplit.exe ;
- [gfsplit.exe](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/gfsplit.exe?forcedownload=1) binaire windows pour découper via une ligne de commandes un gros fichier ;
- [hjsplit.exe](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/hjsplit.exe?forcedownload=1) binaire windows pour découper un gros fichier et rejoindre les morceaux via une IHM ;
- [join_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/join_local_data.bat?forcedownload=1) pour joindre les morceaux pour recréer une grosse archive.

Il faut détruire le répertoire `D:\tools\mysql-8.0.18-winx64\data`, dans le shell MySQL via la commande :
```
rmdir /s /q D:\tools\mysql-8.0.18-winx64\data
```

Que ce soit les alternants ou les formations initiales, il y a [un jeu de données](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/tools_e_22_3xe2100837_j_rempli.zip?forcedownload=1) basé sur une même chaîne de tables des produits collectée l'année dernière pour obtenir un schéma en étoile avant alimentation tout en ayant un schéma de production complet. Il est idéal pour tester les processus ETL en python et beaucoup plus petit que le jeu des alternants limité aux années 2022 et 2023. Il est à décompresser à la racine du lecteur `D:\`.

Il faut démarrer le serveur MySQL local.

En 2023-2024 et en 2024-2025 :

Il faut démarrer le serveur MySQL local.

En 2023-2024, dans la fenêtre du Shell de MySQL faire les commandes :
1. `rename_user_database e_22_3je2100837 e_23_3jexxxxxxx de_22_3je2100837d`
2. `rename_user_database e_22_3pe2100837 e_23_3pexxxxxxx de_22_3pe2100837d`

En 2024-2025, dans la fenêtre du Shell de MySQL faire les commandes :
1. `rename_user_database_d e_23_3pexxxxxxx e_24_3pexxxxxxx de_23_3pexxxxxxxd`
2. `rename_user_database_f e_23_3jexxxxxxx e_24_3jexxxxxxx de_23_3jexxxxxxxd`

En 2025-2026, dans la fenêtre du Shell de MySQL faire les commandes :
1. `rename_user_database e_23_3pexxxxxxx e_25_3pexxxxxxx de_23_3pexxxxxxxd`
2. `rename_user_database e_23_3jexxxxxxx e_25_3jexxxxxxx de_23_3jexxxxxxxd`

Le schéma de production est complet mais la future simulation économique va regénérer les tables `t_objectif`, `t_facture` et `t_ligne_facture`.

## Exécutions des ordres depuis un script SQL
Normalement pour un script SQL composé d'instructions SQL statiques, il suffit d'ouvrir le fichier .sql puis en boucle, exécuter les instructions séparées par "`;`" à l'aide du curseur. Pour cela, il ne faut pas l'instruction `DELIMITER` dans le fichier qui comprend surement du code pour déclencheur, pour des procédures et des fonctions qui doivent être soumis au curseur d'un seul tenant même si ils comprennent plusieurs ";".

Le corps d'une procedure stockée MySQL peut n'être composé que d'ordres SQL statiques et non pas dynamiques (pour des ordres SQL dynamiques, il faut consulter le cas des ordes paramétrés comme pour `programme-18.py`). Dans le cas où l'on n'a pas le droit de soumettre des routines SQL à notre serveur MySQL comme pour notre serveur de production `projetud.univ-ubs:3306`, on peut sauvegarder l'ensemble des ordres statique dans un fichier sql individuel et procédure à la l'exécution en boucle de chacun des ordres contenus dans ce fichier.

Au préalable, dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2), il faut faire :
1. ```DROP DATABASE IF EXISTS `fnuc_python_simple_script`;```
2. ```CREATE DATABASE `fnuc_python_simple_script` COLLATE utf8mb4_0900_ai_ci ;```
3. ```GRANT ALL PRIVILEGES ON `fnuc\_python\_simple\_script`.* TO 'python'@'localhost';```

In [ ]:
import mysql.connector
try:
    mydb = mysql.connector.connect(
                                         host='localhost',
                                         port=3312,
                                         database='fnuc_python_simple_script',
                                         user='python',
                                         password='VOTRE_MDP',
                                         auth_plugin='mysql_native_password'
                                         )
    print("MySQL connection is opened") 
    with open('D:/tools/WinPython-64bit-3.5.3.1Qt5/python-3.5.3.amd64/FnucDB-mysql-8.0-simple.sql', 'r') as f:
        cursor = mydb.cursor()
        # on soumet une chaine de caractères contenant l’intégralité du fichier
        result = cursor.execute(f.read(), multi=True)
        mydb.commit()
except mysql.connector.Error as error:
    print("Failed SQL statement in MySQL: {}".format(error))

finally:
    if mydb.is_connected():
        cursor.close()
        mydb.close()
        print("MySQL connection is closed")

Dans cette version, on ne cherche pas à soumettre des ordres SQL isolés mais le contenu du fichier. Notez `multi=True` pour espérer exécuter sans erreur tous les ordres.

Dans la version suivante, si jamais le terme `DELIMITER` est présent, on ne fait aucune soumission. Sinon on stocke les ordres SQL dans une liste via le séparateur `;`.

In [ ]:
import mysql.connector
try:
    mydb = mysql.connector.connect(
                                         host='localhost',
                                         port=3312,
                                         database='fnuc_python_simple_script',
                                         user='python',
                                         password='VOTRE_MDP',
                                         auth_plugin='mysql_native_password'
                                         )
    print("MySQL connection is opened") 

    myfile = open('D:/tools/WinPython-64bit-3.5.3.1Qt5/python-3.5.3.amd64/FnucDB-mysql-8.0-simple.sql')
    # la variable data contient une chaine de caractères contenant l’intégralité du fichier
    data = myfile.read()
    query_list=[]

    if 'delimiter' not in data:
        query_list = (data.strip()).split(";")
    cursor = mydb.cursor()
    for j in query_list:
        cursor.execute(j)
        mydb.commit()        
except mysql.connector.Error as error:
    print("Failed SQL statement in MySQL: {}".format(error))

finally:
    if mydb.is_connected():
        cursor.close()
        mydb.close()
        print("MySQL connection is closed")            

S'il y a des instructions `DELIMITER` dans le fichier, c'est qu'il comprend surement du code pour déclencheur, pour des procédures et des fonctions qui doivent être soumis au curseur d'un seul tenant même si ils comprennent plusieurs ";"

Au préalable, dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2), il faut faire ;
1. ```DROP DATABASE IF EXISTS `fnuc_python_script`;```
2. ```CREATE DATABASE `fnuc_python_script` COLLATE utf8mb4_0900_ai_ci ;```
3. ```GRANT ALL PRIVILEGES ON `fnuc\_python\_script`.* TO 'python'@'localhost';```

En 2024-2025, votre serveur de développement **mais pas votre serveur de production** bénéficie de l'ajout de configation suivant :
De plus, si on rencontre l'erreur lors de la création de vues, de procédures, etc., sur le serveur local MySQL, pour disposer de la faculté de créer des procédures, fonctions, déclencheurs, évènements  stockés pour un utilisateur sans nécessiter les privilèges `SUPER` et `SYSTEM_USER`, il faut éditer le fichier `my.ini` (`D:\tools\mysql-8.0.18-winx64\my.ini`) du serveur MYSQL de développement et ajouter la ligne à la fin :

```
log_bin_trust_function_creators = 1 
```

Puis arreter le serveur MySQL et le démarrer à nouveau !

Voici comment on peut traiter ces difficultés : 

In [ ]:
# une version pour un script comportant des évènements, des procédures, des fonctions stockées et des déclencheurs
try:
    connection = mysql.connector.connect(
      host='localhost',
      port=3312,
      database='fnuc_python_script',
      user='python',
      password='VOTRE_MDP',
      auth_plugin='mysql_native_password'
    )
    connection.autocommit = False
    cursor = connection.cursor()
    queries = []
    delimiter = ';'
    query = ''

    with open('D:/tools/WinPython-64bit-3.5.3.1Qt5/python-3.5.3.amd64/FnucDB-mysql-8.0_prog26.sql', 'r') as f:
        for line in f.readlines():
            line = line.strip()
            if line.startswith('DELIMITER'):
                delimiter = line[10:]
            else:
                query += line+'\n'
                if line.endswith(delimiter):
                    # Get rid of the delimiter, remove any blank lines and add this query to our list
                    queries.append(query.strip().strip(delimiter))
                    query = ''


    for query in queries:
        if not query.strip():
            continue
        cursor.execute(query)
        print("Running query: ", query) # Will print out a short representation of the query
        if cursor.with_rows:
            print('result:', cursor.fetchall())
        else:
            print("Affected %i rows" %(cursor.rowcount))

    cursor.close()
    print("Succes des exécutions séparees")

    # Commit your changes
    connection.commit()
    print("Succes de la validation de la transaction")

except mysql.connector.Error as error:
    print("Failed to execute SQL statements sequence from file to database rollback: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("La connexion est fermee")
      

## Comment Journaliser dans un notebook Jupyter

Voir [ici](https://www.mineo.app/blog-page/python-logging-in-jupyter-notebooks) et [ici](https://docs.python.org/fr/3.5/howto/logging.html)

### Meilleures pratiques pour rédiger de bons messages de journal
1. Soyez descriptif mais concis

    Les messages des journaux doivent fournir suffisamment de contexte pour comprendre ce qui se passe, mais être suffisamment concis pour ne pas submerger le lecteur. Utilisez un langage clair qui décrit l’action, l’état ou la condition.

        👍 Bon : logging.info("Connexion à la base de données établie.")
        👎 Mauvais : logging.info("DB OK.")

2. Inclure les variables ou les identifiants pertinents

    Lors de la journalisation des événements, incluez toutes les variables, identifiants ou paramètres pertinents qui pourraient être utiles pour le débogage ou l’audit. Utilisez le formatage de chaîne pour les inclure dans le message du journal.

        👍 Bon : logging.info("Utilisateur %s authentifié avec succès.", user_id)
        👎 Mauvais : logging.info("Authentification réussie.")

3. Choisissez le niveau de journalisation approprié

    Utilisez le niveau de journalisation correct pour indiquer la gravité ou l'importance du message de journal. Cela aide à filtrer les journaux et à comprendre rapidement l’état du système.

         DEBUG pour des informations de diagnostic détaillées.
         INFO pour la confirmation des opérations réussies.
         AVERTISSEMENT pour les situations inattendues qui ne provoquent pas d'erreurs.
         ERREUR pour les problèmes qui perturbent la fonctionnalité normale.
         CRITIQUE pour les problèmes graves entraînant l’arrêt du programme.

4. Utilisez un formatage cohérent

    Maintenez un format cohérent pour vos messages de journal. Cela facilite la recherche, le filtrage et l’analyse des journaux. La cohérence doit s'appliquer à la structure du message, à la terminologie utilisée et même au temps.

        👍 Bon : logging.info("Fichier téléchargé : filename={}, size={}KB".format(file_name, file_size))
        👎 Mauvais : logging.info("Fichier téléchargé. Le nom du fichier est {}. La taille est de {} kilo-octets.".format(file_name, file_size))

5. Évitez de consigner des informations sensibles

    Soyez prudent avec les données que vous enregistrez. N'enregistrez jamais d'informations sensibles telles que des mots de passe, des clés API ou des informations personnelles identifiables (PII). Ceci est crucial pour des raisons de sécurité et de conformité.

        👍 Bon : logging.info("L'utilisateur {} a demandé la réinitialisation du mot de passe.".format(user_id))
        👎 Mauvais : logging.info("L'utilisateur {} a demandé la réinitialisation du mot de passe. Le nouveau mot de passe est {}.".format(user_id, new_password))


Mises en garde et solutions de journalisation dans les notebooks Jupyter

- État : les notebooks Jupyter sont avec état, ce qui signifie que les configurations de journalisation persistent dans toutes les cellules. Réinitialisez le noyau pour effacer les configurations.
- Gestionnaires multiples : exécuter une cellule plusieurs fois peut ajouter des gestionnaires en double. Assurez-vous de supprimer les gestionnaires existants avant d’en ajouter de nouveaux.
- Redémarrage du noyau requis pour les modifications globales : si vous apportez des modifications globales à la configuration de journalisation et souhaitez qu'elles prennent effet, vous devrez peut-être redémarrer le noyau Jupyter Notebook, ce qui effacera également toutes vos variables et importations.
- Sortie asynchrone : les notebooks Jupyter peuvent parfois produire une sortie asynchrone, ce qui fait apparaître les journaux dans le désordre. Cela peut prêter à confusion lorsque vous essayez de déboguer la séquence d'événements.

In [ ]:
import logging
# configuration initiale du systeme de journalisation
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")

In [ ]:
#on bascule au niveau debug !
logging.getLogger().setLevel(logging.DEBUG)
logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")

In [ ]:
# on bascule au niveau warning !
logging.getLogger().setLevel(logging.WARNING)
logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")

## Comment dans un notebook gérer les exceptions avec une transaction complexe avec journalisation ?


Il faut tout faire dans une cellule ! Mettre les blocs ```try```, ```except``` et ```finally``` dans cette cellule
On enlève les `print()` !


In [ ]:
import logging
import mysql.connector

connection = None
logging.getLogger().setLevel(logging.DEBUG)
#logging.getLogger().setLevel(logging.INFO)
try:
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )
    connection.autocommit = False
    cursor = connection.cursor()
    # diminution dans Stocks
    sql_update_query = """Update Stocks set niveau = niveau -2 where id = 1"""
    logging.debug("sql_update_query : "+sql_update_query)
    cursor.execute(sql_update_query)

    # Insertion dans Commandes (ne pas mettre la valeur NULL a DATECOM 
    sql_insert_query = """Insert into Commandes(article, client,quantite) values(1,1,2)"""
    logging.debug("sql_insert_query : "+sql_insert_query)
    cursor.execute(sql_insert_query)
    logging.info("Succes de la mise a jour et de l'insertion de donnees dans une transaction complexe")
    # Commit your changes
    connection.commit()

except mysql.connector.Error as error:
    logging.error("Failed to update record to database rollback: {}".format(error))             
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


## Comment écrire ou lire un fichier csv dans un notebook en gérant les exceptions  ?
Là encore tout dans une seule cellule. Le problème, c'est que l'on a des exceptions pour les accès aux fichiers CSV et pour l'accès à la base de données.

A noter que la bibliothèque `pandas` peut lire, écrire des fichiers csv et acceder en lecture ou en écriture à une base de données. Seulement dans la SAE, on a utilisé finement les instructions de définition de données pour MySQL or `pandas` maintient une structure de données centrale (le `dataframe`) et tout passe par cette structure. Pas sûr que les lectures ou écritures n'entrent pas en conflit avec MySQL. Pour une comparaison entre les deux bibliothèques, vous pouvez aller [ici](https://ioflood.com/blog/python-read-csv/). Je vous rappelle qu'il y a un tutoriel pour pandas dans cette SAE [ici](https://moodle.univ-ubs.fr/mod/page/view.php?id=347409&forceview=1)

La bibliothèque `csv` permet de gérer les fichiers CSV avec certains automatismes (Voir [ici](https://docs.python.org/fr/3.5/library/csv.html)).

Voyons un exemple simplifié avec SQLite embarqué dans Mozilla Firefox. Il n' y a aucune gestion des exceptions.
Pour le chemin, en 2023-2024, utilisez `%APPDATA%` dans l'explorateur de fichiers, puis trouvez le fichier `places.sqlite`. En 2024-2025, il semble que ce soit `H:\firefox\rsp17jiy.slt`

In [ ]:
#ecritutre d'un fichier csv dans le repertoire des calepins jupyter
import csv
import sqlite3

conn = sqlite3.connect( # open "places.sqlite" from one of the Firefox profiles
   #'C:/Users/duboism/AppData/Roaming/Mozilla/Firefox/Profiles/9xc28eme.default-esr/places.sqlite'
   'H:/firefox/rsp17jiy.slt/places.sqlite' 
)
cursor = conn.cursor()
cursor.execute("select * from moz_places;")
with open("out.csv", "w", newline='') as csv_file:   
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([i[0] for i in cursor.description]) # write headers
    csv_writer.writerows(cursor)

Il faut consulter avec notepad++, le fichier `out.csv`.

Maintenant, passons au cas de l'export depuis MySQL via l'écriture d'un fichier CSV.

In [ ]:
#ecritutre d'un fichier csv dans le repertoire des calepins jupyter
import csv
import mysql.connector

connection = None
try:
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )
    cursor = connection.cursor()
    cursor.execute("select * from motscles;")
    with open("fnuc_motscles.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    print("Failed to read record from database or write file: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("La connexion est fermee")        

Il nous faut faire de même pour la table clients afin de pouvoir utiliser le fichier ensuite.

In [ ]:
#ecritutre d'un fichier csv dans le repertoire des calepins jupyter
import csv
import mysql.connector

connection = None
try:
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )
    cursor = connection.cursor()
    cursor.execute("select * from clients;")
    with open("fnuc_clients.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    print("Failed to read record from database or write file: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("La connexion est fermee")

Passons au chargement d'un fichier CSV dans une table de la base de données avec utilisation automatique de l'entête.

Si vous obtenez l'erreur `1062 (23000): Duplicate entry '0' for key 'PRIMARY'`, verifiez bien que la table `Clients` a sur la colonne `ID` la propriété `AUTO_INCREMENT` comme `Extra` sous [phpMyadmin](http://localhost:96/phpmyadmin/tbl_structure.php?db=fnuc_python&table=clients&server=2). L'auto increment a été ajouté lors de l'exécution des instructions au début du calepin, même si la table `clients` n'a pas d'auto_increment classiquement. Rappel : le `ID` des tables `Livres` et `Commandes` sont auto-incrémentés alors que les `ID` des tables `Clients`, `Motscles` et `Sujets` ne sont pas auto incrémentés. 

In [ ]:
import csv
from csv import reader
import mysql.connector

connection = None
try:
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )
    connection.autocommit = False
    cursor = connection.cursor()
    i = 1

    with open('fnuc_clients.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)

        for row in csv_reader:
            query = """INSERT INTO clients (nom,motdepasse,cacumul) VALUES ('%s','%s',0)""" % (row['nom'],row['motdepasse'])
            print (query)
            cursor.execute(query)
            print(i)
            i = i + 1
    
    # Commit your changes
    connection.commit()

except (csv.Error,mysql.connector.Error,FileNotFoundError) as error:
    print("Failed to insert record to database rollback or read csv file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("La connexion est fermee")    

On peut accélérer le chargement des données avec l'utilisation de `cursor.executemany` que l'on a déjà vu (voir [ici](https://stackoverflow.com/questions/69082932/how-to-speed-up-python-csv-read-to-mysql-write)). 

## Outil de simulation économique écrit en Python

L'outil de simulation économique à écrire en Python consiste :

- à remplir la table `T_Objectif` qui donnera lieu à la création de fichier CSV par année ;
- à remplir les tables `T_facture` et `T_ligne_facture`. 

### Phase 0 pour la mise en place de l'environnement (à ne pas faire)

Sur le serveur MySQL de développement, exécuter ces ordres dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2) :
```
DROP USER IF EXISTS 'e_25_3jexxxxxxx'@'%' ;
CREATE USER 'e_25_3jexxxxxxx'@'%' IDENTIFIED WITH mysql_native_password BY 'de_25_3jexxxxxxxd';
GRANT SELECT, INSERT, UPDATE, DELETE, CREATE, DROP, FILE, INDEX, ALTER, CREATE TEMPORARY TABLES, CREATE VIEW, EVENT, TRIGGER, SHOW VIEW, CREATE ROUTINE, ALTER ROUTINE, EXECUTE ON *.* TO 'e_25_3jexxxxxxx'@'%';
ALTER USER 'e_25_3jexxxxxxx'@'%' REQUIRE NONE WITH MAX_QUERIES_PER_HOUR 0 MAX_CONNECTIONS_PER_HOUR 0 MAX_UPDATES_PER_HOUR 0 MAX_USER_CONNECTIONS 0;
CREATE DATABASE IF NOT EXISTS `e_25_3jexxxxxxx`;
GRANT ALL PRIVILEGES ON `e\_25\_3jexxxxxxx`.* TO 'e_25_3jexxxxxxx'@'%';
DROP USER IF EXISTS 'e_25_3pexxxxxxx'@'%' ;
CREATE USER 'e_25_3pexxxxxxx'@'%' IDENTIFIED WITH mysql_native_password BY 'de_25_3pexxxxxxxd';
GRANT SELECT, INSERT, UPDATE, DELETE, CREATE, DROP, FILE, INDEX, ALTER, CREATE TEMPORARY TABLES, CREATE VIEW, EVENT, TRIGGER, SHOW VIEW, CREATE ROUTINE, ALTER ROUTINE, EXECUTE ON *.* TO 'e_25_3pexxxxxxx'@'%';
ALTER USER 'e_25_3pexxxxxxx'@'%' REQUIRE NONE WITH MAX_QUERIES_PER_HOUR 0 MAX_CONNECTIONS_PER_HOUR 0 MAX_UPDATES_PER_HOUR 0 MAX_USER_CONNECTIONS 0;
CREATE DATABASE IF NOT EXISTS `e_25_3pexxxxxxx`;
GRANT ALL PRIVILEGES ON `e\_25\_3pexxxxxxx`.* TO 'e_25_3pexxxxxxx'@'%';
```
Dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=e_23_3pexxxxxxx&server=2), il faut déployer le schéma décisionnel qui est disponible dans [cette page](https://moodle.univ-ubs.fr/mod/page/view.php?id=396267). Notez dans cette page la définition des procédures stockées `maj_dim_magasin_star`, `maj_dim_produit_star`, `maj_faits_ventes_star`, `maj_securite_star`, `maj_dwr_faits_ventes_star`. Ainsi le rafraichissement du schéma en étoile à partir du schéma en flocon se fait par la séquence suivante :
```
CALL maj_dim_magasin_star();
CALL maj_dim_produit_star();
CALL maj_faits_ventes_star();
CALL maj_securite_star();
CALL maj_dwr_faits_ventes_star();
```

Pour les alternants, il faut télécharger [ici](https://moodle.univ-ubs.fr/mod/resource/view.php?id=396270) dans le répertoire `D:\tools\mysql-8.0.18-winx64\bin` le schéma de production avant simulation en ouvrant le shell MySQL et en exécutant la commande (une ligne à la fois):
```
mysql -P 3312 -u e_25_3jexxxxxxx -pfe_25_3jexxxxxxxf e_25_3jexxxxxxx
\. my_PRJSAE301_2022_2BVC11_PROD_GRP0.sql
```

Pour les formations initiales, il faut télécharger [ici](https://moodle.univ-ubs.fr/mod/resource/view.php?id=396276) dans le répertoire `D:\tools\mysql-8.0.18-winx64\bin` le schéma de production partiel et [un complément préalable](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/my_PRJSAE301_2022_2BVC01_PROD_GRP0_data_2022_only_utf8_2022_2023.sql?forcedownload=1) à la simulation dans le même répertoire en ouvrant le shell MySQL et en exécutant la commande (une ligne à la fois) :
```
mysql -P 3312 -u e_25_3jexxxxxxx -pde_25_3jexxxxxxxd e_25_3jexxxxxxx
\. my_PRJSAE301_2022_2BVC11_PROD_GRP0.sql
\. my_PRJSAE301_2022_2BVC01_PROD_GRP0_data_2022_only_utf8_2022_2023.sql
```
Si les fichiers ont le même nom, les données spécifiques aux alternants sont des commentaires afin que celui des FI n'ait pas des données sur les produits.

De même, dans le répertoire `D:\tools\mysql-8.0.18-winx64\bin`, on va télécharger :
- [rename_user_database.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/rename_user_database.bat?forcedownload=1) pour expliciter plus tard exxxxxxx ;
- [sauve_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/sauve_local_data.bat?forcedownload=1) pour garder le résultat de mes manipulations sur mon serveur local sur le H: ;
- [restauration_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/restauration_local_data.bat?forcedownload=1) pour restaurer le contenu de l'archive ;

Le jeu de données des FI et des FA n'étant pas les mêmes, celui des alternants étant très gros voire TROP gros car la simulation économique prend des heures a être exécutée et le serveur de production MySQL est saturé. Aussi une phase d'initialisation alternative est proposée.


### Phase 0 alternative beaucoup plus sure (déjà fait en début de séance)

Il faut arréter le serveur MySQL local.

Il faut détruire le répertoire D:\tools\mysql-8.0.18-winx64\data, dans le shell MySQL via la commande :
```
rmdir /s /q D:\tools\mysql-8.0.18-winx64\data
```


Dans le répertoire `D:\tools\mysql-8.0.18-winx64\bin`, on va télécharger :
- [rename_user_database.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/rename_user_database.bat?forcedownload=1) pour expliciter plus tard exxxxxxx ;
- [sauve_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/sauve_local_data.bat?forcedownload=1) pour garder le résultat de mes manipulations sur mon serveur local sur le H: ;
- [restauration_local_data.bat](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/restauration_local_data.bat?forcedownload=1) pour restaurer le contenu de l'archive.

Que ce soit les alternants ou les formations initiales, il y a [un jeu de données](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/tools_e_22_3xe2100837_j_rempli.zip?forcedownload=1) basé sur une même chaîne de tables des produits collectée l'année dernière pour obtenir un schéma en étoile avant alimentation tout en ayant un schéma de production complet. Il est idéal pour tester les processus ETL en python et beaucoup plus petit que le jeu des alternants limité aux années 2022 et 2023. Il est à décompresser à la racine du lecteur `D:\`.

Il faut démarrer le serveur MySQL local.

Dans la fenêtre du Shell de MySQL faire les commandes :
1. `rename_user_database e_22_3je2100837 e_23_3jexxxxxxx de_22_3je2100837d`
2. `rename_user_database e_22_3pe2100837 e_23_3pexxxxxxx de_22_3pe2100837d`


Le schéma de production est complet mais la simulation économique va regénérer les tables t_objectif, t_facture et t_ligne_facture.



### Phase 1 de la simulation économique


La première étape consiste à mettre plutôt les nom des mois que leur numéro, plutôt les villes des magasins plutôt que les numéros des magasins, plutôt les désignations des produits plutôt que les numéros des produits car le fichier CSV a été crée par un humain, le directeur commercial du groupe DARTIES.

Les ordres SQL de ce paragraphe sont compatibles à la fois avec MySQL et MS Access qui visiblement supporte aussi le caractère de protection historique de MySQL backtick`` ` ``avec le duo `[]`
Donc on peut rajouter les contraintes d'unicité suivantes sur le schéma de production :
```
-- ALTER TABLE `t_magasin` ADD UNIQUE(`ville`);
ALTER TABLE `t_produit` ADD UNIQUE(`libelle`); 
```

Remarque : quand on crée la première contrainte, on reçoit un message `Warning: #1831 Duplicate index 'ville_2' defined on the table 'e_23_3jexxxxxxx.t_magasin'. This is deprecated and will be disallowed in a future release.` 
Il y a un index redondant concernant la clé étrangère ville qui a l'origne n'était déjà unique. Cela explique pourquoi on n'a pas de doublon au niveau de ville :-)

Comme la contrainte est mise après l'alimentation des produits et des magasins, il peut déjà exister des doublons qui empèchent l'ajout de cette contrainte d'unicité. Voici un ordre SQL avec sous-requête correlée pour traquer ces doublons et soit les corriger en spécifiant un libellé ou une ville un peu différent, soit les détruire (remplacer `SELECT id_produit` par `DELETE `) pour resoumettre la création de la contrainte, apparement trop tardive.
```
SELECT `id_produit`
FROM `t_produit` T
WHERE `libelle` IN (
  SELECT `libelle`
  FROM `t_produit` 
  GROUP BY `libelle`
  HAVING COUNT(*)>1
) AND `id_produit`<> (
  SELECT MIN(`id_produit`)
  FROM `t_produit`
  WHERE `libelle`=T.`libelle`
)
;
```
Sur le jeu de données de l'année dernière : 526
et 
```
DELETE
FROM `t_produit`
WHERE `id_produit` IN (526)    
```

A noter que MS Access accepte l'ordre DELETE avec sous -requête imbriquée alors que MySQL non d'où l'ordre précédant simple.

et 
```
SELECT `id_magasin`
FROM `t_magasin` T
WHERE `ville` IN (
  SELECT `ville`
  FROM `t_magasin` 
  GROUP BY `ville`
  HAVING COUNT(*)>1
) AND `id_magasin`<> (
  SELECT MIN(`id_magasin`)
  FROM `t_magasin` 
  WHERE `ville`=T.`ville`
)
;
```
Si la simulation économique a déjà été exécutée, et que l'on envisage la suppression des doublons,
Il faut les supprimer dabord des tables T_OBJECTIF, T_FACTURE et T_LIGNE_FACTURE puis des tables T_PRODUIT et T_MAGASIN. Après, il faut relancer les phase 1 et 2 de la simulation économique. La phase 2 dépend de la phase 1 et avant la phase 1, il faut avoir éliminé les doublons sinon ils risque de faire dérailler le traitement des fichiers CSV.
```
DELETE 
FROM `t_objectif` 
WHERE `id_produit` IN (
  SELECT `id_produit`
  FROM `t_produit` T
  WHERE `libelle` IN (
    SELECT `libelle`
    FROM `t_produit` 
    GROUP BY `libelle`
    HAVING COUNT(*)>1
  ) AND `id_produit`<> (
    SELECT MIN(`id_produit`)
    FROM `t_produit`
    WHERE `libelle`=T.`libelle`
  )
) OR `id_magasin` IN (
  SELECT `id_magasin`
  FROM `t_magasin` T
  WHERE `ville` IN (
    SELECT `ville`
    FROM `t_magasin` 
    GROUP BY `ville`
    HAVING COUNT(*)>1
  ) AND `id_magasin`<> (
    SELECT MIN(`id_magasin`)
    FROM `t_magasin` 
    WHERE `ville`=T.`ville`
  )    
)
;
```
Sur le jeu de données de l'année dernière : 2880 lignes affectées. (traitement en 172,0000 seconde(s).)

et
```
DELETE
FROM `t_ligne_facture` 
WHERE `id_produit` IN (
  SELECT `id_produit`
  FROM `t_produit` T
  WHERE `libelle` IN (
    SELECT `libelle`
    FROM `t_produit` 
    GROUP BY `libelle`
    HAVING COUNT(*)>1
  ) AND `id_produit`<> (
    SELECT MIN(`id_produit`)
    FROM `t_produit`
    WHERE `libelle`=T.`libelle`
  ) OR `num_commande` IN (
    SELECT `num_commande` 
    FROM `t_facture`
    WHERE `id_magasin` IN (
      SELECT `id_magasin`
      FROM `t_magasin` T
      WHERE `ville` IN (
        SELECT `ville`
        FROM `t_magasin` 
        GROUP BY `ville`
        HAVING COUNT(*)>1
      ) AND `id_magasin`<> (
        SELECT MIN(`id_magasin`)
        FROM `t_magasin` 
        WHERE `ville`=T.`ville`
      )
    )    
  )    
)
;
```
Sur le jeu de données de l'année dernière : 3165 lignes affectées. (traitement en 180,0000 seconde(s).)

et 
```
DELETE 
FROM `t_facture` 
WHERE `num_commande` NOT IN (
  SELECT `num_commande` 
    FROM `t_ligne_facture`
)
;
```
Sur le jeu de données de l'année dernière : 3165 lignes affectées. (traitement en 3,0000 seconde(s).)


MySQL permet d'obtenir les noms des mois en anglais mais pas en français. Il va falloir créer une fonction stockées pour avoir notamment les noms des mois en français.
```
DELIMITER ;
DROP FUNCTION IF EXISTS DATE_FORMAT_FR;
DELIMITER |
CREATE FUNCTION DATE_FORMAT_FR (dt DATETIME, f VARCHAR(32))
RETURNS VARCHAR(255) DETERMINISTIC NO SQL
COMMENT 'Extends DATE_FORMAT function to get French month and day names'
BEGIN
    DECLARE name VARCHAR(16);
    /*short day name*/
    IF INSTR(f, '%a') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%a')
            WHEN 'Mon' THEN 'Lun'
            WHEN 'Tue' THEN 'Mar'
            WHEN 'Wed' THEN 'Mer'
            WHEN 'Thu' THEN 'Jeu'
            WHEN 'Fri' THEN 'Ven'
            WHEN 'Sat' THEN 'Sam'
            WHEN 'Sun' THEN 'Dim'
            ELSE DATE_FORMAT(dt, '%a')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%a', name);
    END IF;
    /*long day name*/
    IF INSTR(f, '%W') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%W')
            WHEN 'Monday' THEN 'Lundi'
            WHEN 'Tuesday' THEN 'Mardi'
            WHEN 'Wednesday' THEN 'Mercredi'
            WHEN 'Thursday' THEN 'Jeudi'
            WHEN 'Friday' THEN 'Vendredi'
            WHEN 'Saturday' THEN 'Samedi'
            WHEN 'Sunday' THEN 'Dimanche'
            ELSE DATE_FORMAT(dt, '%W')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%W', name);
    END IF;
     /*short month name*/
    IF INSTR(f, '%b') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%b')
            WHEN 'Jan' THEN 'Jan'
            WHEN 'Feb' THEN 'Fév'
            WHEN 'Mar' THEN 'Mar'
            WHEN 'Apr' THEN 'Avr'
            WHEN 'May' THEN 'Mai'
            WHEN 'Jun' THEN 'Juin'
            WHEN 'Jul' THEN 'Juil'
            WHEN 'Aug' THEN 'Août'
            WHEN 'Sep' THEN 'Sept'
            WHEN 'Oct' THEN 'Oct'
            WHEN 'Nov' THEN 'Nov'
            WHEN 'Dec' THEN 'Déc'
            ELSE DATE_FORMAT(dt, '%b')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%b', name);
    END IF;
    /*long month name*/
    IF INSTR(f, '%M') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%b')
            WHEN 'Jan' THEN 'Janvier'
            WHEN 'Feb' THEN 'Février'
            WHEN 'Mar' THEN 'Mars'
            WHEN 'Apr' THEN 'Avril'
            WHEN 'May' THEN 'Mai'
            WHEN 'Jun' THEN 'Juin'
            WHEN 'Jul' THEN 'Juillet'
            WHEN 'Aug' THEN 'Août'
            WHEN 'Sep' THEN 'Septembre'
            WHEN 'Oct' THEN 'Octobre'
            WHEN 'Nov' THEN 'Novembre'
            WHEN 'Dec' THEN 'Décembre'
            ELSE DATE_FORMAT(dt, '%M')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%M', name);
    END IF;
    /*returns standard conversion*/
    RETURN DATE_FORMAT(dt, f);
END |
```


Si on obtient cette journalisation :
```
2023-11-14 16:54:49,917 - ERROR - Failed to read record from database or write file: 1449 (HY000): The user specified as a definer ('"root_php"'@'"%"') does not exist
```
`root_php` est l'utilisateur avec lequel on est connecté dans phpMyAdmin.
Il faut alors soumettre la création de la fonction en tant que l'utilisateur qui se connecte dans les programmes Python.


Vu ce que nous avons déjà travaillé dans notre notebook python, on peut déjà faire la phase 1 : remplir la table T_Objectif et la création de fichier CSV par année.


In [ ]:
# soumission de la définition de la fonction stockee MySQL
try:
    connection = mysql.connector.connect(
     host='localhost',
     port=3312,
     database='e_23_3jexxxxxxx',
     user='e_23_3jexxxxxxx',
     password='de_23_3jexxxxxxxd',
     auth_plugin='mysql_native_password'
    )
    connection.autocommit = False
    cursor = connection.cursor()
    dbString = """
DELIMITER ;
DROP FUNCTION IF EXISTS DATE_FORMAT_FR;

DELIMITER |
CREATE FUNCTION DATE_FORMAT_FR (dt DATETIME, f VARCHAR(32))
RETURNS VARCHAR(255) DETERMINISTIC NO SQL
COMMENT 'Extends DATE_FORMAT function to get French month and day names'
BEGIN
    DECLARE name VARCHAR(16);
    /*short day name*/
    IF INSTR(f, '%a') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%a')
            WHEN 'Mon' THEN 'Lun'
            WHEN 'Tue' THEN 'Mar'
            WHEN 'Wed' THEN 'Mer'
            WHEN 'Thu' THEN 'Jeu'
            WHEN 'Fri' THEN 'Ven'
            WHEN 'Sat' THEN 'Sam'
            WHEN 'Sun' THEN 'Dim'
            ELSE DATE_FORMAT(dt, '%a')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%a', name);
    END IF;
    /*long day name*/
    IF INSTR(f, '%W') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%W')
            WHEN 'Monday' THEN 'Lundi'
            WHEN 'Tuesday' THEN 'Mardi'
            WHEN 'Wednesday' THEN 'Mercredi'
            WHEN 'Thursday' THEN 'Jeudi'
            WHEN 'Friday' THEN 'Vendredi'
            WHEN 'Saturday' THEN 'Samedi'
            WHEN 'Sunday' THEN 'Dimanche'
            ELSE DATE_FORMAT(dt, '%W')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%W', name);
    END IF;
     /*short month name*/
    IF INSTR(f, '%b') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%b')
            WHEN 'Jan' THEN 'Jan'
            WHEN 'Feb' THEN 'Fév'
            WHEN 'Mar' THEN 'Mar'
            WHEN 'Apr' THEN 'Avr'
            WHEN 'May' THEN 'Mai'
            WHEN 'Jun' THEN 'Juin'
            WHEN 'Jul' THEN 'Juil'
            WHEN 'Aug' THEN 'Août'
            WHEN 'Sep' THEN 'Sept'
            WHEN 'Oct' THEN 'Oct'
            WHEN 'Nov' THEN 'Nov'
            WHEN 'Dec' THEN 'Déc'
            ELSE DATE_FORMAT(dt, '%b')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%b', name);
    END IF;
    /*long month name*/
    IF INSTR(f, '%M') >= 0 THEN
        SET name = CASE DATE_FORMAT(dt, '%b')
            WHEN 'Jan' THEN 'Janvier'
            WHEN 'Feb' THEN 'Février'
            WHEN 'Mar' THEN 'Mars'
            WHEN 'Apr' THEN 'Avril'
            WHEN 'May' THEN 'Mai'
            WHEN 'Jun' THEN 'Juin'
            WHEN 'Jul' THEN 'Juillet'
            WHEN 'Aug' THEN 'Août'
            WHEN 'Sep' THEN 'Septembre'
            WHEN 'Oct' THEN 'Octobre'
            WHEN 'Nov' THEN 'Novembre'
            WHEN 'Dec' THEN 'Décembre'
            ELSE DATE_FORMAT(dt, '%M')
            END;
            /*replace in source format string*/
            SET f = REPLACE(f, '%M', name);
    END IF;
    /*returns standard conversion*/
    RETURN DATE_FORMAT(dt, f);
END |
"""
    import re
    # Find special delimiters
    delimiters = re.compile('DELIMITER *(\S*)',re.I)
    result = delimiters.split(dbString)
    
    # Insert default delimiter and separate delimiters and sql
    result.insert(0,';') 
    delimiter = result[0::2]
    section   = result[1::2]
    
    # Split queries on delimiters and execute
    for i in range(len(delimiter)):
        queries = section[i].split(delimiter[i])
        for query in queries:
            if not query.strip():
                continue
            cursor.execute(query)
            logging.info("Running query: ", query) # Will print out a short representation of the query
            if cursor.with_rows:
                logging.info('result:', cursor.fetchall())
            else:
                logging.info("Affected %i rows" %(cursor.rowcount))

    cursor.close()
    logging.info("Succes des exécutions séparees")

    # Commit your changes
    connection.commit()
    logging.info("Succes de la validation de la transaction")

except mysql.connector.Error as error:
    logging.error("Failed to execute SQL statements sequence to database rollback: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
import csv
import mysql.connector
import decimal
from random import *

def insert_t_objectif_create_csv_files():
    
    connection = None
    try:
        connection = mysql.connector.connect(
         host='localhost',
         port=3312,
         database='e_23_3jexxxxxxx',
         user='e_23_3jexxxxxxx',
         password='de_23_3jexxxxxxxd',
         auth_plugin='mysql_native_password'
        )
        connection.autocommit = False
        cursor = connection.cursor()
        # Vidage de la table t_objectif
        # MS Access n'ccepte que cela
        strSQL_DELETE = "DELETE FROM t_objectif;"
        # MySQL considère que c'est un ordre DDL et non MDL
        strSQL_DELETE = "TRUNCATE t_objectif;"
        logging.info("strSQL_DELETE: "+strSQL_DELETE)
        cursor.execute(strSQL_DELETE)
        connection.commit()
        logging.info("Vidage de la table t_objectif reussi")
        store_cursor = connection.cursor()
        sql_select_store_Query = "SELECT id_magasin, id_enseigne FROM t_magasin ORDER BY 1;"
        store_cursor.execute(sql_select_store_Query)
        # only one fetchall for stores but multiple exhaustive reading of the list
        stores_list = store_cursor.fetchall()
        product_cursor = connection.cursor()
        sql_select_product_Query = "SELECT id_produit, prix FROM t_produit ORDER BY 1;"
        product_cursor.execute(sql_select_product_Query)
        # only one fetchall for products but multiple exhaustive reading of the list
        products_list = product_cursor.fetchall()

        # range s'arrete un avant -> de 2019 a 2023
        for a in range(2019, 2024):
            logging.info("a: "+str(a))
            div_covid = 1
            if a == 2021:
                div_covid = 5
            if a == 2022:
                div_covid = 2
                
            # range s'arrete un avant -> de 1 a 12    
            for m in range(1,13):
                logging.debug("m: "+str(m))
                
                for magasin in stores_list:
                    logging.debug(magasin)
                    taille = 1
                    logging.debug('id_magasin: '+str(magasin[0]))
                    logging.debug('id_enseigne: '+str(magasin[1]))
                    if magasin[0] == 32 or magasin[0] == 33:
                        taille = 10
                    if magasin[0] == 23 or magasin[0] == 24:
                        taille = 6
                    marge_obj = 0.2
                    if magasin[1] == 1:
                        marge_obj = 0.25
                    if magasin[1] == 3:
                        marge_obj = 0.3
                    for produit in products_list:    
                        logging.debug(produit)
                        logging.debug('id_produit: '+str(produit[0]))
                        logging.debug('prix: '+str(produit[1]))

                        nb = int((5 * taille * random()) / div_covid) + 1
                        sql_insert_Query="""INSERT INTO t_objectif (annee, numero_mois, id_magasin, id_produit, objectif_nb_ventes, objectif_ca_ventes, objectif_marge_ventes)
                        VALUES (%i,%i,%i,%i,%d,%d,%d)"""%(a,m,magasin[0],produit[0],decimal.Decimal(nb),produit[1]*decimal.Decimal(nb),produit[1]*decimal.Decimal(nb*marge_obj))
                        logging.debug("sql_insert_Query: "+sql_insert_Query)
                        cursor.execute(sql_insert_Query)
            # validation pour chaque annee
            connection.commit()
            product_cursor.close()
            store_cursor.close()    

            # writing CSV file in notebook directory for each year
            logging.info("Ecriture du fichier csv de l'annee "+str(a))    

            # i must escape % by doubling it ! %M is for MySQL, not for Python !
            query="""SELECT ANNEE,
            DATE_FORMAT_FR(CONCAT(CAST(%i AS CHAR),'-',CAST (numero_mois AS CHAR),'-01'), '%%M') AS MOIS_EN_LETTRES,
            VILLE AS VILLE_MAGASIN,
            LIBELLE AS LIBELLE_PRODUIT,
            OBJECTIF_NB_VENTES,
            OBJECTIF_CA_VENTES,
            OBJECTIF_MARGE_VENTES 
            FROM t_objectif 
            INNER JOIN t_magasin 
            ON t_magasin.ID_MAGASIN=t_objectif.ID_MAGASIN 
            INNER JOIN t_produit 
            ON t_objectif.ID_PRODUIT=t_produit.ID_PRODUIT 
            WHERE ANNEE=%i"""%(a,a)
            logging.info("query: "+query)

            cursor.execute(query)
            with open("%s.csv"%(a), "w", newline='') as csv_file:   
                csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
                csv_writer.writerow([i[0] for i in cursor.description]) # write headers
                csv_writer.writerows(cursor)
    except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
        logging.error("Failed to insert record to database rollback or write file: {}".format(error))
        # annulation des changements pour cause exceptionnelle
        connection.rollback()
    finally:
        # fermeture de la connexion au serveur.
        if connection.is_connected():
            cursor.close()
            connection.close()
            logging.info("La connexion est fermee")



In [ ]:
%%time
# logging.getLogger().setLevel(logging.DEBUG)
logging.getLogger().setLevel(logging.INFO)
insert_t_objectif_create_csv_files()
# Wall time: 1min 47s sur les donnees de l'annee derniere (de 2019 à 2023) pour 243 013 tuples
# Wall time: 23min 44s sur les donnees des alternants (de 2019 à 2023) pour 7 043 014 tuples

#### Difference pour MySQL entre DELETE FROM et TRUNCATE

- TRUNCATE TABLE est un ordre DDL et non un ordre DML comme DELETE. TRUNCATE TABLE entraîne  une validation implicite de la transaction avant l'exécution de vidage de la table. Pour éviter cela, utilisez pour le vidage de table via DELETE FROM à la place de TRUNCATE TABLE.
- L'ordre DELETE FROM tblname; peut faire l'objet d'une annulation (rollback). La restauration peut prendre du temps si les paramétrages du moteur de stockage InnoDB ne sont pas optimaux.
- L'ordre TRUNCATE reinitialise la valeur de AUTO_INCREMENT à 1, ce que ne fait pas DELETE.
- L'ordre TRUNCATE ne peut pas s'exécuter sur une table pointée par une clé étrangère existante.


### Phase 2 de la simulation économique
Le principe pour la seconde étape est de générer soit une ligne, soit deux lignes dans les tables T_FACTURE et T_LIGNE_FACTURE pour réaliser les commandes des clients (normalement rattachés à des régions commerciales). Un magasin ne dépend que d'une seule région commerciale. Le réalisé tient compte des périodes de confinement. On s'appuie sur des modèles (des factures déjà existantes dans une autre table) pour rendre les factures produites plus diverses mais raisonnablement cohérentes. L'employé concerné par la facture dépend du client donc du magasin. Le réalisé est une variation des objectifs donc on se base sur ce qui est dans la table t_objectif qui a été remplie à l'étape précédante.

L'identification par chaine de caractères des factures permet de générer des tuples qui se suivent de manière cohérente dans le temps. 

La génération des factures s'appuiera doc sur des modèles de facture (129) qu'il nous faut ajouter à notre schéma de production partiel.

```
DROP TABLE IF EXISTS t_mod_factures;

CREATE TABLE IF NOT EXISTS t_mod_factures(
num_commande VARCHAR(180) PRIMARY KEY ,
code_devise VARCHAR(3),
id_client INT DEFAULT '0',
id_employe VARCHAR(38),
date_commande_client DATE,
date_livraison_client DATE,
condition_reglement VARCHAR(50),
mode_reglement VARCHAR(50),
statut_commande VARCHAR(50),
mode_livraison VARCHAR(50),
disponibilite VARCHAR(50),
channel VARCHAR(50)
) ENGINE=InnoDB DEFAULT CHARSET=UTF8MB4 ;
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2016-000011','EUR',2544,'19','2016-12-16','2017-01-13','Due on order','Credit card','Commande traitée','Colissimo Suivi','3 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2016-000013','EUR',218,'15','2016-12-31','2017-01-27','Due on order','Credit card','Commande traitée','GLS','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2016-000016','EUR',2314,'15','2016-12-31','2017-01-27','Due on order','Credit card','Commande traitée','Lettre Max','2 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2016-000023','EUR',1988,'18','2016-12-31','2017-01-27','Due on order','Credit card','Commande traitée','Transporteur générique','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2016-000026','EUR',237,'15','2016-12-31','2017-01-13','Due on order','Credit card','Commande traitée','UPS','Immediate','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000005','EUR',2376,'15','2017-01-05','2017-01-20','Due on order','Credit card','Commande traitée','Lettre Max','1 week','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000006','EUR',2233,'19','2017-01-06','2017-01-27','Due on order','Credit card','Commande traitée','Transporteur générique','2 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000007','EUR',1589,'19','2017-01-02','2017-02-03','Due on order','Credit card','Commande traitée','KIALA','3 weeks','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000008','EUR',398,'19','2017-01-02','2017-02-03','Due on order','Credit card','Commande traitée','Transporteur générique','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000009','EUR',2033,'20','2017-01-03','2017-01-13','Due on order','Credit card','Commande traitée','UPS','Immediate','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000010','EUR',2075,'15','2017-01-01','2017-02-03','Due on order','Credit card','Commande traitée','KIALA','3 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000012','EUR',471,'17','2017-01-01','2017-02-03','Due on order','Credit card','Commande traitée','Lettre Max','3 weeks','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000014','EUR',1678,'15','2017-01-05','2017-01-13','Due on order','Credit card','Commande traitée','GLS','Immediate','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000015','EUR',2581,'20','2017-01-01','2017-01-20','Due on order','Credit card','Commande traitée','UPS','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000017','EUR',607,'18','2017-01-02','2017-02-03','Due on order','Credit card','Commande traitée','Colissimo Suivi','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000018','EUR',2706,'20','2017-01-06','2017-02-03','Due on order','Credit card','Commande traitée','UPS','3 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000019','EUR',1410,'20','2017-01-02','2017-01-13','Due on order','Credit card','Commande traitée','Retrait entrepôt','Immediate','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000020','EUR',664,'15','2017-01-02','2017-01-20','Due on order','Credit card','Commande traitée','UPS','1 week','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000021','EUR',2358,'19','2017-01-03','2017-01-20','Due on order','Credit card','Commande traitée','Retrait entrepôt','1 week','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000022','EUR',2234,'18','2017-01-01','2017-01-13','Due on order','Credit card','Commande traitée','Lettre Max','Immediate','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000024','EUR',2009,'20','2017-01-04','2017-01-27','Due on order','Credit card','Commande traitée','KIALA','2 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000025','EUR',1585,'20','2017-01-02','2017-02-03','Due on order','Credit card','Commande traitée','GLS','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000027','EUR',393,'20','2017-01-02','2017-01-27','Due on order','Credit card','Commande traitée','GLS','2 weeks','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000028','EUR',1683,'17','2017-01-07','2017-01-30','Due on order','Credit card','Commande traitée','KIALA','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000029','EUR',661,'19','2017-01-05','2017-02-06','Due on order','Credit card','Commande traitée','Colissimo Suivi','3 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000030','EUR',279,'18','2017-01-06','2017-01-16','Due on order','Credit card','Commande traitée','UPS','Immediate','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000031','EUR',1585,'20','2017-01-08','2017-01-16','Due on order','Credit card','Commande traitée','UPS','Immediate','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000032','EUR',2227,'18','2017-01-05','2017-01-16','Due on order','Credit card','Commande traitée','Lettre Max','Immediate','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000033','EUR',1247,'15','2017-01-06','2017-02-06','Due on order','Credit card','Commande validée','Lettre Max','3 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000034','EUR',2371,'20','2017-01-07','2017-01-23','Due on order','Credit card','Commande validée','Transporteur générique','1 week','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000035','EUR',1620,'15','2017-01-08','2017-01-30','Due on order','Credit card','Commande validée','Chronopost','2 weeks','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000036','EUR',1858,'17','2017-01-07','2017-01-16','Due on order','Credit card','Commande traitée','UPS','Immediate','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000037','EUR',1000,'20','2017-01-08','2017-01-18','Due on order','Credit card','Commande traitée','Retrait entrepôt','Immediate','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000038','EUR',401,'20','2017-01-09','2017-02-01','Due on order','Credit card','Commande traitée','Transporteur générique','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000039','EUR',2565,'20','2017-01-06','2017-01-18','Due on order','Credit card','Commande traitée','Chronopost','Immediate','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000040','EUR',261,'20','2017-01-06','2017-02-08','Due on order','Credit card','Commande traitée','Chronopost','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000041','EUR',997,'15','2017-01-05','2017-02-08','Due on order','Credit card','Commande traitée','UPS','3 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000042','EUR',1826,'20','2017-01-10','2017-02-08','Due on order','Credit card','Commande traitée','Transporteur générique','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000043','EUR',2355,'18','2017-01-07','2017-02-08','Due on order','Credit card','Commande traitée','Colissimo Suivi','3 weeks','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000044','EUR',1847,'18','2017-01-09','2017-01-25','Due on order','Credit card','Commande traitée','Transporteur générique','1 week','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000045','EUR',1388,'15','2017-01-07','2017-01-18','Due on order','Credit card','Commande traitée','KIALA','Immediate','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000046','EUR',2364,'15','2017-01-05','2017-01-18','Due on order','Credit card','Commande traitée','Colissimo Suivi','Immediate','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000047','EUR',406,'19','2017-01-05','2017-01-18','Due on order','Credit card','Commande traitée','Retrait entrepôt','Immediate','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000048','EUR',1855,'15','2017-01-08','2017-02-08','Due on order','Credit card','Commande traitée','Colissimo Suivi','3 weeks','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000049','EUR',1658,'20','2017-01-07','2017-01-25','Due on order','Credit card','Commande traitée','Lettre Max','1 week','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000050','EUR',1454,'20','2017-01-06','2017-01-25','Due on order','Credit card','Commande validée','Retrait entrepôt','1 week','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000051','EUR',1880,'19','2017-01-11','2017-02-08','Due on order','Credit card','Commande en cours','Lettre Max','3 weeks','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000052','EUR',989,'20','2017-01-08','2017-02-01','Due on order','Credit card','Commande validée','KIALA','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000053','EUR',218,'15','2017-01-09','2017-01-18','Due on order','Credit card','Commande traitée','Transporteur générique','Immediate','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000054','EUR',2762,'17','2017-01-05','2017-02-01','Due on order','Credit card','Commande validée','Chronopost','2 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000055','EUR',1879,'15','2017-01-11','2017-01-25','Due on order','Credit card','Commande validée','GLS','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000056','EUR',3097,'18','2017-01-09','2017-02-01','Due on order','Credit card','Commande validée','GLS','2 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000057','EUR',453,'19','2017-01-10','2017-02-08','Due on order','Credit card','Commande validée','Colissimo Suivi','3 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000058','EUR',1423,'15','2017-01-08','2017-02-08','Due on order','Credit card','Commande validée','Retrait entrepôt','3 weeks','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000059','EUR',2773,'18','2017-01-06','2017-01-25','Due on order','Credit card','Commande validée','Colissimo Suivi','1 week','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000060','EUR',2907,'15','2017-01-08','2017-02-08','Due on order','Credit card','Commande validée','Lettre Max','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000061','EUR',853,'20','2017-01-17','2017-02-08','Due on order','Credit card','Commande traitée','Chronopost','2 weeks','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000062','EUR',1606,'20','2017-01-12','2017-02-01','Due on order','Credit card','Commande traitée','KIALA','1 week','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000063','EUR',1413,'15','2017-01-14','2017-02-01','Due on order','Credit card','Commande validée','Colissimo Suivi','1 week','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000064','EUR',1585,'20','2017-01-17','2017-02-08','Due on order','Credit card','Commande validée','Chronopost','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000065','EUR',2356,'17','2017-01-16','2017-02-15','Due on order','Credit card','Commande validée','Colissimo Suivi','3 weeks','Word of mouth');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000066','EUR',2267,'20','2017-01-16','2017-02-15','Due on order','Credit card','Commande validée','Retrait entrepôt','3 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000067','EUR',2718,'15','2017-01-17','2017-02-15','Due on order','Credit card','Commande validée','UPS','3 weeks','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000068','EUR',2978,'20','2017-01-18','2017-01-25','Due on order','Credit card','Commande validée','Chronopost','Immediate','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000069','EUR',2190,'15','2017-01-15','2017-02-08','Due on order','Credit card','Commande validée','Lettre Max','2 weeks','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000070','EUR',2937,'15','2017-01-15','2017-01-25','Due on order','Credit card','Commande validée','Retrait entrepôt','Immediate','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000071','EUR',3136,'18','2017-01-14','2017-02-15','Due on order','Credit card','Commande validée','Transporteur générique','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000072','EUR',787,'15','2017-01-13','2017-02-01','Due on order','Credit card','Commande validée','GLS','1 week','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000073','EUR',2953,'18','2017-01-18','2017-02-13','Due on order','Credit card','Commande validée','Transporteur générique','2 weeks','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000074','EUR',206,'20','2017-01-22','2017-02-06','Due on order','Credit card','Commande validée','Transporteur générique','1 week','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000075','EUR',2905,'20','2017-01-21','2017-02-13','Due on order','Credit card','Commande validée','GLS','2 weeks','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000076','EUR',2960,'20','2017-01-19','2017-02-13','Due on order','Credit card','Commande validée','Transporteur générique','2 weeks','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000077','EUR',1227,'19','2017-01-22','2017-02-06','Due on order','Credit card','Commande validée','Retrait entrepôt','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000078','EUR',427,'18','2017-01-22','2017-02-20','Due on order','Credit card','Commande validée','Chronopost','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000079','EUR',594,'20','2017-01-21','2017-02-20','Due on order','Credit card','Commande validée','Transporteur générique','3 weeks','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000080','EUR',1420,'15','2017-01-17','2017-02-06','Due on order','Credit card','Commande validée','Lettre Max','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000081','EUR',2502,'15','2017-01-22','2017-01-30','Due on order','Credit card','Commande validée','KIALA','Immediate','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000082','EUR',2522,'20','2017-01-18','2017-02-06','Due on order','Credit card','Commande validée','Retrait entrepôt','1 week','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000083','EUR',2780,'20','2017-01-22','2017-02-13','Due on order','Credit card','Commande validée','Colissimo Suivi','2 weeks','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000084','EUR',2372,'15','2017-01-20','2017-02-20','Due on order','Credit card','Commande validée','Chronopost','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000085','EUR',664,'15','2017-01-19','2017-02-20','Due on order','Credit card','Commande validée','Transporteur générique','3 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000086','EUR',3179,'20','2017-01-21','2017-02-13','Due on order','Credit card','Commande validée','UPS','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000087','EUR',2760,'15','2017-01-23','2017-02-13','Due on order','Credit card','Commande validée','GLS','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000088','EUR',619,'15','2017-01-19','2017-02-20','Due on order','Credit card','Commande validée','KIALA','3 weeks','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000089','EUR',1663,'15','2017-01-21','2017-01-30','Due on order','Credit card','Commande validée','Colissimo Suivi','Immediate','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000090','EUR',2060,'18','2017-01-18','2017-02-13','Due on order','Credit card','Commande validée','Retrait entrepôt','2 weeks','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000091','EUR',1060,'15','2017-01-20','2017-02-06','Due on order','Credit card','Commande validée','KIALA','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000092','EUR',1414,'15','2017-01-21','2017-02-06','Due on order','Credit card','Commande validée','UPS','1 week','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000093','EUR',998,'18','2017-02-03','2017-02-20','Due on order','Credit card','Commande validée','GLS','1 week','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000094','EUR',3109,'15','2017-02-02','2017-02-27','Due on order','Credit card','Commande validée','GLS','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000095','EUR',1863,'15','2017-02-04','2017-02-13','Due on order','Credit card','Commande validée','Retrait entrepôt','Immediate','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000096','EUR',415,'17','2017-02-03','2017-02-27','Due on order','Credit card','Commande validée','Lettre Max','2 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000097','EUR',1206,'20','2017-02-05','2017-02-13','Due on order','Credit card','Commande validée','Colissimo Suivi','Immediate','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000098','EUR',2488,'20','2017-02-03','2017-03-06','Due on order','Credit card','Commande validée','GLS','3 weeks','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000099','EUR',2984,'17','2017-02-02','2017-03-06','Due on order','Credit card','Commande validée','Retrait entrepôt','3 weeks','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000100','EUR',795,'18','2017-02-01','2017-02-13','Due on order','Credit card','Commande validée','Transporteur générique','Immediate','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000101','EUR',274,'15','2017-02-02','2017-02-13','Due on order','Credit card','Commande validée','Lettre Max','Immediate','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000102','EUR',2320,'15','2017-01-31','2017-02-20','Due on order','Credit card','Commande validée','Transporteur générique','1 week','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000103','EUR',1789,'15','2017-02-03','2017-03-06','Due on order','Credit card','Commande validée','Colissimo Suivi','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000104','EUR',188,'17','2017-02-05','2017-02-13','Due on order','Credit card','Commande validée','UPS','Immediate','Shop contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000105','EUR',3110,'15','2017-02-06','2017-02-13','Due on order','Credit card','Commande validée','Transporteur générique','Immediate','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000106','EUR',284,'20','2017-02-01','2017-02-27','Due on order','Credit card','Commande validée','Transporteur générique','2 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000107','EUR',3123,'20','2017-02-02','2017-02-27','Due on order','Credit card','Commande validée','Colissimo Suivi','2 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000108','EUR',2741,'17','2017-02-04','2017-03-06','Due on order','Credit card','Commande validée','Colissimo Suivi','3 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000109','EUR',1242,'15','2017-02-05','2017-02-13','Due on order','Credit card','Commande validée','Chronopost','Immediate','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000110','EUR',3176,'15','2017-02-06','2017-03-06','Due on order','Credit card','Commande validée','Retrait entrepôt','3 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000111','EUR',1827,'15','2017-01-31','2017-02-20','Due on order','Credit card','Commande validée','Lettre Max','1 week','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000112','EUR',1661,'17','2017-02-03','2017-02-27','Due on order','Credit card','Commande validée','UPS','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000113','EUR',633,'19','2017-01-31','2017-03-06','Due on order','Credit card','Commande validée','Transporteur générique','3 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000114','EUR',852,'19','2017-02-03','2017-03-06','Due on order','Credit card','Commande validée','Retrait entrepôt','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000115','EUR',450,'15','2017-02-03','2017-02-27','Due on order','Credit card','Commande validée','KIALA','2 weeks','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000116','EUR',1257,'15','2017-02-01','2017-02-20','Due on order','Credit card','Commande validée','Colissimo Suivi','1 week','Partner');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000117','EUR',2738,'18','2017-02-06','2017-02-13','Due on order','Credit card','Commande validée','KIALA','Immediate','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000118','EUR',2783,'15','2017-02-04','2017-03-06','Due on order','Credit card','Commande validée','GLS','3 weeks','Phone campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000119','EUR',459,'20','2017-02-01','2017-02-27','Due on order','Credit card','Commande validée','Transporteur générique','2 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000120','EUR',2340,'15','2017-02-02','2017-03-06','Due on order','Credit card','Commande validée','KIALA','3 weeks','Employee');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000121','EUR',641,'15','2017-01-18','2017-02-13','Due on order','Credit card','Commande validée','Chronopost','2 weeks','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000122','EUR',2070,'20','2017-01-18','2017-02-06','Due on order','Credit card','Commande validée','Retrait entrepôt','1 week','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000123','EUR',1665,'20','2017-01-18','2017-01-30','Due on order','Credit card','Commande traitée','Colissimo Suivi','Immediate','Mailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000124','EUR',2037,'15','2017-01-19','2017-02-06','Due on order','Credit card','Commande validée','Colissimo Suivi','1 week','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000125','EUR',820,'19','2017-01-18','2017-02-20','Due on order','Credit card','Commande validée','UPS','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000126','EUR',809,'19','2017-01-18','2017-02-13','Due on order','Credit card','Commande validée','UPS','2 weeks','Commercial contact');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000127','EUR',848,'18','2017-01-18','2017-02-20','Due on order','Credit card','Commande validée','Chronopost','3 weeks','Sponsorship');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000128','EUR',1598,'20','2017-01-21','2017-01-30','Due on order','Credit card','Commande traitée','Colissimo Suivi','Immediate','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000129','EUR',1635,'15','2017-01-20','2017-02-20','Due on order','Credit card','Commande validée','GLS','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000130','EUR',1831,'15','2017-01-21','2017-01-30','Due on order','Credit card','Commande traitée','Retrait entrepôt','Immediate','Web site');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000131','EUR',405,'20','2017-01-22','2017-02-20','Due on order','Credit card','Commande validée','UPS','3 weeks','Fax campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000132','EUR',412,'20','2017-01-23','2017-02-06','Due on order','Credit card','Commande validée','Colissimo Suivi','1 week','EMailing campaign');
INSERT IGNORE INTO t_mod_factures VALUES('CDE- 2017-000133','EUR',3127,'20','2017-01-23','2017-01-30','Due on order','Credit card','Commande traitée','Colissimo Suivi','Immediate','Sponsorship');

```


In [ ]:
import mysql.connector
import decimal
import datetime
#from random import *
from random import random

def insert_reel():
    
    connection = None
    try:
        connection = mysql.connector.connect(
         host='localhost',
         port=3312,
         database='e_23_3jexxxxxxx',
         user='e_23_3jexxxxxxx',
         password='de_23_3jexxxxxxxd',
         auth_plugin='mysql_native_password'
        )
        connection.autocommit = False
        cursor = connection.cursor()
        # Vidage de la table t_ligne_facture
        # MS Access n'ccepte que cela
        strSQL_DELETE = "DELETE FROM t_ligne_facture;"
        # MySQL considère que c'est un ordre DDL et non MDL
        strSQL_DELETE = "TRUNCATE t_ligne_facture;"
        logging.info("strSQL_DELETE: "+strSQL_DELETE)
        cursor.execute(strSQL_DELETE)
        logging.info("Vidage de la table t_ligne_facture reussi")

        # Vidage de la table t_facture
        # MS Access n'ccepte que cela
        strSQL_DELETE = "DELETE FROM t_facture;"
        # MySQL considère que c'est un ordre DDL et non MDL
        #strSQL_DELETE = "TRUNCATE t_facture;"
        logging.info("strSQL_DELETE: "+strSQL_DELETE)
        cursor.execute(strSQL_DELETE)
        logging.info("Vidage de la table t_facture reussi")

        connection.commit()
        
        store_cursor = connection.cursor()
        sql_select_store_Query = "SELECT id_magasin, id_enseigne,dep FROM t_magasin ORDER BY 1;"
        store_cursor.execute(sql_select_store_Query)
        # only one fetchall for stores but multiple exhaustive reading of the list
        stores_list = store_cursor.fetchall()
        product_cursor = connection.cursor()
        sql_select_product_Query = "SELECT id_produit, prix FROM t_produit ORDER BY 1;"
        product_cursor.execute(sql_select_product_Query)
        # only one fetchall for products but multiple exhaustive reading of the list
        products_list = product_cursor.fetchall()
        order_template_cursor = connection.cursor()
        sql_select_order_template_Query = "SELECT date_commande_client,date_livraison_client,condition_reglement,mode_reglement,statut_commande,mode_livraison,disponibilite,channel FROM t_mod_factures;"
        order_template_cursor.execute(sql_select_order_template_Query)
        # only one fetchall for products but multiple exhaustive reading of the list
        order_templates_list = order_template_cursor.fetchall()

        
        #open cursorS in advance
        store_customer_cursor = connection.cursor()
        goal_cursor = connection.cursor()
        
        # range s'arrete un avant -> de 2019 a 2023
        for a in range(2019, 2024):
            logging.info("a: "+str(a))
                
            # range s'arrete un avant -> de 1 a 12    
            for m in range(1,13):
                # logging.info("m: "+str(m))
                
                for magasin in stores_list:
                    logging.debug(magasin)
                    logging.debug('id_magasin: '+str(magasin[0]))
                    logging.debug('id_enseigne: '+str(magasin[1]))
                    marge_obj = 0.2
                    if magasin[1] == 1:
                        marge_obj = 0.25
                    if magasin[1] == 3:
                        marge_obj = 0.3
                    sql_select_store_customer_Query = """SELECT t_client.id_client, t_employe.id_employe 
                    FROM (t_employe 
                    INNER JOIN t_client 
                    ON t_employe.id_employe = t_client.code_responsable) 
                    INNER JOIN t_departement
                    ON t_employe.entite = t_departement.id_region_com 
                    WHERE t_departement.id_departement=%s"""%(str(magasin[2]))
                    store_customer_cursor.execute(sql_select_store_customer_Query)
                    stores_customers_list = store_customer_cursor.fetchall()

                    for produit in products_list:    
                        logging.debug(produit)
                        logging.debug('id_produit: '+str(produit[0]))
                        logging.debug('prix: '+str(produit[1]))

                        sql_select_goal_Query = """SELECT objectif_nb_ventes,objectif_ca_ventes,objectif_marge_ventes 
                        FROM t_objectif 
                        WHERE annee=%i AND numero_mois=%i AND id_magasin=%i AND id_produit=%i;"""%(a,m,magasin[0],produit[0])
                        goal_cursor.execute(sql_select_goal_Query)
                        # un seul tuple car restriction sur la totalite de la cle primaire
                        goal = goal_cursor.fetchone()
                        nb = goal[0] + int(1 + 3 * random()) - int(1 + 3 * random())
                        if nb < 0: 
                            nb = 0
                        # les differentes periode de confinement en France    
                        if ((m >= 3 and m <= 5) or (m == 11 or m == 12)) and a == 2020:
                            nb = 0
                        if m == 4 and a == 2021:
                            nb = 0
            
                        if nb != 0:
                            # une ou deux factures par mois,magasin,produit, avec des clients compatibles avec le magasin mais eventuellement different?
                            f = 0
                            while True:
                                if nb % 2 == 0: 
                                    qte = nb / 2
                                    f_end = 2
                                else: 
                                    qte = nb
                                    f_end = 1
                            
                                if f==f_end:
                                    break
                                while True:
                                    # on evite la difficulte de la variation des nombre de jour dans les mois
                                    d = int(1 + 28 * random())
                                    date_f = datetime.date(a,m,d)
                                    # le dimanche, les magasins sont fermes
                                    if date_f.weekday()!=6:
                                        break
                                # parmi les modeles de facture possibles, j'en choisi un au hasard
                                order_template = order_templates_list[int(len(order_templates_list) * random())]
                                        
                                # parmi les clients possibles pour le magasin, j'en choisi un au hasard
                                store_customer = stores_customers_list[int(len(stores_customers_list) * random())]
                                # la numerotation de facture ne depend pas de l'ordre de generation de celle-ci
                                id = "CDE-{:04d}{:02d}{:02d}{:02d}{:04d}{:01d}".format(a, m,d,magasin[0],produit[0],f)
                                #delai de livraison conforme au modele courant
                                delta = order_template[1] - order_template[0]
                                delivery_date = date_f + datetime.timedelta(days=delta.days)
                                taux_tva = 0.2
                                sql_insert_Query="""INSERT INTO t_facture
                                (num_commande, code_devise, id_client, id_employe, id_magasin, date_commande_client, date_livraison_client, condition_reglement, mode_reglement, statut_commande, mode_livraison, disponibilite, channel)
                                VALUES ('%s','%s',%i,%i,%i,'%s','%s','%s','%s','%s','%s','%s','%s')"""%(id,'EUR',store_customer[0],store_customer[1],magasin[0],date_f.strftime("%Y-%m-%d"),delivery_date.strftime("%Y-%m-%d"),order_template[2],order_template[3],order_template[4],order_template[5],order_template[6],order_template[7])
                                logging.debug("sql_insert_Query: "+sql_insert_Query)
                                cursor.execute(sql_insert_Query)
                                sql_insert_Query="""INSERT INTO  t_ligne_facture(num_commande, id_ligne, taux_tva, quantite, pu_ht, prix_revient, reduction, code_devise, id_produit) 
                                VALUES ('%s',%i,%f,%d,%d,%f,%d,'%s',%i)"""%(id,1,decimal.Decimal(taux_tva),decimal.Decimal(qte),produit[1],produit[1]*decimal.Decimal(1 - marge_obj),0,'EUR',produit[0])
                                logging.debug("sql_insert_Query: "+sql_insert_Query)
                                cursor.execute(sql_insert_Query)
                                f = f + 1

            # validation pour chaque annee
            connection.commit()
        
        # closing cursors
        goal_cursor.close()
        order_template_cursor.close()
        store_customer_cursor.close()
        product_cursor.close()
        store_cursor.close()
    except (mysql.connector.Error) as error:
        logging.error("Failed to insert record to database rollback: {}".format(error))
        # annulation des changements pour cause exceptionnelle
        connection.rollback()

    finally:
        # fermeture de la connexion au serveur.
        if connection.is_connected():
            cursor.close()
            connection.close()
            logging.info("La connexion est fermee")
            


In [ ]:
%%time
# logging.getLogger().setLevel(logging.DEBUG)
logging.getLogger().setLevel(logging.INFO)
insert_reel()
# Pour les donnees des alternants, Wall time: 2h 5min 3s pour 8 058 988 tuples dans t_facture et 8 136 446 dans t_ligne_facture
# 2019 : 44 min, 2020 : 14 min (confinement=>pas de tuples), 2021 : 16 min, 2022 : 20 minutes, 2023 : 22 minutes
# Pour les donnees de l'annee derniere, Wall time: 9min 10s  (de 2019 à 2023) pour 693 739 tuples (t_facture et t_ligne_facture)

A noter que Python 3.6 offre les f-string pour l'interpollation et un enrrichissement du module `random` avec notamment `randint` et `choice` qui fait un tirage aléatoire au sein de la liste sans remise => les 129 modèles de factures sont vite épuisés !

En conclusion, même si la simulation économique n'est pas à la charge des alternants, sa programmation en python constitue deux exemples riches de processus ETL. A méditer !


## Processus ETL simplifié en Python
Nous montrerons comment créer un travail ETL qui extrait les données d'une base de données de production sur le serveur MySQL et les charge dans un entrepôt de données dans une autre base de données de ce même serveur MySQL. Nous mettrons également en œuvre le concept *Change Data Capture* (CDC) pour capturer les changements delta et déclencher cette tâche ETL toutes les heures.

On va ajouter à la table Commandes, deux nouveaux attributs `create_date` et `update_date`. On travaille sur la base de données `fnuc_python_script`. Dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=fnuc_python_script&table=commandes&server=2), il faut faire :
```
ALTER TABLE Commandes ADD 
create_date TIMESTAMP NOT NULL default CURRENT_TIMESTAMP COMMENT 'Instant de creation de la commande';

ALTER TABLE Commandes ADD 
update_date TIMESTAMP NULL on update CURRENT_TIMESTAMP COMMENT 'Instant de modification de la commande';
```

De plus, on a une base de données fnuc_python_OLAP qui ne comporte deux versions d'une seule table faits_commandes :
Au préalable, dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?server=2), il faut faire ;
1. ```DROP DATABASE IF EXISTS `fnuc_python_OLAP`;```
2. ```CREATE DATABASE `fnuc_python_OLAP` COLLATE utf8mb4_0900_ai_ci ;```
3. ```GRANT ALL PRIVILEGES ON `fnuc\_python\_OLAP`.* TO 'python'@'localhost';```

Enfin dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=fnuc_python_OLAP&server=2), il faut faire :
```
CREATE TABLE IF NOT EXISTS Faits_commandes
 (
   DATECOM DATE NOT NULL DEFAULT (CURRENT_DATE) COMMENT 'Date des commandes' ,
   ARTICLE INT NOT NULL COMMENT 'Reference de l''article commandee' ,
   CLIENT INT NOT NULL COMMENT 'Reference du client' ,
   NOMBRE_VENTE DECIMAL NOT NULL COMMENT 'Cumul des ventes',
   CA DECIMAL NOT NULL COMMENT 'Chiffre d''affaires cumule',
   create_date TIMESTAMP NOT NULL default CURRENT_TIMESTAMP COMMENT 'Instant de creation du fait',
   update_date TIMESTAMP NULL on update CURRENT_TIMESTAMP COMMENT 'Instant de modification du fait',
   INDEX FKIndex_Article(ARTICLE),
   INDEX FKIndex_Client(CLIENT),
   PRIMARY KEY (DATECOM,ARTICLE,CLIENT) 
 )
ENGINE=InnoDB
ROW_FORMAT=default
COMMENT='Les indicateurs sur les commandes'
;

CREATE TABLE IF NOT EXISTS Faits_commandes_SCD2
 (
   DATECOM DATE NOT NULL DEFAULT (CURRENT_DATE) COMMENT 'Date des commandes' ,
   ARTICLE INT NOT NULL COMMENT 'Reference de l''article commandee' ,
   CLIENT INT NOT NULL COMMENT 'Reference du client' ,
   NOMBRE_VENTE DECIMAL NOT NULL COMMENT 'Cumul des ventes',
   CA DECIMAL NOT NULL COMMENT 'Chiffre d''affaires cumule',
   create_date TIMESTAMP NOT NULL default CURRENT_TIMESTAMP COMMENT 'Instant de creation du fait',
   update_date TIMESTAMP NULL on update CURRENT_TIMESTAMP COMMENT 'Instant de modification du fait',
   end_date TIMESTAMP NULL COMMENT 'Instant de fin de validite du fait',
   INDEX FKIndex_Article(ARTICLE),
   INDEX FKIndex_Client(CLIENT) 
 )
ENGINE=InnoDB
ROW_FORMAT=default
COMMENT='Les indicateurs sur les commandes avec prise en compte d''une SCD de type 2'
;
```
Si vous souhaitez conserver les données historiques, vous devrez modifier le travail ETL pour en tenir compte. Une approche courante consiste à créer une dimension à évolution lente ou *Slowly Changing Dimension* (SCD) dans notre entrepôt de données MySQL. Il existe plusieurs types de SCD, mais les deux plus courants sont :
- Type 1 : écrasez les anciennes données par de nouvelles modifications, perdant ainsi l'historique des modifications.
- Type 2 : conservez un historique complet des modifications des données dans la dimension.

De fait, c'est plus compliqué : pour voir les 7 types SCD, il faut consulter [ce document](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/MSID_02.pdf#page=23). On pourra consulter d'abord la [première partie](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/MSID_01.pdf#page=1) de ce cours sur la modélisation décisionnelle.

Dans cet exemple, la table `Faits_commandes_SCD2` a une colonne `end_date` supplémentaire. Lorsqu'un nouvel enregistrement est chargé, la tâche ETL vérifie si l'enregistrement existe déjà dans la table. Si tel est le cas, la tâche définit la date de fin de l'enregistrement actuel sur la date actuelle, marquant la fin de la validité de l'enregistrement. Ensuite, la tâche insère le nouvel enregistrement, avec la date de fin définie sur NULL pour indiquer que cet enregistrement est l'enregistrement valide actuel. Bien sûr, cette table n'a pas de clé primaire déclarée.

On vous demande de créer une table de même structure que `Faits_commandes` mais de nom `Faits_commandes_luigi` dans la base de données `fnuc_python_OLAP` afin d'éviter des conflits entre l'ETL avec python classique, l'ETL avec `pandas` et l'ETL avec `luigi`.

### Étape 1 : extraction des données


In [ ]:
%%writefile etl_job.py
import mysql.connector
from datetime import datetime
import logging

logging.getLogger().setLevel(logging.DEBUG)

# set last_update to the start of epoch
last_update = datetime(1970, 1, 1)

def extract_new_records():
    global last_update
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python_script',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )

    cursor = connection.cursor()
    query = """SELECT DATECOM,ARTICLE,CLIENT,SUM(QUANTITE), SUM(PRIX*QUANTITE), MAX(create_date), MAX(update_date) 
               FROM Commandes 
               INNER JOIN Livres
               ON Commandes.ARTICLE=Livres.ID
               WHERE update_date > '%s'
               GROUP BY DATECOM,ARTICLE,CLIENT
               ORDER BY 1,2,3
               """%(last_update.strftime("%Y-%m-%d %H:%M:%S"))
    cursor.execute(query)
    logging.info("query: "+query)
    records = cursor.fetchall()
    last_update = datetime.now()
    cursor.close()
    connection.close()
    return records



In [ ]:

extract_new_records()

### Étape 2 : Chargement des données
L'étape suivante consiste à charger les données extraites dans le schéma décisionnel.

In [ ]:
%%writefile -a etl_job.py

import mysql.connector
def load_to_OLAP(records):
    connection = mysql.connector.connect(
	 host='localhost',
	 port=3312,
	 database='fnuc_python_OLAP',
	 user='python',
	 password='VOTRE_MDP',
	 auth_plugin='mysql_native_password'
    )

    cursor = connection.cursor()
    for record in records:
        insert_query = """INSERT INTO Faits_commandes(DATECOM,ARTICLE,CLIENT,NOMBRE_VENTE,CA,create_date,update_date) 
        VALUES ('%s',%i,%i,%f,%f,'%s','%s')"""%(record[0].strftime("%Y-%m-%d"),
                                          record[1],record[2],record[3],record[4],
                                          record[5].strftime("%Y-%m-%d %H:%M:%S"),
                                          record[6].strftime("%Y-%m-%d %H:%M:%S"))
        logging.info("insert_query: "+insert_query)
        cursor.execute(insert_query)
    connection.commit()
    cursor.close()
    connection.close()
    

On peut alors définir le processus ETL comme étant composé de 2 tâches ordonnées :

In [ ]:
%%writefile -a etl_job.py

def etl_job():
    records = extract_new_records()
    load_to_OLAP(records)

On peut faire un test :

In [ ]:
etl_job()

### Étape 3 : Planifier la tâche ETL
Maintenant que nous disposons de nos fonctions d'extraction et de chargement, nous devons planifier notre tâche ETL pour qu'elle s'exécute toutes les heures. Nous pouvons utiliser la bibliothèque `schedule` pour cela :

In [ ]:
!pip install schedule

In [ ]:
%%writefile -a etl_job.py

import schedule
import time

# Schedule the job every hour
schedule.every(1).hours.do(etl_job)

# Keep the script running
while True:
    schedule.run_pending()
    time.sleep(1)


Malheureuement, ceci ne convient pas pour une exécution depuis un calepin jupyter.

Étapes de développement local :
1. Installez les bibliothèques requises : `mysql-connector-python`, `schedule` dans votre shell Python.
2. Enregistrez le code ci-dessus dans un fichier Python, nommé par exemple `etl_job.py`.
3. Mettez à jour les détails de connexion à MySQL dans les fonctions `extract_new_records` et `load_to_OLAP`.
4. Exécutez le script : Exécutez `python etl_job.py` dans votre shell Python pour démarrer le *job* ou processus ETL.


## Processus ETL simplifié avec Pandas

Tout d’abord, créons un fichier `config.py` pour nos informations d’identification de base de données :

In [ ]:
%%writefile config.py
MYSQL_OLTP_CREDENTIALS = {
    'user': 'python',
    'password': 'python',
    'host': 'localhost',
    'port': 3312,
    'database': 'fnuc_python_script',
    'auth_plugin' : 'mysql_native_password'
}

MYSQL_OLAP_CREDENTIALS = {
    'user': 'python',
    'password': 'python',
    'host': 'localhost',
    'port': 3312,
    'database': 'fnuc_python_OLAP',
    'auth_plugin' : 'mysql_native_password'
}
# There are built-in Magic commands in Jupyter:
# IPython has a set of predefined ‘magic functions’ that you can call with a command line style syntax. There are two kinds of magics, line-oriented and cell-oriented. Line magics are prefixed with the % character and work much like OS command-line calls: they get as an argument the rest of the line, where arguments are passed without parentheses or quotes. Lines magics can return results and can be used in the right hand side of an assignment. Cell magics are prefixed with a double %%, and they are functions that get as an argument not only the rest of the line, but also the lines below it in a separate argument.
# Magics are useful as convenient functions where Python syntax is not the most natural one, or when one want to embed invalid python syntax in their work flow.




Maintenant, créez une tâche ETL avec Pandas :

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime
from urllib.parse import quote_plus
from config import MYSQL_OLTP_CREDENTIALS, MYSQL_OLAP_CREDENTIALS

def etl_job_pandas():
    # Create a connection string for OLTP and OLAP databases on MySQL server
    mysql_oltp_conn_str = "mysql://%s:%s@%s:%i/%s"%(MYSQL_OLTP_CREDENTIALS['user'],quote_plus(MYSQL_OLTP_CREDENTIALS['password']),MYSQL_OLTP_CREDENTIALS['host'],MYSQL_OLTP_CREDENTIALS['port'],MYSQL_OLTP_CREDENTIALS['database'])
    mysql_olap_conn_str = "mysql://%s:%s@%s:%i/%s"%(MYSQL_OLAP_CREDENTIALS['user'],quote_plus(MYSQL_OLAP_CREDENTIALS['password']),MYSQL_OLAP_CREDENTIALS['host'],MYSQL_OLAP_CREDENTIALS['port'],MYSQL_OLAP_CREDENTIALS['database'])


    # Create a MySQL connection for OLTP database
    mysql_oltp_engine = create_engine(mysql_oltp_conn_str)

    # Extract
    with open('last_update_pandas.txt', 'r') as f:
        last_update = datetime.strptime(f.read().strip(),'%Y-%m-%d %H:%M:%S')
        
    query = """SELECT DATECOM,ARTICLE,CLIENT,SUM(QUANTITE) AS NOMBRE_VENTE, SUM(PRIX*QUANTITE) AS CA, MAX(create_date) AS create_date, MAX(update_date) AS update_date
               FROM Commandes 
               INNER JOIN Livres
               ON Commandes.ARTICLE=Livres.ID
               WHERE update_date > '%s'
               GROUP BY DATECOM,ARTICLE,CLIENT
               ORDER BY 1,2,3
               """%(last_update.strftime("%Y-%m-%d %H:%M:%S"))
    logging.info("query: "+query)
    source_df = pd.read_sql_query(query, mysql_oltp_engine)

    # Transform (if needed)
    # ...

    # Create a connection for OLAP database
    mysql_olap_engine = create_engine(mysql_olap_conn_str)

    # Load
    source_df.to_sql('Faits_commandes_pandas', mysql_olap_engine, if_exists='append', index=False)

    # Update last_update
    with open('last_update.txt', 'w') as f:
        f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

On reprend les enseignements d'un calepin précédent concernant `sqlalchemy` :
1. On utilise le connecteur alternatif `mysqlclient` ;
2. On encode le mot de passe pour empécher que la présence de @ en son sein provoque une erreur

Dans ce script, la fonction `etl_job_pandas` se connecte d'abord aux bases de données OLTP et OLAP du serveur MySQL. Ensuite, il lit les données OLTP dans un Pandas DataFrame via la fonction `read_sql_query`. Il écrit le DataFrame dans la base de données OLAP à l'aide de la fonction `to_sql`, qui ajoute les nouvelles données aux données existantes.

Enfin, le script met à jour le fichier nommé `last_update_pandas.txt` avec l'horodatage actuel. Vous pouvez planifier l'exécution de ce script toutes les heures à l'aide de cron ou d'un outil de planification similaire.

Veuillez noter que ce script suppose que la structure de votre table « commandes » dans OLTP et OLAP est la même. Si ce n'est pas le cas, vous devrez peut-être transformer les données pour qu'elles correspondent au schéma de destination.

On peut faire un test de la procedure `etl_job_pandas`.
Avant, il faut créer le fichier `last_update.txt` avec une date correspondant au 1er janvier 1970.

In [ ]:
!pip show sqlalchemy
#!pip install sqlalchemy==1.3.18
#!pip uninstall -y mysqlclient
#!pip install --only-binary :all: mysqlclient

In [ ]:
from datetime import datetime

with open('last_update_pandas.txt', 'w') as f:
        f.write(datetime(1970, 1, 1).strftime("%Y-%m-%d %H:%M:%S"))
etl_job_pandas()

## Processus ETL simplifié avec Luigi

Luigi est un module Python développé par Spotify qui permet de créer des pipelines complexes de tâches par lots. Voici comment implémenter un processus ETL avec Luigi qui extrait les données d'une base de données MySQL, qui implémente le *Change Data Capture* (CDC) et charge ces données dans un entrepôt de données MySQL.

Tout d’abord, créons un fichier `config.py` pour nos informations d’identification de base de données, comme précédemment.

Maintenant, créons nos tâches Luigi dans un fichier nommé `etl_tasks_luigi.py` :

In [ ]:
%%writefile etl_tasks_luigi.py
import luigi
import mysql.connector
from datetime import datetime
from config import MYSQL_OLTP_CREDENTIALS, MYSQL_OLAP_CREDENTIALS


class ExtractFromOLTP(luigi.Task):
    def output(self):
        return luigi.LocalTarget('data.txt')

    def run(self):
        connection = mysql.connector.connect(**MYSQL_OLTP_CREDENTIALS)
        cursor = connection.cursor()

        with open('last_update_luigi.txt', 'r') as f:
            last_update = datetime.strptime(f.read().strip(),'%Y-%m-%d %H:%M:%S')

        query = """SELECT DATECOM,ARTICLE,CLIENT,SUM(QUANTITE), SUM(PRIX*QUANTITE), MAX(create_date), MAX(update_date) 
                   FROM Commandes 
                   INNER JOIN Livres
                   ON Commandes.ARTICLE=Livres.ID
                   WHERE update_date > '%s'
                   GROUP BY DATECOM,ARTICLE,CLIENT
                   ORDER BY 1,2,3
                   """%(last_update.strftime("%Y-%m-%d %H:%M:%S"))
        cursor.execute(query)

        records = cursor.fetchall()
        with self.output().open('w') as f:
            for record in records:
                f.write(str(record) + '\n')

        with open('last_update_luigi.txt', 'w') as f:
            f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

        cursor.close()
        connection.close()

class LoadToOLAP(luigi.Task):
    def requires(self):
        return ExtractFromOLTP()

    def run(self):
        connection = mysql.connector.connect(**MYSQL_OLAP_CREDENTIALS)
        cursor = connection.cursor()

        with self.input().open('r') as f:
            for line in f:
                record = eval(line)
                insert_query = """INSERT INTO Faits_commandes_luigi(DATECOM,ARTICLE,CLIENT,NOMBRE_VENTE,CA,create_date,update_date) 
VALUES ('%s',%i,%i,%f,%f,'%s','%s')"""%(record[0].strftime("%Y-%m-%d"),
                                  record[1],record[2],record[3],record[4],
                                  record[5].strftime("%Y-%m-%d %H:%M:%S"),
                                  record[6].strftime("%Y-%m-%d %H:%M:%S"))
                cursor.execute(insert_query)

        connection.commit()
        cursor.close()
        connection.close()

if __name__ == '__main__':
    luigi.run()

In [ ]:
from datetime import datetime

with open('last_update_luigi.txt', 'w') as f:
        f.write(datetime(1970, 1, 1).strftime("%Y-%m-%d %H:%M:%S"))

Dans ce script, nous définissons deux tâches Luigi : `ExtractFromOLTP` et `LoadToOLAP`. La tâche `ExtractFromOLTP` extrait les nouveaux enregistrements de la base de données OLTP et les écrit dans un fichier local. La tâche `LoadToOLAP` lit les enregistrements du fichier local et les charge dans la base OLAP. La tâche `LoadToOLAP` dépend de la tâche `ExtractFromOLTP`, donc Luigi exécutera automatiquement `ExtractFromOLTP` en premier.

Il faut lancer le planificateur Luigi à partir d'un shell Python :
```
luigid
```
Dans le navigateur, il faut entrer l'URL http://localhost:8082/ pour avoir l'IHM :
![IHM du tableau de bord de Luigi](luigi_01.png)
Pour exécuter cette tâche ETL, exécutez la commande suivante dans un autre shell Python :
```
python etl_tasks_luigi.py LoadToOLAP
```
Cela démarrera le *job ETL* et Luigi gérera automatiquement les dépendances entre les tâches. Si vous souhaitez planifier l'exécution régulière de cette tâche, vous pouvez utiliser un planificateur de tâches tel que cron sur les systèmes Unix ou le Planificateur de tâches sous Windows.

Le *job ETL* tel que décrit est conçu pour remplacer les modifications apportées à la table de faits de la base OLAP par les dernières données de la table des commandes OLTP. Si un enregistrement dans la table des commandes OLTP est mis à jour puis extrait par le processus ETL, il sera inséré dans la table de faits OLAP, créant potentiellement des entrées en double.

Voici comment proposer une nouvelle tâche `LoadToOLAPWithSC2` pour un SCD de type 2, dérivée de `LoadToOLAP` :


In [ ]:
%%writefile etl_tasks_luigi_sc2.py

import luigi
import mysql.connector
from datetime import datetime
from config import MYSQL_OLTP_CREDENTIALS, MYSQL_OLAP_CREDENTIALS


class ExtractFromOLTP(luigi.Task):
    def output(self):
        return luigi.LocalTarget('data.txt')

    def run(self):
        connection = mysql.connector.connect(**MYSQL_OLTP_CREDENTIALS)
        cursor = connection.cursor()

        with open('last_update_luigi.txt', 'r') as f:
            last_update = datetime.strptime(f.read().strip(),'%Y-%m-%d %H:%M:%S')

        query = """SELECT DATECOM,ARTICLE,CLIENT,SUM(QUANTITE), SUM(PRIX*QUANTITE), MAX(create_date), MAX(update_date) 
                   FROM Commandes 
                   INNER JOIN Livres
                   ON Commandes.ARTICLE=Livres.ID
                   WHERE update_date > '%s'
                   GROUP BY DATECOM,ARTICLE,CLIENT
                   ORDER BY 1,2,3
                   """%(last_update.strftime("%Y-%m-%d %H:%M:%S"))
        cursor.execute(query)

        records = cursor.fetchall()
        with self.output().open('w') as f:
            for record in records:
                f.write(str(record) + '\n')

        with open('last_update_luigi.txt', 'w') as f:
            f.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

        cursor.close()
        connection.close()

class LoadToOLAP(luigi.Task):
    def requires(self):
        return ExtractFromOLTP()

    def run(self):
        connection = mysql.connector.connect(**MYSQL_OLAP_CREDENTIALS)
        cursor = connection.cursor()

        with self.input().open('r') as f:
            for line in f:
                record = eval(line)
                insert_query = """INSERT INTO Faits_commandes_luigi(DATECOM,ARTICLE,CLIENT,NOMBRE_VENTE,CA,create_date,update_date) 
VALUES ('%s',%i,%i,%f,%f,'%s','%s')"""%(record[0].strftime("%Y-%m-%d"),
                                  record[1],record[2],record[3],record[4],
                                  record[5].strftime("%Y-%m-%d %H:%M:%S"),
                                  record[6].strftime("%Y-%m-%d %H:%M:%S"))
                cursor.execute(insert_query)

        connection.commit()
        cursor.close()
        connection.close()

class LoadToOLAPWithSC2(luigi.Task):
    def requires(self):
        return ExtractFromOLTP()

    def run(self):
        connection = mysql.connector.connect(**MYSQL_OLAP_CREDENTIALS)
        cursor = connection.cursor()

        with self.input().open('r') as f:
            for line in f:
                record = eval(line)
                # Check if the record already exists
                cursor.execute("SELECT * FROM Faits_commandes_SCD2 WHERE DATECOM = %s AND ARTICLE = %s AND CLIENT = %s", (record[0],record[1],record[2],))
                if cursor.fetchone() is not None:
                    # If the record exists, end the current record
                    cursor.execute("UPDATE Faits_commandes_SCD2 SET end_date = %s WHERE DATECOM = %s AND ARTICLE = %s AND CLIENT = %s AND end_date IS NULL", (datetime.now(), record[0],record[1],record[2]))
                # Insert the new record
                insert_query = """INSERT INTO Faits_commandes_SCD2 (DATECOM,ARTICLE,CLIENT,NOMBRE_VENTE,CA,create_date,update_date,end_date) 
VALUES ('%s',%i,%i,%f,%f,'%s','%s',NULL)"""%(record[0].strftime("%Y-%m-%d"),
                  record[1],record[2],record[3],record[4],
                  record[5].strftime("%Y-%m-%d %H:%M:%S"),
                  record[6].strftime("%Y-%m-%d %H:%M:%S"))
                cursor.execute(insert_query)

        connection.commit()
        cursor.close()
        connection.close()


if __name__ == '__main__':
    luigi.run()        

In [ ]:
!python etl_tasks_luigi.py LoadToOLAP

Dans cet exemple, la table de faits nommée `Faits_commandes_SCD2` dans la base de données OLAP a une colonne `end_date` supplémentaire. Lorsqu'un nouvel enregistrement est chargé, la tâche vérifie si l'enregistrement existe déjà dans la table. Si tel est le cas, la tâche définit la date de fin de l'enregistrement actuel sur la date actuelle, marquant la fin de la validité de l'enregistrement. Ensuite, la tâche insère le nouvel enregistrement, avec la date de fin définie sur NULL pour indiquer que cet enregistrement est l'enregistrement valide actuel.

Pour exécuter cette tâche ETL, exécutez la commande suivante dans un shell Python :
```
python etl_tasks_luigi_sc2.py LoadToOLAPWithSC2
```

Les dernières ressources sur Luigi et les bases de données sont :
- [Luigi, Pandas et SQLAlchemy](https://lovethepenguin.com/python-create-an-etl-with-luigi-pandas-and-sqlalchemy-d3cdc9292bc7)
- [Luigi et le nuage](https://www.alibabacloud.com/blog/598152)
- [Luigi et PostgreSQL](https://medium.com/analytics-vidhya/build-simple-data-pipelines-from-scratch-using-postgresql-luigi-and-python-script-d3423f0a02d8)
- [Luigi et Jupyter](https://intoli.com/blog/luigi-jupyter-notebooks/)


## Processus ETL Python (1, 2 et 3)

1. Chaque mois, importation dans le schéma décisionnel depuis le schéma de production des moyennes des cours journaliers des devises fluctuantes du mois passé, le 5 du mois. (c'est la collecte du web service monétaire qui permet cela.
2. Chaque mois, importation du réalisé dans faits_vente le mois précédent, le 5 du mois. (c'est l'outil de simulation qui vous génère les lignes de T_FACTURE et de T_LIGNE_FACTURE)
3. Chaque année, le 5 janvier, importation dans le schéma décisionnel depuis des fichiers CSV des indicateurs concernant les objectifs dans la table de faits concernant l'année qui arrive. C'est l'outil de simulation qui vous génère le fichier CSV annuel des objectifs.

On considère que les fichiers objectifs contiennent les ouvertures et les fermetures des magasins :
- ouverture : une nouvelle ville apparaît. Les objectifs pour les mois avant l'ouverture sont tous à NULL pour tous les produits
- fermeture définitive dans l'année : Les objectifs pour les mois après la fermeture sont tous à NULL pour tous les produits. Pour une fermeture temporaire pour travaux, par exemple, les objectifs sont tous à 0 pour tous les produits durant la durée prévue de fermeture.
Une ouverture ne peut avoir lieu que dans une commune où il n'y a pas d'autre magasin du groupe, toute enseigne confondues. 

La table `t_departement` ne contient que les départements concernés par un des magasins. La table `t_rgd` donne les départements de la métropole tandis qhe la table `t_rgc` donne l'ensemble des communes de la métropole. A l'ouverture d'un magasin, il y aura création d'un tuple dans `t_magasin`, si la ville de ce nouveau magasin est dans un département absent de `t_departement`, il faudra aussi ajouter un tuple dans `t_departement` issu de `t_rgd`. Dans le schéma décisionnel, la table `france_departements` qui est un réplicat de `t_rgd` n'a pas d'objet et est vide. Elle était utilisée dans le portail décisionnel quand la gestion des ouvertures de magasins était à la charge de celui-ci. On peut considérer que la gestion des ouvertures des magasins est à la charge du processus ETL 3 en python.

Si on appelle vérification l'`UPSERT` (`UPDATE` ou `INSERT`) qui insère ou écrase les anciennes valeurs donc que des SCD de type 1 pour notre schéma en flocon et notre schéma en étoile,

Donc voici l'ordre de difficulté croissante :

1. ETL_1 : soit limitation des changements des données au mois précédant le mois courant, soit pas de limitation
    - lecture via SQL ;
    - vérification `dim_devise`, vérification de `faits_cours`.
2. ETL_2 : soit limitation des changements des données au mois précédant le mois courant, soit pas de limitation
    - lecture via SQL ;
    - vérification de `dim_produit_star` (verification de `dim_produit`, `dim_sous_categorie`, `dim_categorie`, `dim_famille` puis `CALL maj_dim_produit_star();`) ; 
    - vérification de `dim_magasin_star` (verification de `dim_magasin`, `dim_departement`, `dim_geographique_com`, `dim_geographique_admin`, `dim_pays` puis `CALL maj_dim_magasin_star();`) ;
    - vérification de `faits_ventes_star` pour les indicateurs réels (vérification de `faits_ventes` puis `CALL maj_faits_ventes_star(); CALL maj_securite_star(); CALL maj_dwr_faits_ventes_star();`).
3. ETL_3 : soit limitation des changements des données à l'année courante, soit pas de limitation
    - vérification de dim_temps et ajout éventuels des 12 lignes d'une nouvelle année (ne se présentera que le 5 janvier 2026 !)
    - lecture des données depuis un fichier CSV avec jointure sur `libelle` des produits et `ville` des magasins et interprétation des noms de mois en français ;
    - vérification de `dim_produit_star` (verification de `dim_produit`, `dim_sous_categorie`, `dim_categorie`, `dim_famille` puis `CALL maj_dim_produit_star();`) ; 
    - vérification de `dim_magasin_star` (verification de `dim_magasin`, `dim_departement`, `dim_geographique_com`, `dim_geographique_admin`, `dim_pays` puis `CALL maj_dim_magasin_star();`) ;
    - vérification de `faits_ventes_star` pour les indicateurs objectifs (vérification de `faits_ventes` puis `CALL maj_faits_ventes_star(); CALL maj_securite_star(); CALL maj_dwr_faits_ventes_star();`).

Le processus ETL 3 présuppose, pour un traitement du fichier CSV sans mauvaise surprise, une contrainte d'unicité sur les villes des magasins, ce que le jeu de l'année dernière respectait déjà et une contrainte d'unicité sur le libelle des produits hélas non respectée. Pour un jeu de fichiers CSV et un jeu de données 2022-2023 du serveur local ayant ces contraintes et ayant la simulation en Python corrigée, il faut télécharger et décompresser dans `D:\tools\WinPython-64bit-3.5.3.1Qt5\notebooks` l'archive simple [`simulation_python_corrigee_csv.zip`](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/simulation_python_corrigee_csv.zip?forcedownload=1) et l'archive multiple ([`mysql_data.zip.001`](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/mysql_data.zip.001?forcedownload=1) et [`mysql_data.zip.002`](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/mysql_data.zip.002?forcedownload=1)) à déposer dans D:\. On peut réunir les deux morceaux avec `join_local_data.bat` et déployer ce jeu de données sur le serveur local MySQL  avec `restauration_local_data.bat` via un shell MySQL. Il vaut mieux avant détruire `h:\mysql_data.zip` puisque le dernier fichier `.bat` va le chercher et risque d'écraser notre archive réassemblée. Pour l'installation et la description de ces utilitaires, il faut revenir en [début de séance](#seance_2).

Vous disposez alors d'un jeu de données idéal pour écrire de tester vos processus ETL. Avant de continuer, je vous invite à faire un export dans phpMyAdmin de votre serveur local MySQL de votre base OLTP et de votre base OLAP dans deux fichiers sql nommés respectivement `OLTP.sql` et `OLAP.sql`. Les options par défaut de phpMyAdmin créent les tables, les remplies puis à la fin du script, toutes les clés étrangères sont crées. Ces fichiers seront mis dans `D:\tools\mysql-8.0.18-winx64\bin` mais aussi par sécurité dans `H:\`. Ces fichiers sernot utilisés pour obtenir un état convenable sur votre serveur de production MySQL.

Tous les éléments constitutifs de ces processus ETL viennent d'être abordés : La journalisation, passage de la journalisation en mode DEBUG ou en mode INFO avant exécution d'une procédure ou d'une fonction, la lecture et écriture de fichiers CSV, alimentation de plusieurs tables à partir de plusieurs autres tables, exécution de plusieurs instructions SQL en séquence, exécution de script SQL plus ou moins complexe en Python, configuration des différentes connexions aux bases de données dans un fichier exterieur py. La factorisation via des procédures et fonctions paramétrées vous permettra aussi d'écrire vos processus ETL de manière plus concise.

Sur le serveur de production :
- les ordres SQL inter-schema sont impossibles ; 
- les procédures, fonctions et déclencheurs et les vues sont impossibles ;

L'utilisation de fichiers textuels intermédiaires a été étudiée.

Rappel : le schéma décisionnel comporte la définition des procédures stockées `maj_dim_magasin_star`, `maj_dim_produit_star`, `maj_faits_ventes_star`, `maj_securite_star`, `maj_dwr_faits_ventes_star`. Ainsi le rafraichissement du schéma en étoile à partir du schéma en flocon se fait par la séquence suivante :
```
CALL maj_dim_magasin_star();
CALL maj_dim_produit_star();
CALL maj_faits_ventes_star();
CALL maj_securite_star();
CALL maj_dwr_faits_ventes_star();
```
Une partie du schéma en étoile avec les procédures ne peut pas être utilisée sur le serveur MySQL de production : On la remplace par des fichiers statiques d'ordres SQL qui sont exécutés en Pyhon en séquence.

La tâche "alimentation de la table de faits" est précédée par les tâches concernant l'alimentation des tables de dimension. 

Remarque : pour les alternants, le jeu de données est tellement dense que quand j'ai rempli l'année dernière sur la base de données de production (e_22_3je2100923) du serveur mysql de production, l'ensemble des données réelles et les objectifs pour 2022 et les objectifs pour 2023 ainsi que les cours journaliers eux vont de 2019 à 2022, j'ai saturé ce serveur MySQL de production. Pour les formations initiales, l'année dernière, je n'ai pris qu'un échantillon de la chaîne des produits collectée par un "bon étudiant".

<a id='seance_SQL_dec'></a>
Vous pouvez étudier la [séquence d'ordres SQL inter-schéma](https://moodle.univ-ubs.fr/mod/forum/discuss.php?d=35850#p65106) pour faire une alimentation quasi automatique du schéma en flocon puis en étoile via la table t_objectif notamment, alimentation concurrente à vos processus ETL 1, 2 et 3. Ces ordres SQL, exécutés sous [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=e_24_3pexxxxxxx&server=2), vous permettent d'obtenir un schéma en flocon rempli. La séquence d'appels aux procédures stockées de ce paragraphe vous permettra obtenir un schéma en étoile mis à jour avec les dernières données. Ceci permet à un membre du binôme de s'attaquer au serveur Mondrian, lorsque le second continue à faire les processus ETL en Python.
Les ordres contenus dans cette séquence illustrent le UPSERT avancé de MySQL et peuvent être réexécutés autant de fois et correspond à un rafraichissement complet du schéma en flocon à partir des données du schéma de production. En pratique tous les processus ETL doivent s'exécuter dans la fenêtre temporelle d'une seule nuit, la production et les analystes travaillant le jour. Il s'agit de limiter la quantité de données à modifier au strict nécessaire.

Cette séquence d'ordres SQL, propres à MySQL, inter-schéma est rappelée ici :
```
-- alim dim_devise
INSERT INTO  e_25_3pexxxxxxx.`dim_devise`(`id_devise`, `lib_devise`, `isocode`, `symbole`, `format_bo`, `cours_fixe`)
SELECT `id_devise`, `lib_devise`, `isocode`, `symbole`,`format_bo`, `cours_fixe` 
FROM  e_25_3jexxxxxxx.`t_devise` t
WHERE 1 -- AND `id_devise`=2
ON DUPLICATE KEY UPDATE `lib_devise`=t.`lib_devise`, `isocode`=t.`isocode`, `symbole`=t.`symbole`, `format_bo`=t.`format_bo`, `cours_fixe`=t.`cours_fixe`
;


-- alim faits_cours
INSERT INTO  e_25_3pexxxxxxx.`faits_cours`(`id_devise`, `id_temps`, `cours`)
SELECT sb.`id_devise`, sb.id_temps, sb.avg_cours
FROM (
  SELECT `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0)) as id_temps , AVG(cours) as avg_cours 
  FROM e_25_3jexxxxxxx.`t_cours` 
  WHERE 1 -- AND `id_devise`=2 AND CONCAT(cast( `annee` AS char) ,lpad(`mois`, 2, 0))='202211'
  GROUP BY `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0))
) sb
ON DUPLICATE KEY UPDATE `cours`=sb.avg_cours
;
```

Il est impossible d'exécuter de tels ordres sur le serveur de production où les droits de vos utilisateurs sont limités à leur schéma. Un ordre interbase (ou inter schéma) ne peut qu'échouer. Il vous faudra soit créer des fichiers csv intermédiaires qui sont un pré-requis à l'upsert ou soit via Python ouvrir deux connexions en parallèle, une pour le schéma de production pour la lecture et l'autre pour le schéma en flocon pour l'écriture.
Pour les autres processus ETL, je vous donne à titre indicatif aussi ce qui marche sur le serveur de développement :
```
-- alim dim_enseigne :
INSERT INTO  e_25_3pexxxxxxx.`dim_enseigne`(`id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne`) 
SELECT `id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne` 
FROM  e_25_3jexxxxxxx.`t_enseigne` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_enseigne`=t.`lib_enseigne`, `fichier_image_logo_enseigne`=t.`fichier_image_logo_enseigne`
;

-- alim dim_pays
INSERT INTO  e_25_3pexxxxxxx.`dim_pays`(`id_pays`, `lib_pays`, `iso_3166_1_numeric`, `iso_3166_alpha_2`, `fichier_image_carte_pays`)
SELECT `id_pays`, `lib_pays`, `iso_3166_1_numeric`, `iso_3166_alpha_2`,`fichier_image_carte_pays` 
FROM  e_25_3jexxxxxxx.`t_pays` t 
WHERE 1
ON DUPLICATE KEY UPDATE `lib_pays`=t.`lib_pays`, `iso_3166_1_numeric`=t.`iso_3166_1_numeric`, `iso_3166_alpha_2`=t.`iso_3166_alpha_2`, `fichier_image_carte_pays`=t.`fichier_image_carte_pays`
;

-- alim dim_comm
INSERT INTO  e_25_3pexxxxxxx.`dim_geographique_com`(`id_region_com`, `lib_region_com`, `date_debut_valid_com`, `date_fin_valid_com`, `fichier_image_carte_regcom`, `id_pays`)
SELECT `id_region_com`, `lib_region_com`, `date_debut_valid_com`, `date_fin_valid_com`, `fichier_image_carte_regcom`, `id_pays` 
FROM  e_25_3jexxxxxxx.`t_region_commerciale` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_region_com`=t.`lib_region_com` , `date_debut_valid_com`=t.`date_debut_valid_com`, `date_fin_valid_com`=t.`date_fin_valid_com`,  `fichier_image_carte_regcom`=t.`fichier_image_carte_regcom`, `id_pays`=t.`id_pays`
;


-- alim dim_adm
INSERT INTO e_25_3pexxxxxxx.`dim_geographique_admin`(`id_region_admin`, `lib_region_admin`, `date_debut_valid_admin`, `date_fin_valid_admin`, `fichier_image_carte_regadm`, `sas_map_id`, `sas_map_name`, `id_pays`)
SELECT `id_region_admin`, `lib_region_admin`, `date_debut_valid_admin`, `date_fin_valid_admin`, `fichier_image_carte_regadm`, `sas_map_id`, `sas_map_name`, `id_pays` 
FROM  e_25_3jexxxxxxx.`t_region_administrative` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_region_admin`=t.`lib_region_admin`, `date_debut_valid_admin`=t.`date_debut_valid_admin`, `date_fin_valid_admin`=t.`date_fin_valid_admin`, `fichier_image_carte_regadm`=t.`fichier_image_carte_regadm`, `sas_map_id`=t.`sas_map_id`, `sas_map_name`=t.`sas_map_name`, `id_pays`=t.`id_pays` 
;

-- alim dim_dept
INSERT INTO e_25_3pexxxxxxx.`dim_departement`(`id_departement`, `code_departement`, `lib_departement`, `id_region_admin1`, `id_region_admin2`, `id_region_com`) 
SELECT `id_departement`, `code_departement`, `lib_departement`, `id_region_admin1`, `id_region_admin2`,`id_region_com` 
FROM  e_25_3jexxxxxxx.`t_departement` t
WHERE 1
ON DUPLICATE KEY UPDATE `code_departement`=t.`code_departement`, `lib_departement`=t.`lib_departement`, `id_region_admin1`=t.`id_region_admin1`, `id_region_admin2`=t.`id_region_admin2`, `id_region_com`=t.`id_region_com`
;

-- alim dim_magasin
INSERT INTO  e_25_3pexxxxxxx.`dim_magasin`(`id_magasin`, `id_enseigne`, `actif`, `date_ouverture`, `date_fermeture`, `emplacements`, `nb_caisses`, `ville`, `dep`, `fichier_image_carte_magasin`)
SELECT `id_magasin`, `id_enseigne`, `actif`, `date_ouverture`, `date_fermeture`, `emplacements`, `nb_caisses`, `ville`, `dep`, `fichier_image_carte_magasin` 
FROM  e_25_3jexxxxxxx.`t_magasin` t 
WHERE 1
ON DUPLICATE KEY UPDATE `id_enseigne`=t.`id_enseigne`, `actif`=t.`actif`, `date_ouverture`=t.`date_ouverture`, `date_fermeture`=t.`date_fermeture`, `emplacements`=t.`emplacements`, `nb_caisses`=t.`nb_caisses`, `ville`=t.`ville`, `dep`=t.`dep`, `fichier_image_carte_magasin`=t.`fichier_image_carte_magasin`
;

-- alim dim_famille
INSERT INTO  e_25_3pexxxxxxx.`dim_famille_produit`(`id_famille_produit`, `lib_famille_produit`)
SELECT `id_famille_produit`,`lib_famille_produit` 
FROM  e_25_3jexxxxxxx.`t_famille_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_famille_produit`=t.`lib_famille_produit`
;

-- alim dim_categorie
INSERT INTO  e_25_3pexxxxxxx.`dim_categorie_produit`(`id_categorie_produit`, `lib_categorie_produit`, `fk_famille_produit`) 
SELECT `id_categorie_produit`, `lib_categorie_produit`, `fk_famille_produit` 
FROM  e_25_3jexxxxxxx.`t_categorie_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `id_categorie_produit`=t.`id_categorie_produit`, `lib_categorie_produit`=t.`lib_categorie_produit`, `fk_famille_produit`=t.`fk_famille_produit`
;

-- alim dim_sous_categorie
INSERT INTO  e_25_3pexxxxxxx.`dim_sous_categorie_produit`(`id_sous_categorie_produit`, `lib_sous_categorie_produit`, `fk_categorie_produit`) 
SELECT `id_sous_categorie_produit`, `lib_sous_categorie_produit`, `id_categorie_produit` 
FROM  e_25_3jexxxxxxx.`t_sous_categorie_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `id_sous_categorie_produit`=t.`id_sous_categorie_produit`, `lib_sous_categorie_produit`=t.`lib_sous_categorie_produit`, `fk_categorie_produit`=t.`id_categorie_produit`
;

-- alim dim_produit
INSERT INTO e_25_3pexxxxxxx.`dim_produit`(`id_produit`, `libelle`, `description`, `en_vente`, `en_achat`, `fk_sous_categorie_produit`)
SELECT `id_produit`, `libelle`, `description`, `en_vente`, `en_achat`, `id_sous_categorie_produit` 
FROM  e_25_3jexxxxxxx.`t_produit` t 
WHERE 1
ON DUPLICATE KEY UPDATE `libelle`=t.`libelle`, `description`=t.`description`, `en_vente`=t.`en_vente`, `en_achat`=t.`en_achat`, `fk_sous_categorie_produit`=t.`id_sous_categorie_produit`
;

-- alim faits_ventes :
-- Objectifs, que INSERT
INSERT INTO e_25_3pexxxxxxx.`faits_ventes`(`id_magasin`, `id_produit`, `id_temps`, `ventes_objectif`, `ca_objectif`,  `marge_objectif`) 
SELECT `id_magasin`,`id_produit`,CONCAT(cast( `annee` AS char) ,lpad(`numero_mois`, 2, 0)) as id_temps,`objectif_nb_ventes`,`objectif_ca_ventes`,`objectif_marge_ventes` FROM e_24_3jexxxxxxx.`t_objectif`
order by 1,2,3

-- Objectifs, UPSERT (à préférer)
INSERT INTO e_25_3pexxxxxxx.`faits_ventes`(`id_magasin`, `id_produit`, `id_temps`, `ventes_objectif`, `ca_objectif`,  `marge_objectif`) 
SELECT `id_magasin`,`id_produit`,CONCAT(cast( `annee` AS char) ,lpad(`numero_mois`, 2, 0)) as id_temps,`objectif_nb_ventes`,`objectif_ca_ventes`,`objectif_marge_ventes` 
FROM e_25_3jexxxxxxx.`t_objectif` t
WHERE 1
ORDER BY 1,2,3
ON DUPLICATE KEY UPDATE `ventes_objectif`=t.`objectif_nb_ventes`, `ca_objectif`=t.`objectif_ca_ventes`,  `marge_objectif`=t.`objectif_marge_ventes`
;
-- 7248960 lignes insérées


CREATE VIEW e_25_3jexxxxxxx.v_vente AS 
SELECT  `t_facture`.`num_commande`,`id_ligne`,`quantite`,`pu_ht`,`prix_revient`,`id_produit`, `id_client`, `id_employe`, `id_magasin`, `date_commande_client` 
FROM e_25_3jexxxxxxx.`t_ligne_facture`
INNER JOIN e_25_3jexxxxxxx.`t_facture`
ON `t_ligne_facture`.`num_commande`=`t_facture`.`num_commande`
ORDER BY 1,2
;


-- Réalisé, que insert
INSERT INTO e_25_3pexxxxxxx.`faits_ventes`(`id_magasin`, `id_produit`, `id_temps`, `ventes_reel`, `ca_reel`, `marge_reel`) 
SELECT `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`), 2, 0)) , SUM(`quantite`), SUM(`pu_ht`* `quantite`-`reduction`), SUM((`pu_ht`-`prix_revient`)*`quantite`-`reduction`) 
FROM (`e_25_3jexxxxxxx`.`t_ligne_facture` 
INNER JOIN `e_25_3jexxxxxxx`.`t_facture` 
ON((`e_25_3jexxxxxxx`.`t_ligne_facture`.`num_commande` = `e_25_3jexxxxxxx`.`t_facture`.`num_commande`)))
WHERE 1
GROUP BY `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`), 2, 0))   
;

-- Réalisé, UPSERT (à préférer)
INSERT INTO e_25_3pexxxxxxx.`faits_ventes`(`id_magasin`, `id_produit`, `id_temps`, `ventes_reel`, `ca_reel`, `marge_reel`) 
SELECT sb.`id_magasin`, sb.`id_produit`, sb.id_temps, sb.snb, sb.sca, sb.smarge
FROM (
  SELECT `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`) , 2, 0)) as id_temps, SUM(`quantite`) as snb, SUM(`pu_ht`* `quantite`-`reduction`) as sca, SUM((`pu_ht`-`prix_revient`)*`quantite`-`reduction`) as smarge 
  FROM (`e_25_3jexxxxxxx`.`t_ligne_facture` 
  INNER JOIN `e_25_3jexxxxxxx`.`t_facture` 
  ON((`e_25_3jexxxxxxx`.`t_ligne_facture`.`num_commande` = `e_25_3jexxxxxxx`.`t_facture`.`num_commande`)))
  WHERE 1
  GROUP BY `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`), 2, 0))
) as sb
ON DUPLICATE KEY UPDATE `ventes_reel`=sb.snb, `ca_reel`=sb.sca, `marge_reel`=sb.smarge
;
```

Bien entendu, ceci n'est que l'alimentation du schéma en flocon. Il faut lancer le rafraichissement du schéma en étoile une fois le schéma en flocon stabilisé.
 
Il faut rajouter :
```
UPDATE `faits_ventes_star` SET `ventes_reel`=0,`ca_reel`=0,`marge_reel`=0 WHERE `id_temps` LIKE '2022%' AND `ventes_reel` IS NULL;
UPDATE `faits_ventes` SET `ventes_reel`=0,`ca_reel`=0,`marge_reel`=0 WHERE `id_temps` LIKE '2022%' AND `ventes_reel` IS NULL;

```

On peut caractériser ces ordres :
```
-- alim des dimensions

INSERT INTO  base_OLAP.`dim_name`(liste des labels des colonnes avec d'abord les parties de clé primaires, sans les colonnes à valorisation automatique par MySQL dans la table dim_name)
SELECT liste de valeurs lues ou calculees au même nombre que la liste de colonnes précédante 
FROM  base_OLTP.`t_name` t où t est la source du lien de correlation
ON DUPLICATE KEY UPDATE 
liste des colonnes correspondant à la première liste sans les colonnes parties de clé primaire de la forme `colonne_name`=t.`colonne_name` où t est le lien de correlation
;


-- alim des faits

INSERT INTO  base_OLAP.`faits_name`(liste des labels colonnes avec d'abord les parties de clé primaires, sans les colonnes à valorisation automatique par MySQL dans la table faits_name, sans les indicateurs non pertinents)
SELECT liste sous la forme sb.label correspondant à ceux de la liste suivante
FROM (
  SELECT liste de valeurs lues ou calculees avec label au même nombre que la première liste de colonnes 
  FROM base_OLTP.`t_name` avec jointures éventuelles de tables
  GROUP BY liste des colonnes ou des calculs qui n'utilisent pas MIN, MAX, COUNT, SUM, AVG
) sb où sb est la source du lien de correlation
ON DUPLICATE KEY UPDATE liste des indicateurs pertinents de la forme label dans première liste=sb.label dans la deuxième liste où sb est le lien de correlation
;
```
La "vue implicite", c'est à dire la sous requête dans le FROM est obligatoire si on fait dans le calcul un GROUP BY.

En généralisant plus avant le UPSERT de MySQL:
```
INSERT OLAP.table_destination (...) : Partie 1
SELECT ...FROM OLTP.table_source ... WHERE ... GROUP BY : Partie 2
ON DUPLICATE KEY
UPDATE (sans SET)... Partie 3 
```
En Python, cela peut donner, pour la prise en compte d'une limite de temps avec P1 l'écriture de CSV et P2, l'injection dans la destination :
```
from config import MYSQL_OLTP_CREDENTIALS, MYSQL_OLAP_CREDENTIALS

def ETL_P1_limit_id_temps:  
  connection = None
  try:
    je me connecte à OLTP en utilisant config.py
    logging.info('...')
    Ecrit_CSV_from_database(filename='ETL_P1_dim',SQL='''Partie 2 pour dim WHERE Cle primaire IN (SELECT part_cle_primaire FROM t_faits_temporels WHERE ref_temps ...)''', base=OLTP)
    logging.debug('...')
    Ecrit_CSV_from_database(filename='ETL_P1_faits',SQL='Partie 2 pour fait WHERE ref_temps ...', base=OLTP)
  Except(...):
    logging.error('...')
  finally:
    logging.info('...')
    connection.close()
    
def ETL_P2_limit_id_temps:
  connection = None
  try:
    je me connecte à OLAP en utilisant config.py
    connection.autocommit = False
    listedim=lit_CSV(filename='ETL_P1_dim')
    Pour chaque element de la liste listedim:
      counter_dim = Interroge_BD_avec_requete_scalaire('''SELECT COUNT(*) FROM table_destination_dim WHERE clé_primaire_dim=%?'''%(valeur courante de la clé dans l'élément))
      if counter_dim==0:
        Je soumets Partie 1 VALUES valeur de l'élément courant
      else:
        Je soumets UPDATE table_destination SET Partie3 avec valeur de l'élément courant
    listefait=lit_CSV(filename='ETL_P1_faits')
    Pour chaque element de la liste :
      counter_fait = Interroge_BD_avec_requete_scalaire('''SELECT COUNT(*) FROM table_destination_fait WHERE part1_clé_primaire=%? AND part2_clé_primaire=%?'''%(valeur courante de la partie 1 de clé dans l'élément, valeur courante de la partie 2 de clé dans l'élément))
      if counter_fait==0:
        Je soumets Partie 1 VALUES valeur de l'élément courant
      else:
        Je soumets UPDATE table_destination SET Partie3 avec valeur de l'élément courant
    #la transaction complexe c'est les maj des dims et de la table de faits
    connection.commit()

  Except(...):
    logging.error('...')
    connection.rollback()

  finally:
    logging.info('...')
    connection.close()
      
def ETL_limit_id_temps:
  ETL_P1_limit_id_temps()
  ETL_P2_limit_id_temps()
```
Dans une nouvelle cellule :
```
%%time
# logging.getLogger().setLevel(logging.DEBUG)
logging.getLogger().setLevel(logging.INFO)
ETL_limit_id_temps()  
```
Soit le processus est sans erreurs, et via phpMyAdmin, on vérifie que les tables sont bien alimentées au niveau du schéma en flocon puis au niveau du schéma en étoile. On vient de tester la branche INSERT de `ETL_P2_limit_id_temps`.

Soit le processus est avec des erreurs et il faut passer en mode debug de la journalisation.
Il faut corriger les erreurs jusqu'à ce qu'il n'en ait plus aucune.

Dans phpMyAdmin, sur les tables du schéma de production concernées par le processus ETL et surtout sur les tuples concernés par ce processus, il faut faire des updates avec des valeur remarquables (voire aberrantes). Puis réexécuter la cellule précédante. Et vérifier dans la base OLAP via phpMyAdmin, si toutes les mises à jours ont eu lieu dans le schéma en flocon et dans le schéma en étoile. On vient de tester la branche UPDATE de `ETL_P2_limit_id_temps`.

Bref la partie 1 du processus est la plus spécifique : les calculs y sont à faire, les limitations y sont à faire... La partie 2 peut être factorisée dans une procedure ? A voir.

MySQL accepte les ordres 'SELECT ... FROM ... WHERE 1' donc l'application ou non (par défaut oui) de la limitation temporelle peut être :
```
def ETL_P1(with_limit_flag=True)
  where_cond='1'
  if with_limit_flag:
    where_cond='''Cle_primaire IN (SELECT part_cle_primaire FROM faits WHERE id_temps ...)'''
  SQL='... WHERE %s'%(where_cond)

def ETL(with_limit_flag=True):
  ETL_P1(with_limit_flag)
  ETL_P2()

```

L'expression de la limite de temps change pour les processus ETL 1, 2 et 3.
- Pour ETL 1 et ETL 2 :
Ces deux processus prend les données dans le schéma de production pour alimenter le schéma décisionnel.

Je récupère la date courante. Je retire 28 jours à cette date. J'extrais le mois et l'année et je constitue `ref_temps`.
La condition devient `'''Cle primaire IN (SELECT part_cle_primaire FROM t_faits_temporels WHERE id_temps=%s)'''%(ref_temps)`.
- Pour ETL 3 :
Je récupère la date courante. J'extrais l'année et je constitue `ref_annee`. La condition devient `'''Cle primaire IN (SELECT part_cle_primaire FROM faits WHERE id_temps LIKE '%s')'''%(str(ref_annee))`.

Pour la mise à jour du schéma en étoile, il est plus robuste de programmer directement la façon compatible avec le serveur de production.

Pour le processus ETL 3, s'il n'y a pas de limitation temporelle, il faut charger dans l'ordre tous les fichiers annee.csv du répertoire courant et les traiter un par un.

Le fichier `annee.csv` a normalement été créé par le directeur marketing du groupe DARTIES à partir des données du schéma de production. Il faut donc récupérer les données de `t_magasin` et de `t_produit` dans des fichiers csv pour mettre à jour `dim_magasin` et `dim_produit` pour pouvoir faire la "jointure pas naturelle" sur les libellés des produits et les villes des magasins avec les tables `dim_`.

Dans une cellule python 3:
```
%%writefile config.py
# pour le serveur local MySQL
MYSQL_OLTP_CREDENTIALS = {
...
}

MYSQL_OLAP_CREDENTIALS = {
...
}

```
Dans une autre cellule python 3 :
```
%%time
# logging.getLogger().setLevel(logging.DEBUG)
logging.getLogger().setLevel(logging.INFO)
ETL(False)  
```
On vient de tester notre processus ETL sur le serveur MySQL local.
Un conseil : avant de faire le déploiement sur le serveur de production de vos données, peut être faire les autre processus ETL ?

Le déploiement de vos données sur le serveur de production consiste :
- faire un export dans phpMyAdmin de votre serveur local MySQL de votre base OLTP et de votre base OLAP dans deux fichiers sql nommés respectivement `OLTP.sql` et `OLAP.sql`. Les options par défaut de phpMyAdmin créent les tables, les remplies puis à la fin du script, toutes les clés étrangères sont crées. Ces fichiers seront mis dans `D:\tools\mysql-8.0.18-winx64\bin`. Cet export a déjà été fait normalement. Reste qu'il faut copier depuis votre `H:\` les fichiers `OLTP.sql` et `OLAP.sql` dans `D:\tools\mysql-8.0.18-winx64\bin`.
- A partir de maintenant :
- **Abric--Segarra Maxence doit remplacer e_25_3pexxxxxxx par e_25_3pe2401441 et e_25_3jexxxxxxx par e_24_3je2401441**.
- **Cheruel Jocelyn doit remplacer e_25_3pexxxxxxx par e_25_3pe2401348 et e_25_3jexxxxxxx par e_25_3je2401348**.
- dans un shell MySQL, pour la base de données OLTP (une ligne à la fois) :
```
mysql -h projetud.univ-ubs.fr -u e_25_3jexxxxxxx -pde_25_3jexxxxxxxd e_25_3jexxxxxxx
\. OLTP.sql
quit;
```
puis pour la base OLAP :
```
mysql -h projetud.univ-ubs.fr -u e_25_3pexxxxxxx -pde_25_3pexxxxxxxd e_25_3pexxxxxxx
\. OLAP.sql
quit;
```

On vient de mettre en place les bases de données de production (OLTP=>j) et décisionnelle (OLAP=>p pour PHP, techno pour portail décisionnel historique ) sur le serveur MySQL de production.

Dans une autre cellule python 3 :
```
%%writefile config.py
# pour le serveur de production MySQL (projetud.univ-ubs.fr:3306)
MYSQL_OLTP_CREDENTIALS = {
...
}

MYSQL_OLAP_CREDENTIALS = {
...
}

```
Dans une autre cellule python 3 :
```
%%time
# logging.getLogger().setLevel(logging.DEBUG)
logging.getLogger().setLevel(logging.INFO)
ETL(False)  
```
On vient de tester notre processus ETL sur le serveur MySQL de production.




In [ ]:
%%writefile config.py
MYSQL_OLTP_CREDENTIALS = {
    'user': 'e_25_3jexxxxxxx',
    'password': 'de_25_3jexxxxxxxd',
    'host': 'localhost',
    'port': 3312,
    'database': 'e_25_3jexxxxxxx',
    'auth_plugin' : 'mysql_native_password'
}

MYSQL_OLAP_CREDENTIALS = {
    'user': 'e_25_3pexxxxxxx',
    'password': 'de_25_3pexxxxxxxd',
    'host': 'localhost',
    'port': 3312,
    'database': 'e_25_3pexxxxxxx',
    'auth_plugin' : 'mysql_native_password'
}

In [ ]:
#%%writefile config.py
MYSQL_OLTP_CREDENTIALS = {
    'user': 'e_25_3je2405888',
    'password': 'de_25_3je2405888d',
    'host': 'projetud.univ-ubs.fr',
    'port': 3306,
    'database': 'e_25_3je2405888',
    'auth_plugin' : 'mysql_native_password'
}

MYSQL_OLAP_CREDENTIALS = {
    'user': 'e_25_3je2405888',
    'password': 'de_25_3je2405888d',
    'host': 'projetud.univ-ubs.fr',
    'port': 3306,
    'database': 'e_25_3je2405888',
    'auth_plugin' : 'mysql_native_password'
}

## ETL_1
**Chaque mois, importation dans le schéma décisionnel depuis le schéma de production des moyennes des cours journaliers des devises fluctuantes du mois passé, le 5 du mois. (c'est la collecte du web service monétaire qui permet cela).**

ETL_1 : soit limitation des changements des données au mois précédant le mois courant, soit pas de limitation
    - lecture via SQL ;
    - vérification `dim_devise`, vérification de `faits_cours`.
    
On peut vider les tables `dim_devise` et `faits_cours` par :
```
DELETE FROM faits_cours;
DELETE FROM dim_devise;
ALTER TABLE dim_devise AUTO_INCREMENT=1;
```
### t_devise dans un fichier CSV
```
-- alim dim_devise
INSERT INTO  e_25_3pexxxxxxx.`dim_devise`(`id_devise`, `lib_devise`, `isocode`, `symbole`, `format_bo`, `cours_fixe`)
```
**
SELECT `id_devise`, `lib_devise`, `isocode`, `symbole`,`format_bo`, `cours_fixe` 
FROM  e_25_3jexxxxxxx.`t_devise` t
WHERE 1 -- AND `id_devise`=2
**
```
ON DUPLICATE KEY UPDATE `lib_devise`=t.`lib_devise`, `isocode`=t.`isocode`, `symbole`=t.`symbole`, `format_bo`=t.`format_bo`, `cours_fixe`=t.`cours_fixe`
;
```

**Condition temporelle pour t_devise en vue du remplissage de dim_devise**

In [ ]:
import logging
logging.getLogger().setLevel(logging.DEBUG)
# prise en compte de la limitation temporelle de la table de dimension
from datetime import date
from dateutil.relativedelta import relativedelta
 
today = date.today()
premier = date(today.year, today.month, 1)
dernier = premier + relativedelta(months=1, days=-1)
logging.debug("Mois courant : %s, %s", str(premier), str(dernier))

from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

i = -1  # This could have been any integer, positive or negative
someday = date.today()
# Is there any difference between these two lines?
otherday = someday + timedelta(days=i)
logging.debug("Jour antérieur timedelta : %s", str(otherday))
otherday = someday + relativedelta(days=i)
logging.debug("Jour antérieur relativedelta : %s", str(otherday))
# les données des cours s'arrêtent en 2022 et non en 2025 !
premier = date(today.year-3, today.month-1, 1)
dernier = premier + relativedelta(months=1, days=-1)
logging.debug("Mois antérieur : %s, %s", str(premier), str(dernier))
id_temps=str(premier.year)+str('{:02d}'.format(premier.month))
# limitation des changements des données au mois précédant le mois courant
where_data="where id_devise in (select distinct id_devise from t_cours where pubdate between '%s' and '%s' )"%(premier,dernier)
# pas de limitation des changements des données du schema de production
where_data='where 1'
logging.debug("condition : %s",where_data) 

In [ ]:
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_devise")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_devise, lib_devise, isocode, symbole,format_bo, cours_fixe FROM t_devise t %s
    """%(where_data)
    logging.debug("query: "+query)
    cursor.execute(query)
    with open("t_devises.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

        

### t_devise dans la base e_25_3pexxxxxxx (dim_devise)
```
-- alim dim_devise
```
**
INSERT INTO  e_25_3pexxxxxxx.`dim_devise`(`id_devise`, `lib_devise`, `isocode`, `symbole`, `format_bo`, `cours_fixe`)
**
```
SELECT `id_devise`, `lib_devise`, `isocode`, `symbole`,`format_bo`, `cours_fixe` 
FROM  e_25_3jexxxxxxx.`t_devise` t
WHERE 1 -- AND `id_devise`=2
```
**
ON DUPLICATE KEY UPDATE `lib_devise`=t.`lib_devise`, `isocode`=t.`isocode`, `symbole`=t.`symbole`, `format_bo`=t.`format_bo`, `cours_fixe`=t.`cours_fixe`
**
```
;
```

In [ ]:
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_devises.csv")
    with open('t_devises.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_devise from dim_devise where id_devise = '%s'"
            cursor.execute(request, (row['id_devise']))
            logging.debug(request)
            logging.debug(row['id_devise'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                logging.debug(type(row['cours_fixe']))
                if row['cours_fixe']:
                    query = """INSERT INTO dim_devise (id_devise, lib_devise, isocode, symbole, format_bo, cours_fixe) VALUES (%i,'%s','%s','%s', '%s', %f)"""%(row['id_devise'], row['lib_devise'], row['isocode'], row['symbole'], row['format_bo'], row['cours_fixe'])
                else: 
                    query = """INSERT INTO dim_devise (id_devise, lib_devise, isocode, symbole, format_bo, cours_fixe) VALUES (%i,'%s','%s','%s', '%s', %s)"""%(row['id_devise'], row['lib_devise'], row['isocode'], row['symbole'], row['format_bo'], "Null")
            else: 
                logging.debug(type(row['cours_fixe']))
                
                if row['cours_fixe']:
                    query= """UPDATE dim_devise SET lib_devise='%s', isocode= '%s', symbole = '%s', format_bo = '%s', cours_fixe = %f"""%(row['lib_devise'], row['isocode'], row['symbole'], row['format_bo'], row['cours_fixe'])
                else:
                    query= """UPDATE dim_devise SET lib_devise='%s', isocode= '%s', symbole = '%s', format_bo = '%s', cours_fixe = %s"""%(row['lib_devise'], row['isocode'], row['symbole'], row['format_bo'], "Null")
            
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


**Alternative avec UPSERT**

**Condition temporelle pour t_cours en vue du remplissage de faits_cours**

In [ ]:
# prise en compte de la limitation temporelle de la table de fait
from datetime import date
from dateutil.relativedelta import relativedelta
 
today = date.today()
premier = date(today.year, today.month, 1)
dernier = premier + relativedelta(months=1, days=-1)
logging.debug("Mois courant : %s, %s", str(premier), str(dernier))

from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

i = -1  # This could have been any integer, positive or negative
someday = date.today()
# Is there any difference between these two lines?
otherday = someday + timedelta(days=i)
logging.debug("Jour antérieur timedelta : %s", str(otherday))
otherday = someday + relativedelta(days=i)
logging.debug("Jour antérieur relativedelta : %s", str(otherday))
# les données des cours s'arrêtent en 2022 et non en 2025 !
premier = date(today.year-3, today.month-1, 1)
dernier = premier + relativedelta(months=1, days=-1)
logging.debug("Mois antérieur : %s, %s", str(premier), str(dernier))
id_temps=str(premier.year)+str('{:02d}'.format(premier.month))
# limitation des changements des données au mois précédant le mois courant
where_data="where pubdate between '%s' and '%s'"%(premier,dernier)
# pas de limitation des changements des données du schema de production
#where_data='where 1'
logging.debug("condition : %s",where_data) 

### t_cours dans un ficher CSV 
```
-- alim faits_cours
INSERT INTO  e_25_3pexxxxxxx.`faits_cours`(`id_devise`, `id_temps`, `cours`)
SELECT sb.`id_devise`, sb.id_temps, sb.avg_cours
FROM (
```
**
  SELECT `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0)) as id_temps , AVG(cours) as avg_cours 
  FROM e_25_3jexxxxxxx.`t_cours` 
  WHERE 1 -- AND `id_devise`=2 AND CONCAT(cast( `annee` AS char) ,lpad(`mois`, 2, 0))='202211'
  GROUP BY `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0))
**
```
) sb
ON DUPLICATE KEY UPDATE `cours`=sb.avg_cours
;
```

In [ ]:
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_cours")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_devise, CONCAT(lpad(annee, 4, 0), lpad(mois, 2, 0)) as id_temps , AVG(cours) as avg_cours 
    FROM e_25_3jexxxxxxx.t_cours WHERE 1 -- AND id_devise=2 AND CONCAT(cast( annee AS char) ,lpad(mois, 2, 0))='202211' 
    GROUP BY id_devise, CONCAT(lpad(annee, 4, 0), lpad(mois, 2, 0)) 
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_cours.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

### t_cours dans la base e_25_3pexxxxxxx (faits_cours)
```
-- alim faits_cours
```
**
INSERT INTO  e_25_3pexxxxxxx.`faits_cours`(`id_devise`, `id_temps`, `cours`)
**
```
SELECT sb.`id_devise`, sb.id_temps, sb.avg_cours
FROM (
  SELECT `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0)) as id_temps , AVG(cours) as avg_cours 
  FROM e_25_3jexxxxxxx.`t_cours` 
  WHERE 1 -- AND `id_devise`=2 AND CONCAT(cast( `annee` AS char) ,lpad(`mois`, 2, 0))='202211'
  GROUP BY `id_devise`, CONCAT(cast( `annee` AS char), lpad(`mois`, 2, 0))
) sb
```
**
ON DUPLICATE KEY UPDATE `cours`=sb.avg_cours
**
```
;
```

In [ ]:
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_cours.csv")
    with open('t_cours.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_devise from faits_cours where id_devise = '%s'"
            cursor.execute(request, (row['id_devise']))
            logging.debug(request)
            logging.debug(row['id_devise'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO faits_cours (id_devise, id_temps, cours) VALUES (%i,'%s',%f)"""%(row['id_devise'], row['id_temps'], row['avg_cours'])
            else: 
                query= """UPDATE faits_cours SET id_temps = '%s', cours = %f"""%(row['id_temps'], row['avg_cours'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )
    logging.info("La connexion à e_25_3pexxxxxxx est ouverte.")
    cursor = connection.cursor()

    logging.info("Traitement du fichier t_cours.csv.")
    with open('t_cours.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        
        for row in csv_reader:
            # Execute your first query (fetch avg_cours for the given id_devise and id_temps)
            request = """
                SELECT sb.id_devise, sb.id_temps, sb.avg_cours 
                FROM (
                    SELECT id_devise, CONCAT(CAST(annee AS CHAR), LPAD(mois, 2, '0')) AS id_temps, AVG(cours) AS avg_cours 
                    FROM e_25_3jexxxxxxx.t_cours 
                    WHERE id_devise = %s 
                    GROUP BY id_devise, CONCAT(CAST(annee AS CHAR), LPAD(mois, 2, '0'))
                ) sb
            """
            cursor.execute(request, (row['id_devise'],))
            logging.debug("Request: %s, id_devise: %s" % (request, row['id_devise']))
            result = cursor.fetchone()

            # If the data doesn't exist, you would insert new data into the `dim_devise` table
            if result is None:
                logging.debug("Data not found, performing INSERT.")
                insert_query = """
                    INSERT INTO dim_devise (id_devise, lib_devise, isocode, symbole, format_bo, cours_fixe) 
                    VALUES (%s, %s, %s, %s, %s, %s)
                """
                data_to_insert = (
                    row['id_devise'], row['lib_devise'], row['isocode'], 
                    row['symbole'], row['format_bo'], row['cours_fixe'] if row['cours_fixe'] else None
                )
                cursor.execute(insert_query, data_to_insert)
            else:
                logging.debug("Data exists, performing UPDATE.")
                update_query = """
                    UPDATE dim_devise 
                    SET lib_devise = %s, isocode = %s, symbole = %s, format_bo = %s, cours_fixe = %s 
                    WHERE id_devise = %s
                """
                data_to_update = (
                    row['lib_devise'], row['isocode'], row['symbole'], 
                    row['format_bo'], row['cours_fixe'] if row['cours_fixe'] else None, row['id_devise']
                )
                cursor.execute(update_query, data_to_update)

            connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error("Failed to insert/update record in the database: %s" % error)
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée.")


**Alternative avec UPSERT**

**Alternative avec 2 connexions ouvertes en parallèle**

In [ ]:
"""
Programme principal: activation des fonctions définies précédemment
"""

import mysql.connector
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

SERVEUR_LOCAL = True

if SERVEUR_LOCAL:
    # Connection à la base de donnée décisionnelle
    dbD = mysql.connector.connect(
        host = 'localhost',
        port = 3312,
        user = 'python',
        password='VOTRE_MDP',
        database = 'e_25_3pexxxxxxx'
    )
    #cD = dbD.cursor()

    # Connection à la base de donnée de production
    dbP = mysql.connector.connect(
        host = 'localhost',
        port = 3312,
        user = 'python',
        password='VOTRE_MDP',
        database = 'e_25_3jexxxxxxx'
    )
    #cP = dbP.cursor()
    
else:
    # Connection à la base de donnée décisionnelle
    dbD = mysql.connector.connect(
        host = 'projetud.univ-ubs.fr',
        user = 'e_25_3pe2302343',
        password = 'de_25_3pe2302343d',
        database = 'e_25_3pe2302343'
    )
    #cD = dbD.cursor()

    # Connection à la base de donnée de production
    dbP = mysql.connector.connect(
        host = 'projetud.univ-ubs.fr',
        user = 'e_25_3je2302343',
        password = 'de_25_3je2302343d',
        database = 'e_25_3je2302343'
    )
    #cP = dbP.cursor()

try:
    logging.info("Début de la tache ETL (schéma en flocon) - remplissage de la table dim_devise")
    ETL_dim_devise(dbD, dbP)
    
    logging.info("Table dim_devise remplie - remplissage de la table faits_cours")
    # La table dim_temps à été pré-remplie
    ETL_faits_cours(dbD, dbP)
    

    
    logging.info("Fin de la tache ETL")
    
    
finally:
    #cD.close()
    dbD.close()
    
    #cP.close()
    dbP.close()
    print("Fin du programme - connexion fermée")

## ETL_2
**Chaque mois, importation du réalisé dans faits_vente le mois précédent, le 5 du mois. (c'est l'outil de simulation qui vous génère les lignes de T_FACTURE et de T_LIGNE_FACTURE)**

ETL_2 : soit limitation des changements des données au mois précédant le mois courant, soit pas de limitation
    - lecture via SQL ;
    - vérification de `dim_produit_star` (verification de `dim_produit`, `dim_sous_categorie`, `dim_categorie`, `dim_famille` puis `CALL maj_dim_produit_star();`) ; 
    - vérification de `dim_magasin_star` (verification de `dim_magasin`, `dim_departement`, `dim_geographique_com`, `dim_geographique_admin`, `dim_pays` puis `CALL maj_dim_magasin_star();`) ;
    - vérification de `faits_ventes_star` pour les indicateurs réels (vérification de `faits_ventes` puis `CALL maj_faits_ventes_star(); CALL maj_securite_star(); CALL maj_dwr_faits_ventes_star();`).

    
### t_enseigne dans un fichier CSV
```
-- alim dim_enseigne :
INSERT INTO  e_25_3pexxxxxxx.`dim_enseigne`(`id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne`) 
```
**
SELECT `id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne` 
FROM  e_25_3jexxxxxxx.`t_enseigne` t
WHERE 1
**
```
ON DUPLICATE KEY UPDATE `lib_enseigne`=t.`lib_enseigne`, `fichier_image_logo_enseigne`=t.`fichier_image_logo_enseigne`
;

```

In [ ]:
#t_enseigne dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_enseigne")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_enseigne, lib_enseigne, fichier_image_logo_enseigne 
    FROM e_25_3jexxxxxxx.t_enseigne t 
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_enseigne.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

### t_enseigne dans la base e_25_3pexxxxxxx (dim_enseigne)
```
-- alim dim_enseigne :
```
**
INSERT INTO  e_25_3pexxxxxxx.`dim_enseigne`(`id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne`) 
**
```

SELECT `id_enseigne`, `lib_enseigne`, `fichier_image_logo_enseigne` 
FROM  e_25_3jexxxxxxx.`t_enseigne` t
WHERE 1
```
**
ON DUPLICATE KEY UPDATE `lib_enseigne`=t.`lib_enseigne`, `fichier_image_logo_enseigne`=t.`fichier_image_logo_enseigne`
**
```
;

```

In [ ]:
#t_enseigne dans la base e_25_3pexxxxxxx (dim_enseigne)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_enseigne.csv")
    with open('t_enseigne.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_enseigne from dim_enseigne where id_enseigne = '%s'"
            cursor.execute(request, (row['id_enseigne']))
            logging.debug(request)
            logging.debug(row['id_enseigne'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_enseigne (id_enseigne, lib_enseigne, fichier_image_logo_enseigne) VALUES (%i,'%s', '%s')"""%(row['id_enseigne'], row['lib_enseigne'], row['fichier_image_logo_enseigne'])
            else: 
                query= """UPDATE dim_enseigne SET lib_enseigne = '%s', fichier_image_logo_enseigne = '%s'"""%(row['lib_enseigne'], row['fichier_image_logo_enseigne'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_pays dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_pays")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_pays, lib_pays, iso_3166_1_numeric, iso_3166_alpha_2, fichier_image_carte_pays 
    FROM  e_25_3jexxxxxxx.t_pays t 
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_pays.csv", "w", newline='') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_pays dans la base e_25_3pexxxxxxx (dim_pays)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_pays.csv")
    with open('t_pays.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_pays from dim_pays where id_pays = '%s'"
            cursor.execute(request, (row['id_pays']))
            logging.debug(request)
            logging.debug(row['id_pays'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_pays (id_pays, lib_pays, iso_3166_1_numeric, iso_3166_alpha_2, fichier_image_carte_pays) VALUES (%i,'%s', '%s', '%s', '%s')"""%(row['id_pays'], row['lib_pays'], row['iso_3166_1_numeric'], row['iso_3166_alpha_2'], row['fichier_image_carte_pays'])
            else: 
                query= """UPDATE dim_pays SET lib_pays = '%s', iso_3166_1_numeric = '%s', iso_3166_alpha_2 = '%s', fichier_image_carte_pays = '%s'"""%(row['lib_pays'], row['iso_3166_1_numeric'], row['iso_3166_alpha_2'], row['fichier_image_carte_pays'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_region_commerciale dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_region_commerciale")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_region_com, lib_region_com, date_debut_valid_com, date_fin_valid_com, fichier_image_carte_regcom, id_pays 
    FROM  e_25_3jexxxxxxx.t_region_commerciale t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_region_commerciale.csv", "w", newline='', encoding='utf-8') as csv_file:   
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        csv_writer.writerow([i[0] for i in cursor.description]) # write headers
        csv_writer.writerows(cursor)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_region_commerciale dans la base e_25_3pexxxxxxx (dim_geographique_com)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_region_commerciale.csv")
    with open('t_region_commerciale.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_region_com from dim_geographique_com where id_region_com = '%s'"
            cursor.execute(request, (row['id_region_com']))
            logging.debug(request)
            logging.debug(row['id_region_com'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_geographique_com (id_region_com, lib_region_com, date_debut_valid_com, date_fin_valid_com, fichier_image_carte_regcom, id_pays) VALUES (%i,'%s', '%s', '%s', '%s', '%s')"""%(row['id_region_com'], row['lib_region_com'], row['date_debut_valid_com'], row['date_fin_valid_com'], row['fichier_image_carte_regcom'], row['id_pays'])
            else: 
                query= """UPDATE dim_geographique_com SET lib_region_com = '%s', date_debut_valid_com = '%s', date_fin_valid_com = '%s', fichier_image_carte_regcom = '%s', id_pays = '%s'"""%(row['lib_region_com'], row['date_debut_valid_com'], row['date_fin_valid_com'], row['fichier_image_carte_regcom'], row['id_pays'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_region_administrative dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging

def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)

connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_region_administrative")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_region_admin, lib_region_admin, date_debut_valid_admin, date_fin_valid_admin, fichier_image_carte_regadm, sas_map_id, sas_map_name, id_pays 
    FROM  e_25_3jexxxxxxx.t_region_administrative t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_region_administrative.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)
        
        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_region_administrative dans la base e_25_3pexxxxxxx (dim_geographique_admin)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_region_administrative.csv")
    with open('t_region_administrative.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_region_admin from dim_geographique_admin where id_region_admin = '%s'"
            cursor.execute(request, (row['id_region_admin']))
            logging.debug(request)
            logging.debug(row['id_region_admin'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_geographique_admin (id_region_admin, lib_region_admin, date_debut_valid_admin, date_fin_valid_admin, fichier_image_carte_regadm, sas_map_id, sas_map_name, id_pays) VALUES (%i,'%s', '%s', '%s', '%s', '%s', '%s', %i)"""%(row['id_region_admin'], row['lib_region_admin'], row['date_debut_valid_admin'], row['date_fin_valid_admin'], row['fichier_image_carte_regadm'], row['sas_map_id'], row['sas_map_name'], row['id_pays'])
            else: 
                query= """UPDATE dim_geographique_admin SET lib_region_admin = '%s', date_debut_valid_admin = '%s', date_fin_valid_admin = '%s', fichier_image_carte_regadm = '%s', sas_map_id = '%s', sas_map_name = '%s', id_pays = %i"""%(row['lib_region_admin'], row['date_debut_valid_admin'], row['date_fin_valid_admin'], row['fichier_image_carte_regadm'], row['sas_map_id'], row['sas_map_name'], row['id_pays'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_departement dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging

def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)

connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_departement")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_departement, code_departement, lib_departement, id_region_admin1, id_region_admin2,id_region_com 
    FROM  e_25_3jexxxxxxx.t_departement t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query) 
    with open("t_departement.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_departement dans la base e_25_3pexxxxxxx (dim_departement)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_departement.csv")
    with open('t_departement.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_departement from dim_departement where id_departement = '%s'"
            cursor.execute(request, (row['id_departement']))
            logging.debug(request)
            logging.debug(row['id_departement'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_departement (id_departement, code_departement, lib_departement, id_region_admin1, id_region_admin2, id_region_com) VALUES (%i, '%s', '%s', %i, %i, %i)"""%(row['id_departement'], row['code_departement'], row['lib_departement'], row['id_region_admin1'], row['id_region_admin2'], row['id_region_com'])
            else: 
                query= """UPDATE dim_departement SET code_departement = '%s', lib_departement = '%s', id_region_admin1 = %i, id_region_admin2 = %i, id_region_com = %i"""%(row['code_departement'], row['lib_departement'], row['id_region_admin1'], row['id_region_admin2'], row['id_region_com'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_magasin dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging

def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)

connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_magasin")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_magasin, id_enseigne, actif, date_ouverture, date_fermeture, emplacements, nb_caisses, ville, dep, fichier_image_carte_magasin 
    FROM  e_25_3jexxxxxxx.t_magasin t 
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)   
    with open("t_magasin.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_magasin dans la base e_25_3pexxxxxxx (dim_magasin)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_magasin.csv")
    with open('t_magasin.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_magasin from dim_magasin where id_magasin = '%s'"
            cursor.execute(request, (row['id_magasin']))
            logging.debug(request)
            logging.debug(row['id_magasin'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_magasin (id_magasin, id_enseigne, actif, date_ouverture, date_fermeture, emplacements, nb_caisses, ville, dep, fichier_image_carte_magasin) VALUES (%i, %i, '%s', '%s', '%s', '%s', %i, '%s', '%s', '%s')"""%(row['id_magasin'], row['id_enseigne'], row['actif'], row['date_ouverture'], row['date_fermeture'], row['emplacements'],  row['nb_caisses'], row['ville'], row['dep'], row['fichier_image_carte_magasin'])
            else: 
                query= """UPDATE dim_magasin SET id_enseigne = %i, actif = '%s', date_ouverture = '%s', date_fermeture = '%s', emplacements = '%s',  nb_caisses = %i, ville = '%s', dep = '%s', fichier_image_carte_magasin = '%s'"""%(row['id_enseigne'], row['actif'], row['date_ouverture'], row['date_fermeture'], row['emplacements'],  row['nb_caisses'], row['ville'], row['dep'], row['fichier_image_carte_magasin'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_famille_produit dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging

def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)

connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_famille_produit")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_famille_produit,lib_famille_produit 
    FROM  e_25_3jexxxxxxx.t_famille_produit t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)
    with open("t_famille_produit.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_famille_produit dans la base e_25_3pexxxxxxx (dim_famille_produit)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_famille_produit.csv")
    with open('t_famille_produit.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_famille_produit from dim_famille_produit where id_famille_produit = '%s'"
            cursor.execute(request, (row['id_famille_produit']))
            logging.debug(request)
            logging.debug(row['id_famille_produit'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_famille_produit (id_famille_produit, lib_famille_produit) VALUES (%i, '%s')"""%(row['id_famille_produit'], row['lib_famille_produit'])
            else: 
                query= """UPDATE dim_famille_produit SET lib_famille_produit = '%s'"""%(row['lib_famille_produit'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_categorie_produit dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
   
def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_categorie_produit")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_categorie_produit, lib_categorie_produit, fk_famille_produit 
    FROM  e_25_3jexxxxxxx.t_categorie_produit t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)   
    with open("t_categorie_produit.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_categorie_produit dans la base e_25_3pexxxxxxx (dim_categorie_produit)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_categorie_produit.csv")
    with open('t_categorie_produit.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_categorie_produit from dim_categorie_produit where id_categorie_produit = '%s'"
            cursor.execute(request, (row['id_categorie_produit']))
            logging.debug(request)
            logging.debug(row['id_categorie_produit'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_categorie_produit (id_categorie_produit, lib_categorie_produit, fk_famille_produit) VALUES (%i, '%s', %i)"""%(row['id_categorie_produit'], row['lib_categorie_produit'], row['fk_famille_produit'])
            else: 
                query= """UPDATE dim_categorie_produit SET lib_categorie_produit = '%s', fk_famille_produit = %i"""%(row['lib_categorie_produit'], row['fk_famille_produit'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_sous_categorie_produit dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_sous_categorie_produit")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_sous_categorie_produit, lib_sous_categorie_produit, id_categorie_produit 
    FROM  e_25_3jexxxxxxx.t_sous_categorie_produit t
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query)   
    with open("t_sous_categorie_produit.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_sous_categorie_produit dans la base e_25_3pexxxxxxx (dim_sous_categorie_produit)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_sous_categorie_produit.csv")
    with open('t_sous_categorie_produit.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_sous_categorie_produit from dim_sous_categorie_produit where id_sous_categorie_produit = '%s'"
            cursor.execute(request, (row['id_sous_categorie_produit']))
            logging.debug(request)
            logging.debug(row['id_sous_categorie_produit'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_sous_categorie_produit (id_sous_categorie_produit, lib_sous_categorie_produit, fk_categorie_produit) VALUES (%i, '%s', %i)"""%(row['id_sous_categorie_produit'], row['lib_sous_categorie_produit'], row['id_categorie_produit'])
            else: 
                query= """UPDATE dim_sous_categorie_produit SET lib_sous_categorie_produit = '%s', fk_categorie_produit = %i"""%(row['lib_sous_categorie_produit'], row['id_categorie_produit'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#t_produit dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging
    
def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de t_produit")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT id_produit, libelle, description, en_vente, en_achat, id_sous_categorie_produit 
    FROM  e_25_3jexxxxxxx.t_produit t 
    WHERE 1
    """
    logging.debug(query)
    cursor.execute(query) 
    with open("t_produit.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#t_produit dans la base e_25_3pexxxxxxx (dim_produit)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("t_produit.csv")
    with open('t_produit.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = "select id_produit from dim_produit where id_produit = '%s'"
            cursor.execute(request, (row['id_produit']))
            logging.debug(request)
            logging.debug(row['id_produit'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO dim_produit (id_produit, libelle, description, en_vente, en_achat, fk_sous_categorie_produit) VALUES (%i, '%s', '%s', %i, %i, %i)"""%(row['id_produit'], row['libelle'], row['description'], row['en_vente'], row['en_achat'], row['id_sous_categorie_produit'])
            else: 
                query= """UPDATE dim_produit SET libelle = '%s', description = '%s', en_vente = %i, en_achat = %i, fk_sous_categorie_produit = %i"""%(row['libelle'], row['description'], row['en_vente'], row['en_achat'], row['id_sous_categorie_produit'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


In [ ]:
#faits_ventes dans un fichier CSV
import csv
import mysql.connector
from config import MYSQL_OLTP_CREDENTIALS
import logging

def replace_apostrophe(value):
    """Replace apostrophes in strings only."""
    if isinstance(value, str):  # Only replace in strings
        return value.replace("'", "''")
    return value  # Leave other types unchanged (like int, float, etc.)

connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLTP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLTP_CREDENTIALS['database'],
    user=(MYSQL_OLTP_CREDENTIALS['user']),
    password=(MYSQL_OLTP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLTP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3jexxxxxxx est ouverte")
    cursor = connection.cursor()

    logging.info("Ecriture du fichier csv de faits_ventes")    

    # i must escape % by doubling it ! %M is for MySQL, not for Python !
    query="""
    SELECT sb.id_magasin, sb.id_produit, sb.id_temps, sb.snb, sb.sca, sb.smarge
    FROM (
      SELECT id_magasin,id_produit, CONCAT(cast( YEAR(date_commande_client) AS char) ,lpad(MONTH(date_commande_client) , 2, 0)) as id_temps, SUM(quantite) as snb, SUM(pu_ht* quantite-reduction) as sca, SUM((pu_ht-prix_revient)*quantite-reduction) as smarge 
      FROM (e_25_3jexxxxxxx.t_ligne_facture
      INNER JOIN e_25_3jexxxxxxx.t_facture 
      ON((e_25_3jexxxxxxx.t_ligne_facture.num_commande = e_25_3jexxxxxxx.t_facture.num_commande)))
      WHERE 1
      GROUP BY id_magasin,id_produit, CONCAT(cast( YEAR(date_commande_client) AS char) ,lpad(MONTH(date_commande_client), 2, 0))
    ) as sb
    """
    logging.debug(query)
    cursor.execute(query)   
    with open("faits_ventes.csv", "w", newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file, quoting=csv.QUOTE_NONNUMERIC)

        # Write headers (column names)
        csv_writer.writerow([i[0] for i in cursor.description])

        # Write data rows, replacing single quotes in strings only
        for row in cursor:
            cleaned_row = [replace_apostrophe(value) for value in row]
            csv_writer.writerow(cleaned_row)
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record into file or read database: {}".format(error))
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")

In [ ]:
#faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)
import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging
    
connection = None
#logging.getlogger().setLevel(logging.DEBUG)
#logging.getlogger().setLevel(Logging.INFO)
logging.getLogger().setLevel(logging.DEBUG)
try:
    connection = mysql.connector.connect(
    host=(MYSQL_OLAP_CREDENTIALS['host']),
    port=3312,
    database=MYSQL_OLAP_CREDENTIALS['database'],
    user=(MYSQL_OLAP_CREDENTIALS['user']),
    password=(MYSQL_OLAP_CREDENTIALS['password']),
    auth_plugin=(MYSQL_OLAP_CREDENTIALS['auth_plugin'])
    )
    logging.info("La connection a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()
    i = 1
    logging.info("faits_ventes.csv")
    with open('faits_ventes.csv', 'r') as read_obj:
        csv_reader = csv.DictReader(read_obj, quoting=csv.QUOTE_NONNUMERIC)
        for row in csv_reader:
            request = """
            select id_magasin, id_produit, id_temps 
            from faits_ventes 
            where id_magasin = %s
                and id_produit = %s
                and id_temps = %s
            """
            cursor.execute(request, (row['id_magasin'], row['id_produit'], row['id_temps']))
            logging.debug(request)
            logging.debug(row['id_magasin'], row['id_produit'], row['id_temps'])
            exist_object = cursor.fetchone()
            if exist_object is None:
                query = """INSERT INTO faits_ventes (id_magasin, id_produit, id_temps, ventes_reel, ca_reel, marge_reel) VALUES (%i, %i, '%s', %f, %f, %f)"""%(row['id_magasin'], row['id_produit'], row['id_temps'], row['snb'], row['sca'], row['smarge'])
            else: 
                query= """UPDATE faits_ventes SET ventes_reel = %f, ca_reel = %f, marge_reel = %f"""%(row['snb'], row['sca'], row['smarge'])
            logging.debug(query)
            cursor.execute(query)
            connection.commit()
            
except (csv.Error,mysql.connector.Error,PermissionError,OSError) as error:
    logging.error("Failed to insert record to database rollback or write file: {}".format(error))
    # annulation des changements pour cause exceptionnelle
    connection.rollback()
finally:
    # fermeture de la connexion au serveur.
    if connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermee")


voici ce qui reste à implanter :
```
-- alim dim_pays
INSERT INTO  e_25_3pexxxxxxx.`dim_pays`(`id_pays`, `lib_pays`, `iso_3166_1_numeric`, `iso_3166_alpha_2`, `fichier_image_carte_pays`)
SELECT `id_pays`, `lib_pays`, `iso_3166_1_numeric`, `iso_3166_alpha_2`,`fichier_image_carte_pays` 
FROM  e_25_3jexxxxxxx.`t_pays` t 
WHERE 1
ON DUPLICATE KEY UPDATE `lib_pays`=t.`lib_pays`, `iso_3166_1_numeric`=t.`iso_3166_1_numeric`, `iso_3166_alpha_2`=t.`iso_3166_alpha_2`, `fichier_image_carte_pays`=t.`fichier_image_carte_pays`
;

-- alim dim_comm
INSERT INTO  e_25_3pexxxxxxx.`dim_geographique_com`(`id_region_com`, `lib_region_com`, `date_debut_valid_com`, `date_fin_valid_com`, `fichier_image_carte_regcom`, `id_pays`)
SELECT `id_region_com`, `lib_region_com`, `date_debut_valid_com`, `date_fin_valid_com`, `fichier_image_carte_regcom`, `id_pays` 
FROM  e_25_3jexxxxxxx.`t_region_commerciale` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_region_com`=t.`lib_region_com` , `date_debut_valid_com`=t.`date_debut_valid_com`, `date_fin_valid_com`=t.`date_fin_valid_com`,  `fichier_image_carte_regcom`=t.`fichier_image_carte_regcom`, `id_pays`=t.`id_pays`
;


-- alim dim_adm
INSERT INTO e_25_3pexxxxxxx.`dim_geographique_admin`(`id_region_admin`, `lib_region_admin`, `date_debut_valid_admin`, `date_fin_valid_admin`, `fichier_image_carte_regadm`, `sas_map_id`, `sas_map_name`, `id_pays`)
SELECT `id_region_admin`, `lib_region_admin`, `date_debut_valid_admin`, `date_fin_valid_admin`, `fichier_image_carte_regadm`, `sas_map_id`, `sas_map_name`, `id_pays` 
FROM  e_25_3jexxxxxxx.`t_region_administrative` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_region_admin`=t.`lib_region_admin`, `date_debut_valid_admin`=t.`date_debut_valid_admin`, `date_fin_valid_admin`=t.`date_fin_valid_admin`, `fichier_image_carte_regadm`=t.`fichier_image_carte_regadm`, `sas_map_id`=t.`sas_map_id`, `sas_map_name`=t.`sas_map_name`, `id_pays`=t.`id_pays` 
;

-- alim dim_dept
INSERT INTO e_25_3pexxxxxxx.`dim_departement`(`id_departement`, `code_departement`, `lib_departement`, `id_region_admin1`, `id_region_admin2`, `id_region_com`) 
SELECT `id_departement`, `code_departement`, `lib_departement`, `id_region_admin1`, `id_region_admin2`,`id_region_com` 
FROM  e_25_3jexxxxxxx.`t_departement` t
WHERE 1
ON DUPLICATE KEY UPDATE `code_departement`=t.`code_departement`, `lib_departement`=t.`lib_departement`, `id_region_admin1`=t.`id_region_admin1`, `id_region_admin2`=t.`id_region_admin2`, `id_region_com`=t.`id_region_com`
;

-- alim dim_magasin
INSERT INTO  e_25_3pexxxxxxx.`dim_magasin`(`id_magasin`, `id_enseigne`, `actif`, `date_ouverture`, `date_fermeture`, `emplacements`, `nb_caisses`, `ville`, `dep`, `fichier_image_carte_magasin`)
SELECT `id_magasin`, `id_enseigne`, `actif`, `date_ouverture`, `date_fermeture`, `emplacements`, `nb_caisses`, `ville`, `dep`, `fichier_image_carte_magasin` 
FROM  e_25_3jexxxxxxx.`t_magasin` t 
WHERE 1
ON DUPLICATE KEY UPDATE `id_enseigne`=t.`id_enseigne`, `actif`=t.`actif`, `date_ouverture`=t.`date_ouverture`, `date_fermeture`=t.`date_fermeture`, `emplacements`=t.`emplacements`, `nb_caisses`=t.`nb_caisses`, `ville`=t.`ville`, `dep`=t.`dep`, `fichier_image_carte_magasin`=t.`fichier_image_carte_magasin`
;

-- alim dim_famille
INSERT INTO  e_25_3pexxxxxxx.`dim_famille_produit`(`id_famille_produit`, `lib_famille_produit`)
SELECT `id_famille_produit`,`lib_famille_produit` 
FROM  e_25_3jexxxxxxx.`t_famille_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `lib_famille_produit`=t.`lib_famille_produit`
;

-- alim dim_categorie
INSERT INTO  e_25_3pexxxxxxx.`dim_categorie_produit`(`id_categorie_produit`, `lib_categorie_produit`, `fk_famille_produit`) 
SELECT `id_categorie_produit`, `lib_categorie_produit`, `fk_famille_produit` 
FROM  e_25_3jexxxxxxx.`t_categorie_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `id_categorie_produit`=t.`id_categorie_produit`, `lib_categorie_produit`=t.`lib_categorie_produit`, `fk_famille_produit`=t.`fk_famille_produit`
;

-- alim dim_sous_categorie
INSERT INTO  e_25_3pexxxxxxx.`dim_sous_categorie_produit`(`id_sous_categorie_produit`, `lib_sous_categorie_produit`, `fk_categorie_produit`) 
SELECT `id_sous_categorie_produit`, `lib_sous_categorie_produit`, `id_categorie_produit` 
FROM  e_25_3jexxxxxxx.`t_sous_categorie_produit` t
WHERE 1
ON DUPLICATE KEY UPDATE `id_sous_categorie_produit`=t.`id_sous_categorie_produit`, `lib_sous_categorie_produit`=t.`lib_sous_categorie_produit`, `fk_categorie_produit`=t.`id_categorie_produit`
;

-- alim dim_produit
INSERT INTO e_25_3pexxxxxxx.`dim_produit`(`id_produit`, `libelle`, `description`, `en_vente`, `en_achat`, `fk_sous_categorie_produit`)
SELECT `id_produit`, `libelle`, `description`, `en_vente`, `en_achat`, `id_sous_categorie_produit` 
FROM  e_25_3jexxxxxxx.`t_produit` t 
WHERE 1
ON DUPLICATE KEY UPDATE `libelle`=t.`libelle`, `description`=t.`description`, `en_vente`=t.`en_vente`, `en_achat`=t.`en_achat`, `fk_sous_categorie_produit`=t.`id_sous_categorie_produit`
;

-- alim faits_ventes :
-- Réalisé, UPSERT (à préférer)
INSERT INTO e_25_3pexxxxxxx.`faits_ventes`(`id_magasin`, `id_produit`, `id_temps`, `ventes_reel`, `ca_reel`, `marge_reel`) 
SELECT sb.`id_magasin`, sb.`id_produit`, sb.id_temps, sb.snb, sb.sca, sb.smarge
FROM (
  SELECT `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`) , 2, 0)) as id_temps, SUM(`quantite`) as snb, SUM(`pu_ht`* `quantite`-`reduction`) as sca, SUM((`pu_ht`-`prix_revient`)*`quantite`-`reduction`) as smarge 
  FROM (`e_25_3jexxxxxxx`.`t_ligne_facture` 
  INNER JOIN `e_25_3jexxxxxxx`.`t_facture` 
  ON((`e_25_3jexxxxxxx`.`t_ligne_facture`.`num_commande` = `e_24_3jexxxxxxx`.`t_facture`.`num_commande`)))
  WHERE 1
  GROUP BY `id_magasin`,`id_produit`, CONCAT(cast( YEAR(`date_commande_client`) AS char) ,lpad(MONTH(`date_commande_client`), 2, 0))
) as sb
ON DUPLICATE KEY UPDATE `ventes_reel`=sb.snb, `ca_reel`=sb.sca, `marge_reel`=sb.smarge
;
```
A noter que la dernière instruction utilise une sous-requête dans la clause FROM, ce que l'on appelle une vue implicite. Ceci est fait pour que les données calculées melant les fonctions d'agrégations (`SUM`, `AVG`, `COUNT`, `MIN`, `MAX`) et des calculs arithmétiques soient disponibles à la fois pour le `INSERT` et à la fois pour le `UPDATE`.

La sous-requête est à utiliser comme requête de l'étape de création de fichier CSV en ajoutant des alias de colonne si vous constatez ce besoin. Dans l'étape suivante, vous pouvez utiliser l'ordre paramétré ou la chaîne interpolée `INSERT INTO faits_ventes ... VALUES ... ON DUPLICATE KEY UPDATE ... `.

Cette solution est meilleure que l'utilisation de tables temporaires dans la base de données décisionnelle pour recréer la structure de la base de production. Il n'est pas gagné que l'administrateur du serveur de données vous ait donné ce privilège.


## ETL_3
**Chaque année, le 5 janvier, importation dans le schéma décisionnel depuis des fichiers CSV des indicateurs concernant les objectifs dans la table de faits concernant l'année qui arrive. C'est l'outil de simulation qui vous génère le fichier CSV annuel des objectifs.**

ETL_3 : soit limitation des changements des données à l'année courante, soit pas de limitation
    - vérification de dim_temps et ajout éventuels des 12 lignes d'une nouvelle année (ne se présentera que le 5 janvier 2026 !)
    - lecture des données depuis un fichier CSV avec jointure sur `libelle` des produits et `ville` des magasins et interprétation des noms de mois en français ;
    - vérification de `dim_produit_star` (verification de `dim_produit`, `dim_sous_categorie`, `dim_categorie`, `dim_famille` puis `CALL maj_dim_produit_star();`) ; 
    - vérification de `dim_magasin_star` (verification de `dim_magasin`, `dim_departement`, `dim_geographique_com`, `dim_geographique_admin`, `dim_pays` puis `CALL maj_dim_magasin_star();`) ;
    - vérification de `faits_ventes_star` pour les indicateurs objectifs (vérification de `faits_ventes` puis `CALL maj_faits_ventes_star(); CALL maj_securite_star(); CALL maj_dwr_faits_ventes_star();`).
    
- Pour ETL 3 :
Je récupère la date courante. J'extrais l'année et je constitue `ref_annee`. La condition devient `'''Cle primaire IN (SELECT part_cle_primaire FROM faits WHERE id_temps LIKE '%s')'''%(str(ref_annee))`.

Pour la mise à jour du schéma en étoile, il est plus robuste de programmer directement la façon compatible avec le serveur de production.

Pour le processus ETL 3, s'il n'y a pas de limitation temporelle, il faut charger dans l'ordre tous les fichiers annee.csv du répertoire courant et les traiter un par un.

Le fichier `annee.csv` a normalement été créé par le directeur marketing du groupe DARTIES à partir des données du schéma de production. Il faut donc récupérer les données de `t_magasin` et de `t_produit` dans des fichiers csv pour mettre à jour `dim_magasin` et `dim_produit` pour pouvoir faire la "jointure pas naturelle" sur les libellés des produits et les villes des magasins avec les tables `dim_`.
    

In [ ]:
MOIS_MAP = {
    "JANVIER": "01",
    "FÉVRIER": "02",
    "MARS": "03",
    "AVRIL": "04",
    "MAI": "05",
    "JUIN": "06",
    "JUILLET": "07",
    "AOÛT": "08",
    "SEPTEMBRE": "09",
    "OCTOBRE": "10",
    "NOVEMBRE": "11",
    "DÉCEMBRE": "12"
}

def get_id_temps(annee, mois_lettres):
    mois = MOIS_MAP[mois_lettres.strip().upper()]
    return str(annee) + mois


In [ ]:
def get_id_magasin(cursor, ville_magasin):
    query = """
        SELECT id_magasin
        FROM dim_magasin
        WHERE ville = %s
    """
    cursor.execute(query, (ville_magasin,))
    result = cursor.fetchone()
    return result[0] if result else None


In [ ]:
def get_id_produit(cursor, libelle_produit):
    query = """
        SELECT id_produit
        FROM dim_produit
        WHERE libelle = %s
    """
    cursor.execute(query, (libelle_produit,))
    result = cursor.fetchone()
    return result[0] if result else None

In [ ]:
# faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)

import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging

connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )

    logging.info("La connexion a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()

    # range s'arrete un avant -> 2022 uniquement
    for annee in range(2022, 2023):

        logging.info("Traitement fichier : %s.csv", annee)

        with open(str(annee) + ".csv", "r") as read_obj:
            csv_reader = csv.DictReader(read_obj)

            for row in csv_reader:

                id_temps = get_id_temps(
                    row["ANNEE"],
                    row["MOIS_EN_LETTRES"]
                )

                id_magasin = get_id_magasin(
                    cursor,
                    row["VILLE_MAGASIN"]
                )

                id_produit = get_id_produit(
                    cursor,
                    row["LIBELLE_PRODUIT"]
                )

                if id_magasin is None or id_produit is None:
                    logging.warning("ID manquant, ligne ignorée : %s", row)
                    continue

                select_query = """
                    SELECT 1
                    FROM faits_ventes
                    WHERE id_magasin = %s
                      AND id_produit = %s
                      AND id_temps = %s
                """
                cursor.execute(
                    select_query,
                    (id_magasin, id_produit, id_temps)
                )

                exists = cursor.fetchone()

                if exists is None:
                    insert_query = """
                        INSERT INTO faits_ventes
                        (id_magasin, id_produit, id_temps, ventes_objectif, ca_objectif, marge_objectif)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(
                        insert_query,
                        (
                            id_magasin,
                            id_produit,
                            id_temps,
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"]
                        )
                    )
                else:
                    update_query = """
                        UPDATE faits_ventes
                        SET ventes_objectif = %s,
                            ca_objectif = %s,
                            marge_objectif = %s
                        WHERE id_magasin = %s
                          AND id_produit = %s
                          AND id_temps = %s
                    """
                    cursor.execute(
                        update_query,
                        (
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"],
                            id_magasin,
                            id_produit,
                            id_temps
                        )
                    )
        connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error(
        "Failed to insert record to database rollback or write file: %s",
        error
    )
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée")


In [ ]:
# faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)

import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging

connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )

    logging.info("La connexion a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()

    # range s'arrete un avant -> 2022 uniquement
    for annee in range(2019, 2020):

        logging.info("Traitement fichier : %s.csv", annee)

        with open(str(annee) + ".csv", "r") as read_obj:
            csv_reader = csv.DictReader(read_obj)

            for row in csv_reader:

                id_temps = get_id_temps(
                    row["ANNEE"],
                    row["MOIS_EN_LETTRES"]
                )

                id_magasin = get_id_magasin(
                    cursor,
                    row["VILLE_MAGASIN"]
                )

                id_produit = get_id_produit(
                    cursor,
                    row["LIBELLE_PRODUIT"]
                )

                if id_magasin is None or id_produit is None:
                    logging.warning("ID manquant, ligne ignorée : %s", row)
                    continue

                select_query = """
                    SELECT 1
                    FROM faits_ventes
                    WHERE id_magasin = %s
                      AND id_produit = %s
                      AND id_temps = %s
                """
                cursor.execute(
                    select_query,
                    (id_magasin, id_produit, id_temps)
                )

                exists = cursor.fetchone()

                if exists is None:
                    insert_query = """
                        INSERT INTO faits_ventes
                        (id_magasin, id_produit, id_temps, ventes_objectif, ca_objectif, marge_objectif)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(
                        insert_query,
                        (
                            id_magasin,
                            id_produit,
                            id_temps,
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"]
                        )
                    )
                else:
                    update_query = """
                        UPDATE faits_ventes
                        SET ventes_objectif = %s,
                            ca_objectif = %s,
                            marge_objectif = %s
                        WHERE id_magasin = %s
                          AND id_produit = %s
                          AND id_temps = %s
                    """
                    cursor.execute(
                        update_query,
                        (
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"],
                            id_magasin,
                            id_produit,
                            id_temps
                        )
                    )
        connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error(
        "Failed to insert record to database rollback or write file: %s",
        error
    )
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée")


In [ ]:
# faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)

import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging

connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )

    logging.info("La connexion a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()

    # range s'arrete un avant -> 2022 uniquement
    for annee in range(2020, 2021):

        logging.info("Traitement fichier : %s.csv", annee)

        with open(str(annee) + ".csv", "r") as read_obj:
            csv_reader = csv.DictReader(read_obj)

            for row in csv_reader:

                id_temps = get_id_temps(
                    row["ANNEE"],
                    row["MOIS_EN_LETTRES"]
                )

                id_magasin = get_id_magasin(
                    cursor,
                    row["VILLE_MAGASIN"]
                )

                id_produit = get_id_produit(
                    cursor,
                    row["LIBELLE_PRODUIT"]
                )

                if id_magasin is None or id_produit is None:
                    logging.warning("ID manquant, ligne ignorée : %s", row)
                    continue

                select_query = """
                    SELECT 1
                    FROM faits_ventes
                    WHERE id_magasin = %s
                      AND id_produit = %s
                      AND id_temps = %s
                """
                cursor.execute(
                    select_query,
                    (id_magasin, id_produit, id_temps)
                )

                exists = cursor.fetchone()

                if exists is None:
                    insert_query = """
                        INSERT INTO faits_ventes
                        (id_magasin, id_produit, id_temps, ventes_objectif, ca_objectif, marge_objectif)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(
                        insert_query,
                        (
                            id_magasin,
                            id_produit,
                            id_temps,
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"]
                        )
                    )
                else:
                    update_query = """
                        UPDATE faits_ventes
                        SET ventes_objectif = %s,
                            ca_objectif = %s,
                            marge_objectif = %s
                        WHERE id_magasin = %s
                          AND id_produit = %s
                          AND id_temps = %s
                    """
                    cursor.execute(
                        update_query,
                        (
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"],
                            id_magasin,
                            id_produit,
                            id_temps
                        )
                    )
        connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error(
        "Failed to insert record to database rollback or write file: %s",
        error
    )
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée")


In [ ]:
# faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)

import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging

connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )

    logging.info("La connexion a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()

    # range s'arrete un avant -> 2022 uniquement
    for annee in range(2021, 2022):

        logging.info("Traitement fichier : %s.csv", annee)

        with open(str(annee) + ".csv", "r") as read_obj:
            csv_reader = csv.DictReader(read_obj)

            for row in csv_reader:

                id_temps = get_id_temps(
                    row["ANNEE"],
                    row["MOIS_EN_LETTRES"]
                )

                id_magasin = get_id_magasin(
                    cursor,
                    row["VILLE_MAGASIN"]
                )

                id_produit = get_id_produit(
                    cursor,
                    row["LIBELLE_PRODUIT"]
                )

                if id_magasin is None or id_produit is None:
                    logging.warning("ID manquant, ligne ignorée : %s", row)
                    continue

                select_query = """
                    SELECT 1
                    FROM faits_ventes
                    WHERE id_magasin = %s
                      AND id_produit = %s
                      AND id_temps = %s
                """
                cursor.execute(
                    select_query,
                    (id_magasin, id_produit, id_temps)
                )

                exists = cursor.fetchone()

                if exists is None:
                    insert_query = """
                        INSERT INTO faits_ventes
                        (id_magasin, id_produit, id_temps, ventes_objectif, ca_objectif, marge_objectif)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(
                        insert_query,
                        (
                            id_magasin,
                            id_produit,
                            id_temps,
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"]
                        )
                    )
                else:
                    update_query = """
                        UPDATE faits_ventes
                        SET ventes_objectif = %s,
                            ca_objectif = %s,
                            marge_objectif = %s
                        WHERE id_magasin = %s
                          AND id_produit = %s
                          AND id_temps = %s
                    """
                    cursor.execute(
                        update_query,
                        (
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"],
                            id_magasin,
                            id_produit,
                            id_temps
                        )
                    )
        connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error(
        "Failed to insert record to database rollback or write file: %s",
        error
    )
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée")


In [ ]:
# faits_ventes dans la base e_25_3pexxxxxxx (faits_ventes)

import csv
import mysql.connector
from config import MYSQL_OLAP_CREDENTIALS
import logging

connection = None
logging.getLogger().setLevel(logging.DEBUG)

try:
    connection = mysql.connector.connect(
        host=MYSQL_OLAP_CREDENTIALS['host'],
        port=3312,
        database=MYSQL_OLAP_CREDENTIALS['database'],
        user=MYSQL_OLAP_CREDENTIALS['user'],
        password=MYSQL_OLAP_CREDENTIALS['password'],
        auth_plugin=MYSQL_OLAP_CREDENTIALS['auth_plugin']
    )

    logging.info("La connexion a e_25_3pexxxxxxx est ouverte")
    cursor = connection.cursor()

    # range s'arrete un avant -> 2022 uniquement
    for annee in range(2023, 2024):

        logging.info("Traitement fichier : %s.csv", annee)

        with open(str(annee) + ".csv", "r") as read_obj:
            csv_reader = csv.DictReader(read_obj)

            for row in csv_reader:

                id_temps = get_id_temps(
                    row["ANNEE"],
                    row["MOIS_EN_LETTRES"]
                )

                id_magasin = get_id_magasin(
                    cursor,
                    row["VILLE_MAGASIN"]
                )

                id_produit = get_id_produit(
                    cursor,
                    row["LIBELLE_PRODUIT"]
                )

                if id_magasin is None or id_produit is None:
                    logging.warning("ID manquant, ligne ignorée : %s", row)
                    continue

                select_query = """
                    SELECT 1
                    FROM faits_ventes
                    WHERE id_magasin = %s
                      AND id_produit = %s
                      AND id_temps = %s
                """
                cursor.execute(
                    select_query,
                    (id_magasin, id_produit, id_temps)
                )

                exists = cursor.fetchone()

                if exists is None:
                    insert_query = """
                        INSERT INTO faits_ventes
                        (id_magasin, id_produit, id_temps, ventes_objectif, ca_objectif, marge_objectif)
                        VALUES (%s, %s, %s, %s, %s, %s)
                    """
                    cursor.execute(
                        insert_query,
                        (
                            id_magasin,
                            id_produit,
                            id_temps,
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"]
                        )
                    )
                else:
                    update_query = """
                        UPDATE faits_ventes
                        SET ventes_objectif = %s,
                            ca_objectif = %s,
                            marge_objectif = %s
                        WHERE id_magasin = %s
                          AND id_produit = %s
                          AND id_temps = %s
                    """
                    cursor.execute(
                        update_query,
                        (
                            row["OBJECTIF_NB_VENTES"],
                            row["OBJECTIF_CA_VENTES"],
                            row["OBJECTIF_MARGE_VENTES"],
                            id_magasin,
                            id_produit,
                            id_temps
                        )
                    )
        connection.commit()

except (csv.Error, mysql.connector.Error, PermissionError, OSError) as error:
    logging.error(
        "Failed to insert record to database rollback or write file: %s",
        error
    )
    if connection:
        connection.rollback()

finally:
    if connection and connection.is_connected():
        cursor.close()
        connection.close()
        logging.info("La connexion est fermée")


### Archive à rendre
Il faut rendre dans l'arborescence `6-ETL pour les processus ETL 1, 2 et 3 en Python`, un fichier `ALire.txt` qui explique, énonce vos résultats atteints par rapport à l'utilisation des ordres interbases d'alimentation du schema décisionnel en flocon depuis le schéma de production. Vous indiquerez aussi comment vous alimentez le schéma en étoile à partir du schéma en flocon. Vous expliquerez comment fonctionnent vos processus ETL dans le contexte du serveur de développement et dans le contexte du serveur de production. Vous ajouterez vos fichiers `*.py` ou vos calepins jupyter `*.ipynb`.

<a id='seance_mondrian'></a>
## Utilisation d'un serveur Mondrian

L'URL [ici](http://www-i.univ-ubs.fr/etud/projets/p_22_md_301java/xavier/index.html) vous permet d'accéder à un serveur Mondrian branché au cas DARTIES.

Il faudrait dans ce début de séance sauvegarder l'état courant sur le serveur local des données :
1. Arret du serveur MySQL de développement
2. dans le shell MySQL, faire la commande : `sauve_local_data`

Ainsi à la fin de séance, on pourra restaurer cet état courant :
1. Arret du serveur MySQL de développement
2. dans le shell MySQL, faire la commande : `restauration_local_data`

Préalables : dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=e_25_3pexxxxxxx&server=2):
1. Il faut d'abord, alimenter le schéma décisionnel en flocon (voir [ici](#seance_SQL_dec))
2. Il faut ensuite alimenter le schéma décisionnel en étoile. Pour cela, définir et exécuter les procédures stockées suivantes :

```
-- Vous n'avez pas le droit de manipuler des procédures sur le serveur de production
DELIMITER ;
DROP PROCEDURE IF EXISTS maj_dim_magasin_star;
DELIMITER |
CREATE PROCEDURE maj_dim_magasin_star()
BEGIN
    ALTER TABLE faits_ventes_star DROP FOREIGN KEY FK_faits_ventes_SR_dim_magasin;
    ALTER TABLE securite_star DROP FOREIGN KEY securite_star_i1_fkey ;
    TRUNCATE dim_magasin_star;
    ALTER TABLE faits_ventes_star ADD CONSTRAINT FK_faits_ventes_SR_dim_magasin FOREIGN KEY (ID_MAGASIN)
      REFERENCES dim_magasin_star (ID_MAGASIN);
    ALTER TABLE securite_star ADD CONSTRAINT securite_star_i1_fkey FOREIGN KEY (id_magasin)
      REFERENCES dim_magasin_star(id_magasin);
    INSERT INTO dim_magasin_star
SELECT DISTINCT id_magasin,
    actif,
    date_ouverture ,
    date_fermeture ,
    emplacements ,
    nb_caisses ,
    ville ,
    fichier_image_carte_magasin ,
    date_maj ,
  dim_enseigne.id_enseigne,
  lib_enseigne,
    fichier_image_logo_enseigne,
    datemaj_enseigne,
  id_departement,
    code_departement,
    lib_departement ,
    datemaj_dep,
    anc_admin.id_region_admin as id_region_anc_admin,
    anc_admin.lib_region_admin as lib_region_anc_admin,
    anc_admin.date_debut_valid_admin as date_debut_valid_anc_admin,
    anc_admin.date_fin_valid_admin as date_fin_valid_anc_admin,
    anc_admin.fichier_image_carte_regadm as fichier_img_anc_reg_admin,
    anc_admin.sas_map_id as sas_map_id_anc_reg_admin,
    anc_admin.sas_map_name as sas_map_name_anc_reg_admin,
    anc_admin.datemaj_geo_admin as datemaj_geo_anc_reg_admin,
  nouv_admin.id_region_admin as id_region_nouv_admin,
    nouv_admin.lib_region_admin as lib_region_nouv_admin,
    nouv_admin.date_debut_valid_admin as date_debut_valid_nouv_admin,
    nouv_admin.date_fin_valid_admin as date_fin_valid_nouv_admin,
    nouv_admin.fichier_image_carte_regadm as fichier_img_nouv_reg_admin,
    nouv_admin.sas_map_id as sas_map_id_nouv_reg_admin,
    nouv_admin.sas_map_name as sas_map_name_nouv_reg_admin,
    nouv_admin.datemaj_geo_admin as datemaj_geo_nouv_reg_admin,   
  dim_geographique_com.id_region_com,
    lib_region_com,
    date_debut_valid_com ,
    date_fin_valid_com ,
    fichier_image_carte_regcom,
    datemaj_geo_com,
    dim_pays.id_pays,
    lib_pays,
    iso_3166_1_numeric,
    iso_3166_alpha_2,
    fichier_image_carte_pays,
    datemaj_pays
  from dim_magasin, dim_enseigne, dim_departement, dim_geographique_com, dim_geographique_admin anc_admin, dim_geographique_admin nouv_admin, dim_pays
  where dim_magasin.id_enseigne=dim_enseigne.id_enseigne
  and dim_departement.id_departement=dim_magasin.dep
  and dim_geographique_com.id_region_com=dim_departement.id_region_com
  and anc_admin.id_region_admin=id_region_admin1
  and nouv_admin.id_region_admin=id_region_admin2
  and dim_geographique_com.id_pays=dim_pays.id_pays
  ;
END |
DELIMITER ;
DROP PROCEDURE IF EXISTS maj_dim_produit_star;
DELIMITER |
CREATE PROCEDURE maj_dim_produit_star()
BEGIN
    ALTER TABLE faits_ventes_star DROP FOREIGN KEY FK_faits_ventes_STAR_DIM_PRODUIT;
    TRUNCATE dim_produit_star;
    ALTER TABLE faits_ventes_star ADD CONSTRAINT FK_faits_ventes_STAR_DIM_PRODUIT FOREIGN KEY (ID_PRODUIT)
      REFERENCES dim_produit_star (ID_PRODUIT);
    INSERT INTO dim_produit_star
    SELECT DISTINCT
    id_produit,libelle,description,en_vente,en_achat,date_maj,
    id_sous_categorie_produit,lib_sous_categorie_produit,datemaj_sous_categorie,
    id_categorie_produit,lib_categorie_produit,datemaj_categorie,
    id_famille_produit,lib_famille_produit,datemaj_famille
    FROM dim_produit
    INNER JOIN dim_sous_categorie_produit
    ON fk_sous_categorie_produit=id_sous_categorie_produit
    INNER JOIN dim_categorie_produit
    ON fk_categorie_produit=id_categorie_produit
    INNER JOIN dim_famille_produit
    ON fk_famille_produit=id_famille_produit
    ;
END |
DELIMITER ;
DROP PROCEDURE IF EXISTS maj_faits_ventes_star;
DELIMITER |
CREATE PROCEDURE maj_faits_ventes_star()
BEGIN
    TRUNCATE faits_ventes_star;
    INSERT INTO faits_ventes_star
    SELECT * FROM faits_ventes;
END |
DELIMITER ;
DROP PROCEDURE IF EXISTS maj_securite_star;
DELIMITER |
CREATE PROCEDURE maj_securite_star()
BEGIN
    TRUNCATE securite_star;
    INSERT INTO securite_star
    SELECT * FROM securite;
END |
DELIMITER ;
DROP PROCEDURE IF EXISTS maj_dwr_faits_ventes_star;
DELIMITER |
CREATE PROCEDURE maj_dwr_faits_ventes_star()

COMMENT 'Refresh pivoted data like a materialized view'
BEGIN
    TRUNCATE dwr_faits_ventes_star;
    INSERT INTO dwr_faits_ventes_star
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'VENTES' AS INDICATEUR , SUM(VENTES_OBJECTIF) AS OBJECTIF , SUM(VENTES_REEL)AS REEL,NULL AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW()) as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS, DATE_MAJ
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'CA' AS INDICATEUR , SUM(CA_OBJECTIF) AS OBJECTIF , SUM(CA_REEL)AS REEL,NULL AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW()) as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , DATE_MAJ
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'MARGE' AS INDICATEUR , AVG(MARGE_OBJECTIF) AS OBJECTIF , AVG(MARGE_REEL) AS REEL,NULL AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW()) as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , DATE_MAJ
    ORDER by 1,2,3,4
    ;
END |

DELIMITER ;
CALL maj_dim_magasin_star();
CALL maj_dim_produit_star();
CALL maj_faits_ventes_star();
```

Il faudrait personnaliser les utilisateurs sur le serveur de développement MySQL pour qu'ils correspondent à ceux disponibles pour le serveur de production MySQL.

Dans la fenêtre du Shell de MySQL faire les commandes :
1. `rename_user_database e_25_3pexxxxxxx e_25_3p%USERNAME% de_25_3pexxxxxxxd`
2. `rename_user_database e_25_3jexxxxxxx e_25_3j%USERNAME% de_25_3jexxxxxxxd`

Comme cela vos utilisateurs/bases de donnéees partagent les mêmes noms qu'ils soient sur le serveur MySQL local ou sur le serveur MySQL de production.

A partir de maintenant :
- **Abric--Segarra Maxence doit remplacer e_25_3pexxxxxxx par e_25_3pe2401441 et e_25_3jexxxxxxx par e_24_3je2401441**.
- **Cheruel Jocelyn doit remplacer e_25_3pexxxxxxx par e_25_3pe2401348 et e_25_3jexxxxxxx par e_25_3je2401348**.

Vous pouvez vous rendre sur la [page dédiée](https://moodle.univ-ubs.fr/mod/page/view.php?id=396281) du cours sur le serveur Mondrian.

Il faut étudier les tutoriels de cette page qui étaient à l'origine sous forme de séquences de photos d'écran contenues dans le répertoire `D:\doc`.

Il faut faire :
- La définition du cube mondrian sur le schéma en étoile avec Mondrian Schema Workbench. Un résultat intermédiaire est dans `e_22_3pexxxxxxx_star.xml` qui se trouve dans `D:\tools\apache-tomcat-10.0.22\webapps\p_22_md_09java\WEB-INF\schema`.
- Attention, il faut corriger ce résultat intermédiaire de préférence via Pentaho Schema Workbench voire l'enrichir. Faire plutôt référence à `24` qu'à `22`.
![Connexion à la base de données](connexion_p_duboism_view.png)
![Calculated Member](PSW_CM.png)
- Test du cube du serveur Mondrian local avec les differents clients (MS Excel, LibreOffice Calc, Jrubik, JPalo, Xavier, XMLa4js). Pour cela, il faut lancer le serveur tomcat 10 après avoir cloné la *webapps* `p_22_md_09java` en `e_25_3jexxxxxxx` pour y déployer `WEB-INF/schema/e_25_3pexxxxxxx_star.xml`, fait la configuration de la connexion au serveur mysql local via JDBC et référencé le premier fichier dans `WEB-INF/datasources.xml`, avoir déployé le pilote JDBC pour MySQL et modifié `xmla4js/samples/defaultXmlaUrl.js`, `xmla.wsdl` et `index.html`.
![configuration originelle datasources.xml](Mondrian_conf_0.png)
![configuration originelle index.html](Mondrian_conf_1.png)
![configuration originelle xmla.wsdl](Mondrian_conf_2.png)
![configuration originelle defaultXmlaUrl.js](Mondrian_conf_3.png)
- L'optimisation du cube mondrian avec Pentaho Agregation Designer. Pensez à sauvegarder les scripts SQL de création des tables agrégées que vous nommerez `WEB-INF/sql/e_25_3pexxxxxxx_star_agg_create.sql` et celui d'alimentation des tables agrégées que vous nommerez `WEB-INF/sql/e_25_3pexxxxxxx_star_agg_alim.sql`. MySQL comme Oracle a une limitation de la taille du nom des objets (base de données, table, colonne, fonction, procedures, etc.) à 30 caractères et non à 64 comme MS Access. L'application PAD cherche à respecter cette limitation mais des fois elle échoue à générer un nom unique pour les objets. Ainsi ce fut le cas pour la table agrégée `e_22_3pexx_ventes_3` qui avait un certain nombre de problèmes, ce qui peut amener à l'abandonner. Dans ce même répertoire `WEB-INF/sql/`, vous ferez un export sql via phpMyAdmin de vos bases [de production](http://localhost:96/phpmyadmin/db_export.php?db=e_25_3jexxxxxxx&server=2) et [décisionnelle](http://localhost:96/phpmyadmin/db_export.php?db=e_25_3pexxxxxxx&server=2) nommés respectivement `WEB-INF/sql/e_25_3jexxxxxxx.sql` et `WEB-INF/sql/e_25_3pexxxxxxx.sql`. Normalement ce dernier fichier doit aussi contenir et les tables agrégées et leur données. L'exécution du script d'alimentation des tables agrégées devient alors l'étape ultime des processus ETL n°2 (indicateurs réels) et n°3 (indicateurs objectifs) que vous avez écrits en Pyton. Puis vous devez faire l'adaptation de votre webapps sur le serveur tomcat 10 local dont la définition du cube est `e_25_3pexxxxxxx_star_agg.xml`. Dans le fichier `WEB-INF/datasources.xml`, vous pouvez ajouter un deuxième catalogue pour faire cohabiter la solution sans optimisation et la solution avec optimisation.  Via la journalisation du serveur Mondrian **local** obtenue par l'édition de `WEB-INF/classes/log4j.xml`, vérifiez la bonne utilisation de vos tables agrégées. Vous créerez un sous répertoire `logs` dans `WEB-INF`. Vous changerez la ligne 26 de configuration de journalisation pour pointer vers le fichier `D:/tools/apache-tomcat-10.0.22/webapps/e_24_3jexxxxxxx/WEB-INF/logs/mondrian_e_25_3pexxxxxxx.log`. Après le redémarrage du serveur tomcat local, et une interaction avec le client leger `Xavier`, constatez, dans le fichier de log, l'utilisation résiduelle de la table de fait `faits_ventes_star` et celle plus souvent des tables agrégées `e_22_3pexx_ventes_1`, `e_22_3pexx_ventes_2` et `e_22_3pexx_ventes_4` par exemple au niveau des entrées `[mondrian.sql]`. Vous pouvez voir les requêtes MDX générées par `Xavier` au niveau des entrées `[mondrian.mdx]`.

![configuration optimisée](Mondrian_conf_optim_log.png)

- La "publication" de votre cube mondrian sur votre espace `P:/e_25_3jexxxxxxx/private_html/webapps/`. Il faut que votre base de données MySQL `e_25_3pexxxxxxx` et en particulier les tables du schema en étoile soient remplies sur le serveur de production MySQL `projetud.univ-ubs.fr:3306`. Les fichiers ont donc été modifiés en conséquence. Notamment mettre la valeur `<DataSourceInfo>Provider=mondrian;Jdbc=jdbc:mysql://projetud.univ-ubs.fr:3306/e_24_3pexxxxxxx;JdbcUser=e_25_3pexxxxxxx;JdbcPassword=de_25_3pexxxxxxxd;JdbcDrivers=com.mysql.jdbc.Driver</DataSourceInfo>`
- Une fois la publication **complète** de votre cube Mondrian sur votre espace `P:/e_25_3jexxxxxxx/private_html/webapps/`, il faut demander à la DSI le redémarrage de son serveur Apache Tomcat 10.x. **Il faut passer par l'intermédiaire de l'enseignant Michel Dubois**.
- Sans passer par le redémarrage du serveur Apache Tomcat, vous pouvez influer sur le rechargement par le serveur tomcat distant de votre application java web :
     - Si vous sauvegardez WEB-INF/web.xml en l'ouvrant sans changement mais que la date change (touch) , cela doit hâter la prise en compte de  vos changements, votre webapp est rechargée dans  Tomcat si jamais ce fichier change ... 
     - Voir ici : https://kostenko.org/blog/2020/09/tomcat-reload-application.html
     - L'équivalent à la commande unix/linux `touch`, est disponible, après déploiement par décompréssion à la racine du lecteur `D:` du patch disponible [ici](https://moodle.univ-ubs.fr/pluginfile.php/583741/mod_folder/content/0/tools_MYSQL_8_XAMPP_7.2_tomcat_10_python_3.5_D_patch.zip?forcedownload=1), dans le shell `xampp-7.2.15-0-VC15 shell_xampp-32 - Lecteur D` :
```
         touch P:\e_25_3jexxxxxxx\private_html\webapps\WEB-INF\web.xml
```

- Test du cube du serveur Mondrian de votre espace de projet hébergé sur les serveurs de la DSI avec les differents clients (MS Excel, LibreOffice Calc, Jrubik, JPalo, Xavier, XMLa4js). 
- Les phases de tests (serveur local ou serveur de la DSI) peuvent donner lieu à la création d'une vidéo de démonstration, à mettre dans le dépot de rendu final et/ou à utiliser lors de la soutenance.



### Archive à rendre
Il faut rendre dans l'arborescence `7-Mondrian pour la configuration de votre cube Mondrian`, un fichier `ALire.txt` qui explique, énonce vos modifications par rapport à l'application java web `p_22_md_09java` livrée d'origine. 
Notamment vous aborderez les modifications faites aux fichiers `e_23_3pexxxxxxx_star.xml` et/ou `e_23_3pexxxxxxx_star_agg.xml`, aux fichiers `datasources.xml`, `defaultXmlaUrl.js`, `xmla.wsdl` et `mondrian-sql.log`. 

Vous préciserez si votre application java web est configurée par un serveur local ou de production. Dans les deux cas, vous mettrez la totalité de l'application Java Web. Si vous avez fait un déploiement sur le serveur de production, vous indiquerez l'URL pour atteindre cette dernière, si on est sur le réseau enseignement de l'UBS.

<a id='seance_darties_2025'></a>
## Utilisation du portail local ou de production php 7 / MySQL 8 /  bootstrap 5 / DataTables / Vega-Lite

[retour au début du calepin](#seance_1)

Il s'agit de faire, par enrichissement du prototype fourni :

- Un moteur d'invocation de rapports pouvant supporter, selon la configuration centralisée du tableau de bord PHP 7, de la manière la plus sécurisée :
  - Pour des produits web externes au portail (session **Xavier** de la distribution xmondrian, tableaux de bord / requêtes adhoc tabulaires / requêtes adhoc OLAP / rapports du portail Pentaho, tableaux de bord / rapports paginés **PowerBI** publiés voire [embarqués avec le javascript](https://stackoverflow.com/questions/57284193/how-to-set-filters-in-reports-power-bi-embedded-javascript), rapports / requêtes OLAP du portail InfoView de SAP Business Objects, rapports / tableaux de bord SAS invoque via le web, rapports et requêtes OLAP du portail IBM Cognos, etc.) :
    - un `iframe` dont l'URL complète est modifiée via Javascript ;
    - un `div` rempli via l'objet javascript `XMLHttpRequest` ;
    - le même `div` rempli via l'API `fetch` avec promesse ou API `fetch` avec `async`/`await` ;
  - une inclusion d'un script PHP comme c'est le cas traditionnellement dans une application full php, notamment pour le moteur interne de production de rapports.
- Un moteur interne de production de rapport PHP 7.2 enrichi de Adminer (administration MySQL, PostgreSQL voire Oracle) et RPFBC, version comptatible PHP 7.2 de PFBC, un générateur de formulaires Model-View-Controler utilisant bootstrap et offrant des classes PHP à manipuler. Il devra différencier, en fonction de maquettes ou de cahier de spécifications fonctionnelles détaillés ou de cahiers de recette pré-existants, différents profils de décideur : Directeur commercial, Directeur d'une région commerciale, directeur d'un magasin. Enfin, un profil Administrateur gérera les utilisateurs et leur profils voire pilotera l'alimentation (processus ETL), la sauvegarde et la restauration de la base de données de manière distante. Ce moteur respectera le principe MVC :
   - *Model* : le modèle doit être neutre du point de vue du SGBDR sous-jacent (MySQL, PostgreSQL ou Oracle) en soumettant les bon ordres SQL au SGBDR cible défini dans la configuration centralisée. Il devra utiliser l'API unifiée PDO mis aussi les commandes PDO qui permettent de préférence un comportment identique entre les différent type de SGBDR.
   - *Controler* : responsable de l'interaction avec l'utilisateur, c'est à lui d'appeler le modèle et de choisir quelle vue il faut inclure. En cas de formulaire, c'est lui qui est responsable de la définition du formulaire et de son traitement.
   - *View* : La page html5/CSS3 avec l'enrichissement javascript jQuery Datatables, javascript Bootstrap 5 et Vega-Lite. En cas de formulaire, c'est la vue qui s'occupe de son rendu.
- Revamping de l'IHM du cœur du tableau de bord qui utilise encore des pages XHTML et non HTML5.
- **Des rapports pertinents avec le moteur interne pour les profils directeur commercial (1), directeur régionale (5) et directeur de magasins (48).**  

Pour avoir accès à des exemples de rapports Web sur le cas DARTIES des années précédentes utilisant principalement les technologies SAS dans le cadre d'un projet décisionnel conséquent : 
1. il faut se rendre dans <a href="https://moodle.univ-ubs.fr/mod/folder/view.php?id=338294" target="_blank">les outils pour TP du cours de la SAE</a>
2. il faut télécharger à la racine du lecteur `D:` les parties (14?) de l'archive multiple `SAS-IntrNet-Share-Connect-USBGIS-PHP_5.2_Xampp-PHP_5.4_D.zip.0??`. 
3. Une fois **toutes** les parties téléchargées d'une manière **complète**, il faut sélectionner la partie `SAS-IntrNet-Share-Connect-USBGIS-PHP_5.2_Xampp-PHP_5.4_D.zip.001` puis, via le menu contextuel de l'explorateur de fichiers de Windows 11, après avoir sélectionner `Afficher d'autres options`, la décompresser avec 7-zip.exe dans le répertoire courant ( `7-zip` > `Extraire ici`). De manière transparente, les autres parties seront mises à contribution lors de la décompression.

A la suite de la décompression, pour avoir des raccouris Windows voire internet, il faut se rendre dans le réperoire `D:\Raccourcis`. Les deux serveurs web écoutent le port `80` donc sont incompatibles. Les sites web avec PHP 5.2.3 sont diffusés par le serveur web démarré avec le raccourci `startWebServer` tandis que les sites web avec PHP 5.4.31 sont diffusés par le serveur web démarré avec le raccourci `startWebServer - Xampp portable win32 1.8.2-6-VC9`.
- Le répertoire `D:\architecture` contient des programmes et des datasets SAS.
- Le répertoire `D:\usbgis\apps\xampplite\htdocs\sasweb` contient des projets SAS Web (portails PHP 5.2.3) utilisant la technologie SAS/intrnet . Les programmes SAS appelés via une URL se trouvent dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet`.
- Le répertoire `D:\tools\xampp\htdocs` contient les portails écrits en PHP 5.4.31.
- Les exemples jusqu'à DARTIES4 utilisent PHP 5.2 (xampplite). Les autres exemples utilisent PHP 5.4 (xampp)
- A part le serveur Apache PHP 5.4.31 et tomcat 7.0.42 dans `D:\tools\xampp`, le répertoire `D:\tools` contient des utilitaires pour la publication en tant que application Java Web Apache Axis de jobs Talend conçus avant la version 6.4 non incluse du TOS utilsés dans la partie Administration du portail décisionnel, notamment la distribution Apache Axis 1.4 et des compléments utiles (`extrawar`) avec un exemple de déploiement (`Exemple`).


Il convient d'étudier les projets SAS Web réalisés les années de la LP CSD :
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES1` utilisent l'étape DATA, la proc SQL, la proc Report, la proc tabulate, la proc gmap avec un dataset Annotate, la proc gchart et la proc template. Ils concernent notamment l'ongmet Prévisionnel (tableau avec proc report et graphique avec la proc gplot) en plus des onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM. L'export vers Word, Excel et vers PDF sont aussi réalisés via SAS.
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES2` utilisent l'étape DATA, la proc SQL, la proc Report, la proc tabulate, la proc gmap avec un dataset Annotate, la proc gchart et la proc template. Ils concernent notamment l'ongmet Prévisionnel (tableau avec proc report et graphique avec la proc gplot) en plus des onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM. L'export vers Excel et vers PDF sont aussi réalisés via SAS.
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES3` utilisent l'étape DATA, la proc SQL, la proc Report, la proc tabulate, la proc gmap avec un dataset Annotate, la proc gchart. Ils concernent les onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM. L'export vers Excel et vers PDF sont aussi réalisés via SAS.
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES4` utilisent l'étape DATA, la proc SQL, la proc Report, la proc tabulate, la proc gmap avec un dataset Annotate, la proc gchart. Ils concernent les onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM.
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES5` utilisent l'étape DATA, la proc SQL, la proc Report, la proc gmap avec un dataset Annotate. Ils concernent les onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM.
- Les programmes SAS dans `D:\architecture\PlanSASAca\Lev1\SASMain\IntrNet\sas_DARTIES9` utilisent la proc SQL et la proc Report. Il n'y a pas de graphiques. Ils concernent les onglets Accueil, Détail, Historique et Palmarès pour les profils DC, DR et DM.


### Le prototype du portail décisionnel, version 2025, compatible PHP 7.2

#### Aperçu du prototype :
L'URL [ici](http://www-i.univ-ubs.fr/etud/projets/p_22_md_301php/) vous permet d'accéder au prototype courant quand les prérequis sont satisfaits.

#### Prérequis :
- Votre base de données décisionnelle doit être remplie comme pour l'utilisation du serveur Mondrian. Vous pouvez y accéder avec [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=e_25_3pexxxxxxx&server=2).
- Des vues/tables `requete_*` et `select_*` doivent être créées via des scripts SQL qui se trouvent dans l'archive dans les sous_répertoires `DARTIES-2025\sql\mysql` au sein de votre base décisionnelle.
- le fichier `config.php` doit être cohérent s'il se trouve dans le serveur de production ou dans le serveur local de développement. Deux versions sont fournies. Par défaut, c'est sur le serveur local.

#### Alimentation de la table `dwr_faits_ventes_star` et de la table de la sécurité du portail
Le serveur Mondrian n'utilise pas la table `dwr_`. C'est le portail décisionnel qui l'utilise pour éliminer des indicateurs dans les rapports. De plus, les requêtes des rapports doivent s'adapter au profil de l'utilisateur courant. Alors qu'un directeur commercial peut voir l'ensemble des données des magasins, ce n'est pas le cas pour les autres profils de gestionnaires.

Préalables : dans [phpMyAdmin](http://localhost:96/phpmyadmin/sql.php?db=e_25_3pexxxxxxx&server=2):
1. Il faut d'abord, alimenter le schéma décisionnel en flocon (voir [ici](#seance_SQL_dec))
2. Il faut ensuite alimenter le schéma décisionnel en étoile. Pour cela, définir (déjà fait pour Mondrian) et exécuter les procédures stockées suivantes :

```
DELIMITER ;
CALL maj_securite_star();
CALL maj_dwr_faits_ventes_star();
```

Remarque, le jeu de données étant vieux de trois ans, la dernière invocation de la procédure n'insère aucun tuple dans `dwr_faits_ventes_star`. On peut, pour avoir des données sur la dernière année, exécuter :

pour la dernière l'année des données (2025-3=2022): 
```
INSERT INTO dwr_faits_ventes_star (id_magasin, id_produit, id_temps, indicateur, objectif , reel, date_maj)
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'VENTES' AS INDICATEUR , SUM(VENTES_OBJECTIF) AS OBJECTIF , SUM(VENTES_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-3 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS, CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'CA' AS INDICATEUR , SUM(CA_OBJECTIF) AS OBJECTIF , SUM(CA_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-3 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'MARGE' AS INDICATEUR , AVG(MARGE_OBJECTIF) AS OBJECTIF , AVG(MARGE_REEL) AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-3 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    ORDER by 1,2,3,4;
```	
On obtient comme réponse :
```
313701 lignes insérées. (traitement en 3,0000 seconde(s).)	
```

Pour l'année précédente des données (2025-4=2021): 
```
INSERT INTO dwr_faits_ventes_star (id_magasin, id_produit, id_temps, indicateur, objectif , reel, date_maj)
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'VENTES' AS INDICATEUR , SUM(VENTES_OBJECTIF) AS OBJECTIF , SUM(VENTES_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-4 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS, CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'CA' AS INDICATEUR , SUM(CA_OBJECTIF) AS OBJECTIF , SUM(CA_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-4 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'MARGE' AS INDICATEUR , AVG(MARGE_OBJECTIF) AS OBJECTIF , AVG(MARGE_REEL) AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-4 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    ORDER by 1,2,3,4;
```	
La réponse est : `374976 lignes insérées. (traitement en 7,0000 seconde(s).)`

Pour l'année précédente des données (2025-5=2020): 
```
INSERT INTO dwr_faits_ventes_star (id_magasin, id_produit, id_temps, indicateur, objectif , reel, date_maj)
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'VENTES' AS INDICATEUR , SUM(VENTES_OBJECTIF) AS OBJECTIF , SUM(VENTES_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-5 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS, CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'CA' AS INDICATEUR , SUM(CA_OBJECTIF) AS OBJECTIF , SUM(CA_REEL)AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-5 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    UNION ALL
    SELECT ID_MAGASIN , ID_PRODUIT , ID_TEMPS , 'MARGE' AS INDICATEUR , AVG(MARGE_OBJECTIF) AS OBJECTIF , AVG(MARGE_REEL) AS REEL,CURRENT_TIMESTAMP AS DATE_MAJ
    FROM faits_ventes_star
    WHERE id_temps LIKE CONCAT(CAST(YEAR(NOW())-5 as char),'%') COLLATE utf8mb4_0900_ai_ci
    GROUP BY ID_MAGASIN , ID_PRODUIT , ID_TEMPS , CURRENT_TIMESTAMP
    ORDER by 1,2,3,4;
```	
La réponse est : `374976 lignes insérées. (traitement en 4,0000 seconde(s).)`
#### Installation du protype sur le serveur apache local de développement
En local, à déployer dans `D:\tools\xampp-7.2.15-0-VC15\htdocs` pour que ce site soit diffusé via l'URL http://localhost:96/DARTIES-2025/. Le fichier `config.php` courant est prévu pour un déploiement local. Attention, le fichier `config.php` doit être dans le répertoire `D:\tools\xampp-7.2.15-0-VC15\htdocs\DARTIES-2025`. Le paramétrage pour le déploiement local est dans le fichier `config_local.php`. Ces informations sont à retranscrire dans `config.php`.
#### Installation du protype sur le serveur apache de production (contrôlé par la DSI)
A déployer dans `P:\e_25_3pexxxxxxx\private_html` en adaptant le fichier `config.php` pour que ce site soit diffusé via l'URL http://www-i.univ-ubs.fr/etud/projets/e_25_3pexxxxxxx/. Attention, le fichier `config.php` doit être dans le répertoire `P:\e_25_3pexxxxxxx\private_html` et non pas dans le répertoire `P:\e_25_3pexxxxxxx\private_html\DARTIES-2025`. Le paramétrage pour le déploiement distant est dans le fichier `config_p_22_md_301php.php`. Ces informations sont à retranscrire dans `config.php`.


### Le travail à faire :
- Pour l'utilisateur ayant le profil `administrateur`,  offrir un bouton pour créer un noveau profil 'chaise musicale' qui n'est pas affecté déjà à un utilisateur. Le libellé sera `chaise musicale 1` ou `chaise musicale 2` si un profil `chaise musicale 1` existe déjà.
- Pour l'utilisateur ayant le profil `administrateur`,  offrir un bouton pour constater l'ouverture d'un nouveau magasin. Il faudra lui affecter une ville qui n'est pas encore prise dans un département au choix. découlera alors les régions anciennes (1956), nouvelles (2016) et commerciales. Il faudra renseigner les informations concernants le magasin. Le mois d'ouverture dans l'année courante. Tous les indicateurs prévisionnels à compter du mois d'ouverture seront mis à zero.
- Pour l'utilisateur ayant le profil `directeur commercial`, faire les rapports correspondants aux onglets 'Accueil', 'Historique', 'Palmarès', 'Détail' et 'Prévisionnel'. 
- Pour les 5 utilisateurs ayant le profil `directeur régional`, faire les rapports correspondants aux onglets 'Accueil', 'Historique', 'Palmarès', 'Détail' et 'Prévisionnel'. 
- Pour les 48 utilisateurs ayant le profil `directeur magasin`, faire les rapports correspondants aux onglets 'Accueil', 'Historique', 'Palmarès', 'Détail' et 'Prévisionnel'.

Les rapports doivent donc s'adapter en fonction de l'onglet et du profil utilisateur (sécurité du portail) mais aussi des valeurs courantes des filtres à gauche. Ils sont constitués titres, de tables HTML enrichies avec Datatables, de graphique Vega-Lite. Il sont conformes à des cahiers des charges fournis dans les archives ou ils sont portés depuis des solutions SAS Web vers un moteur de traitement MySQL qui ne connait que le SQL. 

### Structure du prototype
Le modèle consiste en 3 classes (`SAPBusinessObjectsConnectionTokenClient`, `WSTalendRunJobClient` et `Bdd`) qui se trouvent dans le fichier `m_fonction.php` du répertoire `modele`.

Les vues et les contrôleurs d'intérêt se trouvent dans `site\contenu\accueil`, `site\contenu\historique`, `site\contenu\palmares`, `site\contenu\detail`, `site\contenu\previsionnel`. Ces vues et contrôleurs doivent gérer :
- les 3 différents profils des gestionnaires (DC, DR et DM)
- l'export Excel et PDF (à étudier comment : `$_SERVER[HTTP_REFERER]` ? mais [c'est pas sûr, pas fiable](https://stackoverflow.com/questions/36240145/how-to-use-serverhttp-referer-correctly-in-php), stockage de la page courante dans une variable de session ?, plutôt dans le contrôleur car la vue s'occupe du formatage web du rapport)

L'export Excel et PDF se trouvent dans `site\contenu\excel` et dans `site\contenu\pdf`. 

Les vues et les contrôleurs pour l'administration se trouvent dans `site\connection`.

Les images cartographiques des magasins, des régions administratives nouvelles et anciennes, des régions commerciales, de la france au 48 magasins se trouvent dans `image`. Les noms des fichiers se trouvent dans les tables dim_magasin, etc. ou dans dim_magasin_star (plusieurs colonnes). La proc gmap avec annotate de SAS a été utilisée pour prégénérer ces images. Les fichiers se terminent par `_0` pour toutes les enseignes ou `_1` pour Boulanger ou `_2` pour Darty ou `_3` pour Leroy Merlin. Pour un magasin d'une enseigne, il y a un logo à la place de la ville concernée.  

Vous devez explorer le répertoire `docs` :
- Le fichier `ATEM_DARTIES_CDR_VF_Corrigée.pdf` (aussi consultable [ici](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/Darties_ATEM_2013_Cahier_recette.pdf?forcedownload=1)) aborde notamment l'onglet prévisionnel, en plus des onglets accueil, historique, palmarès ayant des tableaux et des graphiques. Comme c'est un cahier de recette, c'est le réalisé et non une maquette.
- Le fichier `Cahier de recette V2.1.pdf` (aussi consultable [ici](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/Darties_TALLAS_2013_Cahier_recette.pdf?forcedownload=1)) aborde notamment l'onglet prévisionnel, en plus des onglets accueil, historique, palmarès ayant des tableaux et des graphiques. Comme c'est un cahier de recette, c'est le réalisé et non une maquette. 
- Le fichier `VSDDARTICDR02.pdf` (aussi consultable [ici](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/Darties_VSD_2013_Cahier_recette.pdf?forcedownload=1)) aborde notamment l'onglet prévisionnel, en plus des onglets accueil, détail, historique, palmarès ayant des tableaux et des graphiques. Comme c'est un cahier de recette, c'est le réalisé et non une maquette.
- Le fichier `SFD_Fratbag.pdf` (aussi consultable [ici](https://moodle.univ-ubs.fr/pluginfile.php/649663/mod_folder/content/0/SFD_Fratbag.pdf?forcedownload=1)) propose les onglets accueil, détail, historique, palmarès. C'est un dossier de spécifications fonctionneles détaillées, c'est plutôt la maquette. Les tableaux sont détaillés mais il n'y a aucun graphiques dynamiques. Seule une carte prégénérée est présentées.

Pour une description technique succincte des solutions antérieure plutôt à base de SAS, vous pouvez vous rendre [ici](http://michel.dubois.ubs.chez.com/index.php?n=Blog.Vid-os-des-projets-BusinessoO?n=Teaching.PjTechn).

### Modes d'inclusion des rapports
Actuellement, il y a deux modes d'inclusion des rapports :
- par `iframe` dont l'attribut `src` est dynamiquement changé par le script `js\ajax.js`. De fait, il y a deux sites : celui à l'extérieur de l'`iframe`, et celui à l'intérieur de l'`iframe`. Ils s'ignorent sauf que l'URL de l'attribut `src` peut être enrichie d'attributs CGI car c'est une invocation HTTP GET. Il n'y a pas d'effets de bord entre scripts CSS ou javascript du site externe ou parent et le site interne. Ce mode d'inclusion est compatible avec les produits externes à PHP comme SAP Business Objects, technologies SAS web, MS PowerBI, serveur Mondrian connecté à un client léger xavier. A l'origine peu sécurisée, l'utilisation d'iframe est tributaire d'au moins un non refus du site intégré. Il se trouve que le serveur web de production contrôlé par la DSI est configuré pour envoyer un refus pour chaque application web qu'il héberge.
- dans un `div` dont le contenu est alimenté par le script `js\ajax.js` via l'objet `XMLhttpRequest`. Par défaut, un script javascript ne peut contacter que le serveur d'origine de son téléchargement par le navigateur, corformément à la politique *`same origin`*. Une technique appelée `CORS` permet de s'affranchir de cette contrainte via des entêtes HTTP de réponse `Access-Control-Allow-Origin: *` et `Access-Control-Allow-Credentials`. Non seulement, il y un seul site donc le style du rapport peut impacter le style du portail mais la page html doit être cohérente : dans le `div`, pas de balise `html`, `head`, `body`.

L'inclusion par `div` est actuellement le mode par défaut. Autant l'enrichissement d'une `table` html par un objet javascript `Datatables` n'a pas d'effet de bord dans l'autre mode, autant on constate des effets indésirables sur le site. Pour la parte administration, l'effet est caché car on compose systématiquement une nouvelle page html. Pour les rapports décisionnels, la recherche, la pagination automatique de table html volumineuse étant les principaux apports de la bibliothèque javascript Datatables, ils ne sont pas d'utilité pour les tables réduites demandées dans le cahier des charges ou dans les maquettes.

### Adapter une requête 100% MySQL en SQL interpolé avec le tableau associatif PHP `$_GET`

Soit la requête 100% SQL pour MySQL contenant des variables de session :
```
USE e_25_3peXXXXXXX;
SET @SELECT_CUMUL=1;
SET @SELECT_COURS=1;
SET @SELECT_USER=2;
SET @SELECT_INDICATEUR=1;
SET @SELECT_PRODUIT=0;
SET @SELECT_ENSEIGNE=0;
SET @SELECT_ONGLET=1;
SET @SELECT_GEO=0;
SET @SELECT_DEVISE=1;
SET @SELECT_TEMPS='2022_1_2022';

select coalesce(sum(ca_reel)*@SELECT_COURS,0) as ca_reel
from faits_ventes_star
-- securite
where faits_ventes_star.id_magasin in (select id_magasin from utilisateur,securite,profil where utilisateur.id_profil=profil.id_profil and utilisateur.id_utilisateur=@SELECT_USER and profil.id_profil=securite.id_profil and securite.id_onglet=@SELECT_ONGLET) 
-- ce que je veux du point de vue des magasins
and faits_ventes_star.id_magasin in (select id_magasin from requete_geo where code=@SELECT_GEO)
-- ce que je veux du point de vue des enseignes de magasin
and faits_ventes_star.id_magasin in (select id_magasin from dim_magasin_star where id_enseigne in (select id_enseigne from requete_enseigne where code=@SELECT_ENSEIGNE))
-- ce que je veux du point de vue du temps
and faits_ventes_star.id_temps in (select id_temps from (SELECT id_temps FROM requete_temps_1 WHERE @SELECT_CUMUL=1 and code=@SELECT_TEMPS UNION ALL SELECT id_temps FROM requete_temps_0 WHERE @SELECT_CUMUL=0 AND code=@SELECT_TEMPS) AS requete_temps)
-- ce que je veux du point de vue des produits
and faits_ventes_star.id_produit in (select id_produit from requete_produit where code=@SELECT_PRODUIT)
;
```

Dans le client lourd console `mysql.exe`, le client lourd graphique `MySQL Workbench` ou le client leger `phpMyAdmin`, il vous faut soumettre cette requête comme un tout. La requête est en euros. Les variables de session MySQL sont introduites par `@`, contrairement aux variables serveur introduites par `@@`.

L'exécution de l'onglet `Accueil` nous affiche pour `print_r($_GET)` :
```
Array
(
    [SELECT_USER] => 2
    [SELECT_INDICATEUR] => 0
    [SELECT_PRODUIT] => 0
    [SELECT_ENSEIGNE] => 0
    [SELECT_TEMPS] => 2022_1_2022
    [SELECT_CUMUL] => 0
    [SELECT_GEO] => 0
    [SELECT_TRI] => 0
    [SELECT_ONGLET] => 1
    [SELECT_DEVISE] => 1
    [SELECT_COURS] => 1.00
)
```
On voit que toutes les valeurs sont numériques sauf pour la clé `SELECT_TEMPS` dont la valeur chaîne n'est pas entourée d'apostrophes bien que ce soit un litéral. Pour la requête SQL, il faudra donc l'entourer d'apostrophes aux deux endroits.

Dans `c_accueil.php`, il faut ajouter à la ligne 36 :
```
$query_ca_reel_euros = "
select coalesce(sum(ca_reel)*{$_GET['SELECT_COURS']},0) as ca_reel
from faits_ventes_star
where faits_ventes_star.id_magasin in (select id_magasin from utilisateur,securite,profil where utilisateur.id_profil=profil.id_profil and utilisateur.id_utilisateur={$_GET['SELECT_USER']} and profil.id_profil=securite.id_profil and securite.id_onglet={$_GET['SELECT_ONGLET']}) 
and faits_ventes_star.id_magasin in (select id_magasin from requete_geo where code={$_GET['SELECT_GEO']})
and faits_ventes_star.id_temps in (select id_temps from (SELECT id_temps FROM requete_temps_1 WHERE {$_GET['SELECT_CUMUL']}=1 and code='{$_GET['SELECT_TEMPS']}' UNION ALL SELECT id_temps FROM requete_temps_0 WHERE {$_GET['SELECT_CUMUL']}=0 AND code='{$_GET['SELECT_TEMPS']}') AS requete_temps)
and faits_ventes_star.id_produit in (select id_produit from requete_produit where code={$_GET['SELECT_PRODUIT']})
and faits_ventes_star.id_magasin in (select id_magasin from dim_magasin_star where id_enseigne in (select id_enseigne from requete_enseigne where code={$_GET['SELECT_ENSEIGNE']}))
	";
```   
Pour pouvoir interpoler en PHP une chaine avec une valeur complexe qu'est un élément du tableau associatif `$_GET`, il faut remplacer `@CLE` correspondant à la variable de session MySQL `CLE` par dans une chaîne introduite par des guillemets permettant l'interpolation par `{$_GET['CLE']}`. 

Dans le même fichier, il faut ajouter à la ligne 49 :
```
$data_ca_reel_euros=$c->row_select_query($query_ca_reel_euros);
$html_ca_reel_euros=$c->html5_table_from_row_select_query($data_ca_reel_euros,'query_ca_reel_euros',"Une requête qui s'adapte aux filtres et a la securite");
```

Dans `v_accueil.php`, il faut ajouter à la ligne 19 :
```
echo ('<hr/>');
echo ($html_ca_reel_euros);
```

Même si on commence à obtenir un résultat intéressant, utiliser directement les valeurs du tableau associatif `$_GET` pour l'interpolation d'un chaîne correspondant à une future requête SQL, s'est s'exposer à l'injection SQL.

Pour l'éviter, il vaut mieux passer par une requête paramétrée avec PDO.



### Adapter une requête 100% MySQL en SQL préparé et paramétré avec le tableau associatif PHP `$_GET`

Dans le fichier `modele\m_functions.php`, à la ligne 361, il faut ajouter :
```
		 /**
         * Cette fonction effectue les requetes de type SELECT de type parametree avec une etape prepare
		 * @param string $sql La requete SQL ayant des parametres nommes
		 * @param array $param_array Le tableau associatif qui désigne les valeurs effectives des parametres nommes. l'ordre des clés n'a pas d'importance		 
		 * @return array Le resultat complet de l'execution de la requete SELECT		 		 
         */
        public function row_select_prepared_query($sql, $param_array=Array()){
			$stmt = $this->pdo->prepare($sql);
			$result=$stmt->execute($param_array);
			// return Array() if the query cannot be executed
			if (!$stmt->execute($param_array)) {
				return Array();
			}
			// return Array() if there was an **error** retrieving the query results
			if (($info = $stmt->fetchAll(\PDO::FETCH_BOTH)) === false) {
				return Array();
			}
			return $info;
        }
```
Dans `c_accueil.php`, il faut ajouter à la ligne 46 :
```
$prepared_query_ca_reel_euros = "
select coalesce(sum(ca_reel)*:SELECT_COURS,0) as ca_reel
from faits_ventes_star
where faits_ventes_star.id_magasin in (select id_magasin from utilisateur,securite,profil where utilisateur.id_profil=profil.id_profil and utilisateur.id_utilisateur=:SELECT_USER and profil.id_profil=securite.id_profil and securite.id_onglet=:SELECT_ONGLET) 
and faits_ventes_star.id_magasin in (select id_magasin from requete_geo where code=:SELECT_GEO)
and faits_ventes_star.id_temps in (select id_temps from (SELECT id_temps FROM requete_temps_1 WHERE :SELECT_CUMUL=1 and code=:SELECT_TEMPS UNION ALL SELECT id_temps FROM requete_temps_0 WHERE :SELECT_CUMUL=0 AND code=:SELECT_TEMPS) AS requete_temps)
and faits_ventes_star.id_produit in (select id_produit from requete_produit where code=:SELECT_PRODUIT)
and faits_ventes_star.id_magasin in (select id_magasin from dim_magasin_star where id_enseigne in (select id_enseigne from requete_enseigne where code=:SELECT_ENSEIGNE))
	";	
```
Pour utiliser des paramètres nommés dans une requête PDO préparée, il faut remplacer `@CLE` correspondant à la variable de session MySQL `CLE` par dans une chaîne introduite par des guillemets ou des apostrophes (qui ne permettent pas l'interpolation) par `:CLE`. De plus, c'est PDO qui se chargera de mettre les apostrophes si un paramètre effectif est un litéral chaîne. On voit que c'est beaucoup plus simple !

On remarque que la requête utilise 7 des 10 variables de session MySQL.

Dans le même fichier, il faut ajouter à la ligne 62 :
```
	$data_prepared_ca_reel_euros=$c->row_select_prepared_query($prepared_query_ca_reel_euros,['SELECT_USER' => $_GET['SELECT_USER'], 'SELECT_PRODUIT' => $_GET['SELECT_PRODUIT'], 'SELECT_ENSEIGNE' => $_GET['SELECT_ENSEIGNE'], 'SELECT_TEMPS' => $_GET['SELECT_TEMPS'], 'SELECT_CUMUL' => $_GET['SELECT_CUMUL'], 'SELECT_GEO' => $_GET['SELECT_GEO'], 'SELECT_ONGLET' => $_GET['SELECT_ONGLET'], 'SELECT_COURS' => $_GET['SELECT_COURS']]);	
	$html_prepared_ca_reel_euros=$c->html5_table_from_row_select_query($data_prepared_ca_reel_euros,'prepared_query_ca_reel_euros',"Une requête préparée qui s'adapte aux filtres et a la securite et qui évite l'injection SQL");

```

Dans `v_accueil.php`, il faut ajouter à la ligne 21 :
```
echo ('<hr/>');
echo ($html_prepared_ca_reel_euros);
```

Techniquement, c'est mieux !

Reste que ce n'est pas satisfaisant du point de vue de la gestion des monnaies. Le cours des monnaies fluctuantes, ce n'est qu'une moyenne sur toute l'étude, moyenne qui n'est pas assez précise. Il faut réellement appliquer les moyennes mensuelles des cours journaliers tant qu'on dispose de ces derniers. 

Il faut aller voir dans `sql\mysql\currency_report_prod_dev.sql` pour trouver une meilleure solution. Elle utilise une table temporaire (locale à la transaction/session MySQL) currency. La relation session MySQL et session PHP est floue. Il vaut mieux sécuriser la solution par une table currency normale. A charge pour le processus ETL de garantir sa mise à jour. 

Ainsi, pour résumer, si pour le mois d'une ligne, il existe une moyenne mensuelle, la solution l'utilise. Si on est hors du min et du max, c'est la moyenne sur toute l'étude qui est utilisée. Bien entenu, si on alimente bien les bases de données, on ne risque pas d'être hors limite.

Par rapport aux solutions SAS existantes, elles pouvaient utiliser la `proc SQL` pour créer un dataset sas qui était rendu par une `proc print` (pas de plus-value par rapport au `SELECT`), la `proc report` (formatage avancé plutôt de tableaux simples) ou `proc tabulate` (formatage avancé plutôt de tableaux croisés).

Autant un tableau simple, se fait facilement à l'aide d'une requête, autant un tableau croisé, c'est plus difficile non seulement car le SQL peut être plus complexe (GROUP BY ROLLUP, GROUP BY CUBE - non supporté par MySQL) mais aussi parce qu'il faut peut être plusieurs requêtes (cf. [ici](https://moko.developpez.com/articles/tableau-croise-php-mysql/)).

La méthode `html5_table_from_row_select_query()` de la classe `Bdd` du modèle est un point de départ pour obtenir les tableaux souhaités. Elle ne crée qu'un tableau simple. Un des arguments est le résultat exhaustif de l'exécution de la requête sous forme d'un tableau associatif. Vous pouvez la cloner pour accepter plusieurs arguments de cette nature. Fusions de colonnes ou de lignes doivent être gérées par une telle methode.

Si vous créez plusieurs methodes PHP utilitaires pour le formatage des tables, il vaudrait mieux creér une nouvelle classe dans le modèle de nom `HTMLReport` plutôt de continuer à les rattacher à `Bdd`. Si vous mélez dans le code des accès au SGBD, la méthode est logiquement rattachée à la classe `Bdd`.

Quelques liens sur l'export en php :
- en PDF la bibliothèque [Free PDF (FPDF)](https://www.fpdf.org/) est très connue ;
- en Excel, cette [page](https://www.nicolashachet.com/blog/developpement-php/generer-fichiers-excel-xlsx-xls-php/) explore diverses possibilitées.

Dans le prototype, une bibliothèque PHP doit se mettre dans `lib`.

Dans le prototype, une bibliothèque CSS/Javascript doit se mettre dans `site\addins`.

Dernier conseil pour vos rapports, la première chose à mettre, c'est un titre qui rappelle tous les éléments du contexte (monnaie, période temporelle, niveau géographique, niveau des produits, niveau des enseignes, etc.). Une ébauche codée en javascript se trouve dans `js\ajax.js` et dans `index.php` (script embarqué, code minimal dupliqué pour se lancer au chargement de la page index.php). Elle consiste à rajouter des paramètres CGI supplémentaires lors de l'invocation des rapports. 


### Integrer XMondrian pour le cas DARTIES au portail décisionnel

L'URL [ici](http://www-i.univ-ubs.fr/etud/projets/p_22_md_301java/xavier/index.html) vous permet d'accéder à un serveur Mondrian branché au cas DARTIES.

Voici les étapes pour intégrer un tel client OLAP à votre protype PHP :
- Il faudra alors basculer dans `config.php` de la valeur `'js XMLHttpRequest'` (ligne 181) à la valeur `'js iframe'` (ligne 237) pour le paramètre `$CFG->InvocationReportEngine`.
- Il faudra choisir l'URL de même origine entre le portail décisionnel et l'application XMondrian. Avec un serveur web configuré en *reverse_proxy* (ou proxy inverse), la même application java web peut être atteinte via le port 8080 (port par défaut de Apache Tomcat) et via le port 80, port par défaut d'un serveur web comme Apache, jouant le rôle de proxy inverse. Un proxy inverse désigne un serveur placé devant les serveurs web et transmettant les requêtes des clients (par exemple, les navigateurs web) à ces serveurs web. Les solutions de proxy inverse sont généralement déployées pour améliorer la sécurité, les performances et la fiabilité (voir [ici](https://www.cloudflare.com/fr-fr/learning/cdn/glossary/reverse-proxy/)).
 - Pour les serveurs de production, il y a un reverse proxy qui permet d'atteindre les projets PHP et les projets Java Web avec le même domaine (`www-i.univ-ubs.fr`) et le même port (80).
 - Pour les serveurs de développement, il y a un reverse proxy, utilisant le protocole AJP actif dans Apache Tomcat, configuré dans Apache qui permet d'atteindre les projets PHP et les projets Java Web avec le même domaine (`localhost`) et le même port (96). 
 - L'adresse à utiliser doit être précisée dans le paramètre `$CFG->XMONDRIANURL` (ligne 299) du fichier `config.php`.
- Il faudra configurer l'application Java Web pour qu'elle renvoie automatiquement des entêtes HTTP de sécurité à propos des iframes.
- Une fois mise au point sur les serveurs locaux (Apache de Xampp, apache Tomcat 10), la configuration peut être portée sur les serveurs de productions représentés par les répertoires `P:\e_25_3peXXXXXXX\private_html` et `P:\e_25_3jeXXXXXXX\private_html\webapps`. 

Une demande de modification de la configuration du serveur Apache httpd contrôlé par la DSI a été envoyée :

```
Le serveur apache gérant les url `http://www-i.univ-ubs.fr/etud/projets/*` semble être configuré pour renvoyer systématiquement l'entête HTTP de réponse X-Frame-Options: DENY sans laisser la possibilité aux applications PHP web hébergées ou aux applications Java Web en proxy inverse d'elles même d'envoyer le bon header (X-Frame-Options: SAMEORIGIN).

Lorsque l'application web fait cela, le navigateur en présence de deux entêtes contradictoires choisit la plus restrictive.

Pouvez-vous changer ce comportement, c'est à dire commenter la directive Header set X-Frame-Options DENY au niveau de httpd.conf.
```
Si la réponse est positive, on n'a plus à se soucier de la configuration de l'application java web XMondrian.
La réponse est positive : La modification à été faite afin de renforcer la sécurité. L'option à été positionnée sur « sameorigin ».

L'intégration du portail Darties avec XMondrian est immédiate :

![integration](DARTIES-2025_XMondrian.png)


#### Tomcat et l'entête de réponse HTTP *X-Frame-Options SAME ORIGIN*
L'en-tête de réponse HTTP `X``-Frame-Options` permet d'indiquer si un navigateur est autorisé à afficher une page dans une `frame`, une `iframe`, un élément intégré ou un objet. Les sites peuvent l'utiliser pour se prémunir contre les attaques de détournement de clic, en s'assurant que leur contenu n'est pas intégré à d'autres sites.

Deux directives sont possibles pour `X-Frame-Options` : `X-Frame-Options: DENY` et `X-Frame-Options: SAMEORIGIN`. Si vous spécifiez `DENY`, la page ne peut pas être affichée dans une frame, quel que soit le site qui tente de le faire. Même l'intégration de contenu provenant du même site web échouera. Si vous spécifiez `SAMEORIGIN`, la page peut uniquement être affichée dans une frame sur des pages de même origine. Une autre option est `ALLOW-FROM`. Cette directive obsolète ne fonctionne plus avec les navigateurs modernes. Sa fonctionnalité permet d'afficher une page dans une iframe uniquement depuis l'URI d'origine spécifiée.

Le problème avec `X-FRAME-OPTIONS` est qu'il ne prend en charge que deux options : un refus total ou une autorisation depuis `SAMEORIGIN`. Or, il est impossible de spécifier l'URL d'origine autorisée à intégrer la page dans une iframe, car l'option `ALLOW-FROM` n'est pas prise en charge par les navigateurs modernes. De plus, même sur les anciens navigateurs, il est impossible de combiner les options `DENY` et `SAMEORIGIN` avec l'en-tête ALLOW-FROM. Par ailleurs, `ALLOW-FROM` ne prend en charge qu'une seule URL d'origine et ne peut pas être un caractère générique comme *.

Pour générer des entêtes HTTP de sécurité (`X-Frame-Options`) pour autoriser les iframes dans une application java web d'un serveur Tomcat 10, il faut ajouter des *filters* dans le fichier `D:\tools\apache-tomcat-10.0.22\webapps\xmondrian\WEB-INF\web.xml` :
```
<filter>
    <filter-name>httpHeaderSecurity</filter-name>
    <filter-class>org.apache.catalina.filters.HttpHeaderSecurityFilter</filter-class>
    <init-param>
        <param-name>antiClickJackingOption</param-name>
        <param-value>SAMEORIGIN</param-value>
    </init-param>
</filter>
```

```
<filter-mapping>
    <filter-name>httpHeaderSecurity</filter-name>
    <url-pattern>/*</url-pattern>
    <dispatcher>REQUEST</dispatcher>
</filter-mapping>
```

C'est la version 7.0.63 qui a intégré la classe `org.apache.catalina.filters.HttpHeaderSecurityFilter` dans la distribution Apache Tomcat. C'est donc l'un des [*Tomcat 7.x built in filters*](https://tomcat.apache.org/tomcat-7.0-doc/config/filter.html#HTTP_Header_Security_Filter)

Avant cette version, il fallait créer et compiler la classe et la placer dans `D:\tools\apache-tomcat-10.0.22\webapps\xmondrian\WEB-INF\classes` :

```
public class XFrameHeaderFilter implements Filter {
    public void doFilter(ServletRequest req, ServletResponse resp, FilterChain chain) throws ServletException {
        ((HttpServletResponse) resp).setHeader("x-frame-options", "allow");
        chain.doFilter(req, resp);
    }
}
```
Il faut ajouter le *filter* dans le fichier `D:\tools\apache-tomcat-10.0.22\webapps\xmondrian\WEB-INF\web.xml` :
```
<filter>
  <filter-name>x-frame-header</filter-name>
  <filter-class>XFrameHeaderFilter</filter-class>
</filter>
<filter-mapping>
  <filter-name>x-frame-header</filter-name>
  <url-pattern>/*</url-pattern>
</filter-mapping>
```

#### Tomcat et l'entête de réponse HTTP *Content-Security-Policy frame-ancestors 'self';*

La directive `CSP frame-ancestors` spécifie les parents valides pouvant intégrer une page à l'aide d'une `frame`, d'une `iframe`, d'un objet, d'une balise embed ou d'une applet. Définir cette directive sur « `none` » est similaire à `X-Frame-Options: deny`. La définir sur « `self` » est similaire à `X-Frame-Options: sameorigin`. Vous pouvez même spécifier les URL d'origine autorisées à intégrer la page dans une iframe ; cela équivaut à la fonctionnalité obsolète `ALLOW-FROM` de `X-FRAME-OPTIONS`. Cette directive peut être combinée avec d'autres et prend également en charge plusieurs URL d'origine et les caractères génériques.

Si `X-FRAME-OPTIONS` et `CSP frame-ancestors` sont tous deux présents dans l'en-tête de réponse, les navigateurs modernes ignorent `X-FRAME-OPTIONS`. `CSP frame-ancestors` est l'option recommandée pour les navigateurs modernes, mais vous devez activer les deux en-têtes si la compatibilité avec les anciens navigateurs est toujours requise.

Pour ajouter des en-têtes spécifiques tels que CSP ou X-XSS-Protection, utilisez la configuration `web.xml` avec le [org.apache.catalina.filters.AddDefaultCharsetFilter](https://tomcat.apache.org/tomcat-7.0-doc/config/filter.html#Add_Default_Character_Set_Filter) :
```
<filter>
  <filter-name>HeaderFilter</filter-name>
  <filter-class>org.apache.catalina.filters.AddDefaultCharsetFilter</filter-class>
  <init-param>
    <param-name>Content-Security-Policy</param-name>
    <param-value>frame-ancestors 'self';</param-value>
  </init-param>
  <init-param>
    <param-name>X-XSS-Protection</param-name>
    <param-value>0</param-value>
  </init-param>
</filter>

<filter-mapping>
  <filter-name>HeaderFilter</filter-name>
  <url-pattern>/*</url-pattern>
</filter-mapping>
```

Après un essai, cela ne semble pas marcher !

Sinon, si l'ajout d'un filtre à votre application est possible, vous pouvez utiliser le code suivant pour ajouter un en-tête à chaque réponse :

```
@WebFilter("/*")
public class MyFilter implements Filter {

    @Override
    public void doFilter(ServletRequest request, ServletResponse response, 
                         FilterChain chain) throws IOException, ServletException {

        HttpServletResponse httpResponse = (HttpServletResponse) response;
        httpResponse.setHeader("Content-Security-Policy", "frame-ancestors 'self'");

        chain.doFilter(request, response);
    }
}
```

L'autre solution est de proposer une valve comme [ici](https://stackoverflow.com/questions/37306300/can-tomcat-7-be-configured-to-insert-content-security-policy-http-header).

#### Tomcat et l'entête de réponse HTTP pour activer CORS
Le partage des ressources entre origines multiples ou *Cross-Origin Resource Sharing* (CORS) est un mécanisme d'intégration des applications. La spécification CORS permet aux applications Web clientes chargées dans un domaine particulier d'interagir avec les ressources d'un autre domaine. C'est par exemple nécessaire quand un script javascript fait appel à un web service SOAP ou à une API REST qui ne partage pas la même origine (domaine:port). Si l'application web et le WS/API sont hébergées sur le même hôte mais ne partagent pas le même port, on peut avoir recours à un proxy inverse. Sinon, il faut activer le CORS. 

Pour ajouter des en-têtes spécifiques à la gestion des CORS, utilisez la configuration `web.xml` avec le [org.apache.catalina.filters.CorsFilter](https://tomcat.apache.org/tomcat-7.0-doc/config/filter.html#CORS_Filter) :
```
<filter>
  <filter-name>CorsFilter</filter-name>
  <filter-class>org.apache.catalina.filters.CorsFilter</filter-class>
  <init-param>
    <param-name>cors.allowed.origins</param-name>
    <param-value>*</param-value>
  </init-param>
  <init-param>
    <param-name>cors.allowed.methods</param-name>
    <param-value>GET,POST,DELETE,PUT,HEAD,OPTIONS</param-value>
  </init-param>
  <init-param>
    <param-name>cors.allowed.headers</param-name>
    <param-value>Content-Type,X-Requested-With,accept,Origin,Access-Control-Request-Method,Access-Control-Request-Headers,pragma,cache-control,soapaction</param-value>
  </init-param>
  <init-param>
    <param-name>cors.exposed.headers</param-name>
    <param-value>Access-Control-Allow-Origin,Access-Control-Allow-Credentials</param-value>
  </init-param>
  <init-param>
    <param-name>cors.support.credentials</param-name>
    <param-value>false</param-value>
  </init-param>
  <init-param>
    <param-name>cors.preflight.maxage</param-name>
    <param-value>10</param-value>
  </init-param>
</filter>
<filter-mapping>
  <filter-name>CorsFilter</filter-name>
  <url-pattern>/*</url-pattern>
</filter-mapping>
```

### Archive à rendre
Il faut rendre dans l'arborescence `8-Portail Darties-2025`, un fichier `ALire.txt` qui explique, énonce vos modifications par rapport à l'archive ([message du mercredi 5 novembre 2025, 17:58](https://moodle.univ-ubs.fr/mod/forum/discuss.php?d=35870)) patchée ([message du vendredi 14 novembre 2025, 18:51](https://moodle.univ-ubs.fr/mod/forum/discuss.php?d=35870#p65089)) puis tous les fichiers modifiés (rien qu'eux) dans l'arborescence d'origine comme cela a été fait pour le patch ! 